# Star Kingdom — Full Prediction and Optimization Pipeline

## Goal

Build and understand the complete Star Kingdom pipeline from the original data to the final 25-year replacement strategy.

The pipeline will be built step by step:

1. Load and understand the original data
2. Merge the pipe and historical event data
3. Clean and preprocess the data
4. Engineer features
5. Build the leak prediction dataset
6. Train and evaluate the leak prediction model
7. Build the cost prediction dataset
8. Train and evaluate the cost prediction model
9. Merge leak and cost predictions
10. Calculate expected leak costs
11. Generate candidate work units
12. Calculate work-unit replacement costs
13. Define heuristic priority rules
14. Simulate replacement decisions under the annual budget
15. Extend the simulation to the full 25-year horizon
16. Compare and improve heuristic strategies
17. Validate all project constraints
18. Generate the final submission

## Principle

At every stage:

**Understand → Implement → Inspect → Validate → Continue**

Existing team files may be used for validation, but the pipeline will be rebuilt from the original data.

# 1. Load and Understand the Original Data

Before merging, modeling, or optimizing anything, first understand the two original datasets:

- `pipes.csv`: pipe-level information
- `train.csv`: historical leak/cost events

Questions to answer:

1. What does one row represent in each dataset?
2. What is the shape of each dataset?
3. What columns are available?
4. What are the data types?
5. Are there missing values?
6. Are Pipe IDs unique?
7. How are `pipes.csv` and `train.csv` related?

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

In [8]:
DATA_DIR = Path("../../data")

pipes_path = DATA_DIR / "pipes.csv"
train_path = DATA_DIR / "train.csv"

print("pipes.csv exists:", pipes_path.exists())
print("train.csv exists:", train_path.exists())

pipes.csv exists: True
train.csv exists: True


In [9]:
pipes = pd.read_csv(pipes_path)
train = pd.read_csv(train_path)

print("pipes shape:", pipes.shape)
print("train shape:", train.shape)

pipes shape: (43039, 8)
train shape: (14113, 4)


In [10]:
print("=== pipes.csv columns ===")
print(pipes.columns.tolist())

print("\n=== pipes.csv first 5 rows ===")
display(pipes.head())

print("\n=== train.csv columns ===")
print(train.columns.tolist())

print("\n=== train.csv first 5 rows ===")
display(train.head())

=== pipes.csv columns ===
['Pipe ID', 'Lay date', 'Material', 'Surface', 'GPS x1', 'GPS y1', 'GPS x2', 'GPS y2']

=== pipes.csv first 5 rows ===


,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2
0,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97
1,P01002,1971-09-19,cast iron,water,6512.35,4147.58,6536.59,4201.94
2,P01003,1971-09-18,cast iron,water,6536.59,4201.94,6518.42,4201.93
3,P01004,1982-05-22,cast iron,structure,2836.68,3377.95,2848.80,3405.12
4,P01005,1971-07-03,cast iron,water,5117.28,4272.91,5126.37,4297.16



=== train.csv columns ===
['Pipe ID', 'Date', 'Time', 'Cost']

=== train.csv first 5 rows ===


,Pipe ID,Date,Time,Cost
0,P19711,2019-01-06,08:20,2074.90
1,P07628,2019-01-07,00:00,618.71
2,P31195,2019-04-16,03:48,1050.07
3,P15715,2019-04-16,16:51,1000.81
4,P10415,2019-04-17,22:08,3157.83


In [11]:
print("Number of rows in pipes.csv:", len(pipes))
print("Unique Pipe IDs in pipes.csv:", pipes["Pipe ID"].nunique())

print()

print("Number of rows in train.csv:", len(train))
print("Unique Pipe IDs in train.csv:", train["Pipe ID"].nunique())

Number of rows in pipes.csv: 43039
Unique Pipe IDs in pipes.csv: 43039

Number of rows in train.csv: 14113
Unique Pipe IDs in train.csv: 14113


In [12]:
print("Repeated leak records in train.csv:")
print(len(train) - train["Pipe ID"].nunique())

Repeated leak records in train.csv:
0


In [13]:
train_not_in_pipes = set(train["Pipe ID"]) - set(pipes["Pipe ID"])

print("Train Pipe IDs not found in pipes.csv:", len(train_not_in_pipes))

Train Pipe IDs not found in pipes.csv: 0


In [14]:
pipes_without_leak = set(pipes["Pipe ID"]) - set(train["Pipe ID"])

print("Pipes without a historical leak record:", len(pipes_without_leak))

Pipes without a historical leak record: 28926


In [15]:
print("=== pipes.csv data types ===")
print(pipes.dtypes)

print("\n=== train.csv data types ===")
print(train.dtypes)

=== pipes.csv data types ===
Pipe ID      object
Lay date     object
Material     object
Surface      object
GPS x1      float64
GPS y1      float64
GPS x2      float64
GPS y2      float64
dtype: object

=== train.csv data types ===
Pipe ID     object
Date        object
Time        object
Cost       float64
dtype: object


In [16]:
print("=== Missing values in pipes.csv ===")
print(pipes.isna().sum())

print("\n=== Missing values in train.csv ===")
print(train.isna().sum())

=== Missing values in pipes.csv ===
Pipe ID      0
Lay date    15
Material     8
Surface      0
GPS x1       0
GPS y1       0
GPS x2       0
GPS y2       0
dtype: int64

=== Missing values in train.csv ===
Pipe ID    0
Date       0
Time       0
Cost       0
dtype: int64


In [17]:
missing_pipes = pipes[
    pipes["Lay date"].isna() |
    pipes["Material"].isna()
]

print("Number of pipes with at least one missing value:", len(missing_pipes))

display(missing_pipes)

Number of pipes with at least one missing value: 23


,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2
8985,P09986,1962-02-20,NaN,water,584.00,2499.90,577.27,2502.69
8986,P09987,1947-11-02,NaN,water,595.04,2497.15,606.16,2490.79
9868,P10869,NaN,wrought iron,water,998.44,4234.71,1004.06,4237.59
9869,P10870,NaN,wrought iron,water,998.44,4234.71,994.69,4236.88
9870,P10871,NaN,wrought iron,water,1087.96,4280.97,1087.76,4276.32
9871,P10872,NaN,wrought iron,grassland,1087.96,4280.97,1089.18,4285.56
9872,P10873,NaN,wrought iron,water,1087.96,4280.97,1082.37,4278.07
9873,P10874,NaN,wrought iron,grassland,1126.90,4427.72,1128.79,4434.31
9874,P10875,NaN,wrought iron,grassland,1125.68,4423.15,1128.61,4434.70
11274,P12275,NaN,wrought iron,grassland,2588.77,3203.34,2591.18,3209.41


In [18]:
missing_pipe_ids = set(missing_pipes["Pipe ID"])
train_pipe_ids = set(train["Pipe ID"])

missing_with_leak = missing_pipe_ids & train_pipe_ids

print("Missing-data pipes with a historical leak:", len(missing_with_leak))
print("Pipe IDs:", sorted(missing_with_leak))

Missing-data pipes with a historical leak: 6
Pipe IDs: ['P10870', 'P10871', 'P10872', 'P10873', 'P10874', 'P10875']


In [19]:
pipes_clean = pipes.copy()
train_clean = train.copy()

In [20]:
pipes_clean["Lay date"] = pd.to_datetime(
    pipes_clean["Lay date"],
    errors="coerce"
)

train_clean["Date"] = pd.to_datetime(
    train_clean["Date"],
    errors="coerce"
)

In [21]:
print("Lay date dtype:", pipes_clean["Lay date"].dtype)
print("Leak Date dtype:", train_clean["Date"].dtype)

print()
print("Missing Lay dates:", pipes_clean["Lay date"].isna().sum())
print("Missing Leak dates:", train_clean["Date"].isna().sum())

Lay date dtype: datetime64[ns]
Leak Date dtype: datetime64[ns]

Missing Lay dates: 15
Missing Leak dates: 0


In [22]:
errors="coerce"

In [23]:
# Check the installation-date and historical leak-date ranges

print("Pipe installation date range:")
print(pipes_clean["Lay date"].min(), "to", pipes_clean["Lay date"].max())

print("\nHistorical leak date range:")
print(train_clean["Date"].min(), "to", train_clean["Date"].max())

Pipe installation date range:
1945-01-01 00:00:00 to 1985-12-28 00:00:00

Historical leak date range:
2019-01-06 00:00:00 to 2026-12-31 00:00:00


In [24]:
# Check the unique categories and counts for Material and Surface

print("=== Material ===")
print(pipes_clean["Material"].value_counts(dropna=False))

print("\n=== Surface ===")
print(pipes_clean["Surface"].value_counts(dropna=False))

=== Material ===
Material
wrought iron    24562
cast iron       10630
gray iron        3263
polyurethane     1678
copper           1613
brass            1285
NaN                 8
Name: count, dtype: int64

=== Surface ===
Surface
grassland    13507
water        12824
farmland      6531
structure     5680
road          2324
swamp         2173
Name: count, dtype: int64


## Section 1 Summary

- `pipes.csv` contains 43,039 unique physical pipes.
- `train.csv` contains 14,113 unique historical leak events.
- Every Pipe ID in `train.csv` exists in `pipes.csv`.
- 28,926 pipes have no historical leak record.
- No pipe has more than one historical leak event in the training data.
- Pipe installation dates range from 1945 to 1985.
- Historical leak events range from 2019 to 2026.
- 15 pipes have missing `Lay date`.
- 8 pipes have missing `Material`.
- 23 pipes have at least one missing value.
- 6 pipes with missing `Lay date` have a historical leak event.
- There are 38,455 iron pipes in the original network.
- `Lay date` and `Date` were converted to datetime format.
- The original raw DataFrames were preserved; transformations are made on copies.

# 2. Merge Pipe Attributes with Historical Leak Events

`train.csv` tells us when a leak occurred and how much it cost, but it does not describe the physical characteristics of the pipe.

`pipes.csv` contains those pipe characteristics.

Therefore, we merge the two datasets using `Pipe ID`.

The resulting dataset will have one row per historical leak event, together with the characteristics of the pipe that leaked.

In [25]:
# Merge each historical leak event with the attributes of its pipe

events = train_clean.merge(
    pipes_clean,
    on="Pipe ID",
    how="left",
    validate="one_to_one"
)

print("Merged dataset shape:", events.shape)

display(events.head())

Merged dataset shape: (14113, 11)


,Pipe ID,Date,Time,Cost,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2
0,P19711,2019-01-06,08:20,2074.90,1962-09-17,wrought iron,farmland,1821.64,3189.18,1789.59,3201.56
1,P07628,2019-01-07,00:00,618.71,1965-03-30,gray iron,grassland,4062.27,2606.80,4007.00,2588.85
2,P31195,2019-04-16,03:48,1050.07,1948-04-13,wrought iron,grassland,3009.11,2473.32,3005.81,2458.59
3,P15715,2019-04-16,16:51,1000.81,1965-12-16,wrought iron,structure,3050.60,3478.42,3051.62,3482.92
4,P10415,2019-04-17,22:08,3157.83,1953-06-13,wrought iron,swamp,3573.19,4657.26,3591.42,4648.12


In [26]:
# Validate that the merge preserved all historical leak events correctly

print("Rows before merge:", len(train_clean))
print("Rows after merge:", len(events))

print("\nUnique Pipe IDs:", events["Pipe ID"].nunique())

print("\nMissing values after merge:")
print(events.isna().sum())

Rows before merge: 14113
Rows after merge: 14113

Unique Pipe IDs: 14113

Missing values after merge:
Pipe ID     0
Date        0
Time        0
Cost        0
Lay date    6
Material    0
Surface     0
GPS x1      0
GPS y1      0
GPS x2      0
GPS y2      0
dtype: int64


# 3. Basic Feature Engineering

The raw data describes each pipe, but several useful model features must be derived from the original columns.

We will first create simple and interpretable features:

1. Pipe age at the time of the leak
2. Pipe length
3. Pipe midpoint coordinates
4. Calendar features from the leak date and time

These features will later be used in the leak and cost prediction pipelines.

In [27]:
# Calculate each pipe's age in years when the historical leak occurred

events["Age at leak"] = (
    (events["Date"] - events["Lay date"]).dt.days / 365.25
)

print(events["Age at leak"].describe())

count    14107.000000
mean        63.326879
std          5.251382
min         34.242300
25%         62.483231
50%         63.471595
75%         64.372348
max         81.771389
Name: Age at leak, dtype: float64


In [28]:
# Calculate pipe length from the two endpoint coordinates

events["Pipe length"] = np.sqrt(
    (events["GPS x2"] - events["GPS x1"]) ** 2
    + (events["GPS y2"] - events["GPS y1"]) ** 2
)

print(events["Pipe length"].describe())

count    14113.000000
mean        19.402756
std         16.960719
min          0.170000
25%          8.387670
50%         11.613484
75%         25.670000
max         74.555194
Name: Pipe length, dtype: float64


In [29]:
# Calculate the midpoint coordinates of each pipe

events["Midpoint x"] = (
    events["GPS x1"] + events["GPS x2"]
) / 2

events["Midpoint y"] = (
    events["GPS y1"] + events["GPS y2"]
) / 2

display(
    events[
        ["Pipe ID", "GPS x1", "GPS y1",
         "GPS x2", "GPS y2", "Midpoint x", "Midpoint y"]
    ].head()
)

,Pipe ID,GPS x1,GPS y1,GPS x2,GPS y2,Midpoint x,Midpoint y
0,P19711,1821.64,3189.18,1789.59,3201.56,1805.615,3195.370
1,P07628,4062.27,2606.80,4007.00,2588.85,4034.635,2597.825
2,P31195,3009.11,2473.32,3005.81,2458.59,3007.460,2465.955
3,P15715,3050.60,3478.42,3051.62,3482.92,3051.110,3480.670
4,P10415,3573.19,4657.26,3591.42,4648.12,3582.305,4652.690


In [30]:
# Extract calendar features from the historical leak date

events["Leak year"] = events["Date"].dt.year
events["Leak month"] = events["Date"].dt.month
events["Leak day of week"] = events["Date"].dt.dayofweek

display(
    events[
        ["Pipe ID", "Date",
         "Leak year", "Leak month", "Leak day of week"]
    ].head()
)

,Pipe ID,Date,Leak year,Leak month,Leak day of week
0,P19711,2019-01-06,2019,1,6
1,P07628,2019-01-07,2019,1,0
2,P31195,2019-04-16,2019,4,1
3,P15715,2019-04-16,2019,4,1
4,P10415,2019-04-17,2019,4,2


In [31]:
# Extract the hour from the historical leak time

events["Leak hour"] = pd.to_datetime(
    events["Time"],
    format="%H:%M"
).dt.hour

display(
    events[
        ["Pipe ID", "Time", "Leak hour"]
    ].head()
)

,Pipe ID,Time,Leak hour
0,P19711,08:20,8
1,P07628,00:00,0
2,P31195,03:48,3
3,P15715,16:51,16
4,P10415,22:08,22


### Feature Availability Note

A feature should only be used for prediction if it would be available at the time the prediction is made.

Historical event features such as leak month, weekday, and hour are known for past leak events, but the exact date and time of a future leak are not known in advance.

Therefore, feature availability must be reconsidered separately for:

- leak prediction,
- cost prediction,
- future 25-year simulation.

Creating a feature does not automatically mean that it is valid for every prediction task.

# 4. Build the Leak Prediction Dataset

The leak prediction problem is a binary classification problem.

For pipe `i` in year `t`:

- `Y = 1` if the pipe experiences its first leak during year `t`.
- `Y = 0` if the pipe is still at risk but does not leak during year `t`.

Each observation therefore represents one **pipe-year**.

After a pipe experiences its first leak, it is removed from the future risk set because it is considered replaced.

The historical observation period is 2019–2026.

In [32]:
# Create a lookup table containing the first observed leak year for each leaking pipe

leak_lookup = train_clean[["Pipe ID", "Date"]].copy()
leak_lookup["Leak year"] = leak_lookup["Date"].dt.year

leak_lookup = leak_lookup[
    ["Pipe ID", "Leak year"]
]

display(leak_lookup.head())

print("\nNumber of leaking pipes:", len(leak_lookup))
print("Leak year range:",
      leak_lookup["Leak year"].min(),
      "to",
      leak_lookup["Leak year"].max())

,Pipe ID,Leak year
0,P19711,2019
1,P07628,2019
2,P31195,2019
3,P15715,2019
4,P10415,2019



Number of leaking pipes: 14113
Leak year range: 2019 to 2026


In [33]:
# Attach the observed leak year to every pipe in the network

pipes_with_leak = pipes_clean.merge(
    leak_lookup,
    on="Pipe ID",
    how="left",
    validate="one_to_one"
)

print("Total pipes:", len(pipes_with_leak))
print("Pipes with observed leak:",
      pipes_with_leak["Leak year"].notna().sum())
print("Pipes without observed leak:",
      pipes_with_leak["Leak year"].isna().sum())

display(
    pipes_with_leak[
        ["Pipe ID", "Lay date", "Material", "Leak year"]
    ].head()
)

Total pipes: 43039
Pipes with observed leak: 14113
Pipes without observed leak: 28926


,Pipe ID,Lay date,Material,Leak year
0,P01001,1955-03-11,copper,NaN
1,P01002,1971-09-19,cast iron,2023.0
2,P01003,1971-09-18,cast iron,2024.0
3,P01004,1982-05-22,cast iron,NaN
4,P01005,1971-07-03,cast iron,2024.0


In [34]:
# Create one row for each year that a pipe remains at risk of its first leak

risk_rows = []

for _, pipe in pipes_with_leak.iterrows():
    leak_year = pipe["Leak year"]

    for year in range(2019, 2027):

        # Stop after the year of the first observed leak
        if pd.notna(leak_year) and year > leak_year:
            break

        row = pipe.to_dict()
        row["Year"] = year

        # Target: 1 if the pipe leaks in this year, otherwise 0
        row["Y"] = int(
            pd.notna(leak_year) and year == leak_year
        )

        risk_rows.append(row)

risk_set = pd.DataFrame(risk_rows)

print("Risk-set shape:", risk_set.shape)
print("Number of pipe-year observations:", len(risk_set))
print("Number of leak observations (Y=1):", risk_set["Y"].sum())
print("Number of non-leak observations (Y=0):", (risk_set["Y"] == 0).sum())

Risk-set shape: (305561, 11)
Number of pipe-year observations: 305561
Number of leak observations (Y=1): 14113
Number of non-leak observations (Y=0): 291448


In [35]:
# Verify the pipe-year risk-set logic using several example pipes

example_pipes = ["P01001", "P01002", "P01003", "P01004"]

display(
    risk_set.loc[
        risk_set["Pipe ID"].isin(example_pipes),
        ["Pipe ID", "Year", "Leak year", "Y"]
    ].sort_values(["Pipe ID", "Year"])
)

,Pipe ID,Year,Leak year,Y
0,P01001,2019,NaN,0
1,P01001,2020,NaN,0
2,P01001,2021,NaN,0
3,P01001,2022,NaN,0
4,P01001,2023,NaN,0
5,P01001,2024,NaN,0
6,P01001,2025,NaN,0
7,P01001,2026,NaN,0
8,P01002,2019,2023.0,0
9,P01002,2020,2023.0,0


In [36]:
# Calculate pipe age at the beginning of each pipe-year

year_start = pd.to_datetime(
    risk_set["Year"].astype(str) + "-01-01"
)

risk_set["Age"] = (
    (year_start - risk_set["Lay date"]).dt.days / 365.25
)

print(risk_set["Age"].describe())

print("\nMissing Age values:")
print(risk_set["Age"].isna().sum())

count    305455.000000
mean         64.019572
std           9.239507
min          33.010267
25%          58.042437
50%          63.178645
75%          71.832991
max          80.999316
Name: Age, dtype: float64

Missing Age values:
106


In [37]:
# Verify that pipe age increases correctly across years

display(
    risk_set.loc[
        risk_set["Pipe ID"].isin(["P01001", "P01002"]),
        ["Pipe ID", "Lay date", "Year", "Age", "Y"]
    ].sort_values(["Pipe ID", "Year"])
)

,Pipe ID,Lay date,Year,Age,Y
0,P01001,1955-03-11,2019,63.811088,0
1,P01001,1955-03-11,2020,64.810404,0
2,P01001,1955-03-11,2021,65.812457,0
3,P01001,1955-03-11,2022,66.811773,0
4,P01001,1955-03-11,2023,67.811088,0
5,P01001,1955-03-11,2024,68.810404,0
6,P01001,1955-03-11,2025,69.812457,0
7,P01001,1955-03-11,2026,70.811773,0
8,P01002,1971-09-19,2019,47.285421,0
9,P01002,1971-09-19,2020,48.284736,0


In [38]:
# Compare pipe age between leak and non-leak pipe-year observations

age_by_target = (
    risk_set
    .groupby("Y")["Age"]
    .agg(["count", "mean", "median", "std"])
)

display(age_by_target)

,count,mean,median,std
Y,,,,
0,291348,64.079721,63.808350,9.385497
1,14107,62.777328,62.915811,5.253238


In [39]:
# Check the class distribution of the leak-prediction target

class_counts = risk_set["Y"].value_counts().sort_index()
class_percent = risk_set["Y"].value_counts(
    normalize=True
).sort_index() * 100

class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percent": class_percent
})

display(class_summary)

,Count,Percent
Y,,
0,291448,95.381282
1,14113,4.618718


In [40]:
# Create pipe length and midpoint features for every pipe-year observation

risk_set["Pipe length"] = np.sqrt(
    (risk_set["GPS x2"] - risk_set["GPS x1"]) ** 2
    + (risk_set["GPS y2"] - risk_set["GPS y1"]) ** 2
)

risk_set["Midpoint x"] = (
    risk_set["GPS x1"] + risk_set["GPS x2"]
) / 2

risk_set["Midpoint y"] = (
    risk_set["GPS y1"] + risk_set["GPS y2"]
) / 2

print(
    risk_set[
        ["Age", "Pipe length", "Midpoint x", "Midpoint y"]
    ].isna().sum()
)

Age            106
Pipe length      0
Midpoint x       0
Midpoint y       0
dtype: int64


In [41]:
# Define the baseline leak-prediction features and inspect missing values

baseline_features = [
    "Age",
    "Material",
    "Surface",
    "Pipe length",
    "Midpoint x",
    "Midpoint y",
    "Year"
]

print("Missing values in baseline features:")
print(risk_set[baseline_features].isna().sum())

Missing values in baseline features:
Age            106
Material        64
Surface          0
Pipe length      0
Midpoint x       0
Midpoint y       0
Year             0
dtype: int64


In [42]:
# Create a model-ready baseline dataset without modifying the original risk set

leak_baseline = risk_set.copy()

# Keep pipes with unknown material as a separate category
leak_baseline["Material"] = (
    leak_baseline["Material"].fillna("Unknown")
)

# Remove pipe-year observations where age cannot be calculated
leak_baseline = leak_baseline.dropna(
    subset=["Age"]
).copy()

print("Original risk-set rows:", len(risk_set))
print("Baseline model rows:", len(leak_baseline))
print("Rows removed:", len(risk_set) - len(leak_baseline))

print("\nMissing values after handling:")
print(leak_baseline[baseline_features].isna().sum())

print("\nTarget counts:")
print(leak_baseline["Y"].value_counts().sort_index())

Original risk-set rows: 305561
Baseline model rows: 305455
Rows removed: 106

Missing values after handling:
Age            0
Material       0
Surface        0
Pipe length    0
Midpoint x     0
Midpoint y     0
Year           0
dtype: int64

Target counts:
Y
0    291348
1     14107
Name: count, dtype: int64


In [43]:
# Summarize the number of at-risk pipe-years and observed leaks by year

year_summary = (
    leak_baseline
    .groupby("Year")
    .agg(
        observations=("Y", "size"),
        leaks=("Y", "sum")
    )
)

year_summary["non_leaks"] = (
    year_summary["observations"] - year_summary["leaks"]
)

year_summary["leak_rate"] = (
    year_summary["leaks"] / year_summary["observations"]
)

display(year_summary)

,observations,leaks,non_leaks,leak_rate
Year,,,,
2019,43024,1078,41946,0.025056
2020,41946,703,41243,0.016760
2021,41243,866,40377,0.020998
2022,40377,2874,37503,0.071179
2023,37503,1824,35679,0.048636
2024,35679,1951,33728,0.054682
2025,33728,1773,31955,0.052568
2026,31955,3038,28917,0.095071


In [44]:
# Define expanding-window temporal validation folds

temporal_folds = [
    {
        "train_years": list(range(2019, 2023)),
        "test_year": 2023
    },
    {
        "train_years": list(range(2019, 2024)),
        "test_year": 2024
    },
    {
        "train_years": list(range(2019, 2025)),
        "test_year": 2025
    },
    {
        "train_years": list(range(2019, 2026)),
        "test_year": 2026
    }
]

for i, fold in enumerate(temporal_folds, start=1):
    print(
        f"Fold {i}: "
        f"{fold['train_years'][0]}-{fold['train_years'][-1]}"
        f" -> {fold['test_year']}"
    )

Fold 1: 2019-2022 -> 2023
Fold 2: 2019-2023 -> 2024
Fold 3: 2019-2024 -> 2025
Fold 4: 2019-2025 -> 2026


In [45]:
# Check the training and test sizes for each temporal fold

fold_summary = []

for i, fold in enumerate(temporal_folds, start=1):

    train_fold = leak_baseline[
        leak_baseline["Year"].isin(fold["train_years"])
    ]

    test_fold = leak_baseline[
        leak_baseline["Year"] == fold["test_year"]
    ]

    fold_summary.append({
        "Fold": i,
        "Train years": (
            f"{fold['train_years'][0]}-"
            f"{fold['train_years'][-1]}"
        ),
        "Test year": fold["test_year"],
        "Train rows": len(train_fold),
        "Train leaks": int(train_fold["Y"].sum()),
        "Test rows": len(test_fold),
        "Test leaks": int(test_fold["Y"].sum())
    })

fold_summary = pd.DataFrame(fold_summary)

display(fold_summary)

,Fold,Train years,Test year,Train rows,Train leaks,Test rows,Test leaks
0,1,2019-2022,2023,166590,5521,37503,1824
1,2,2019-2023,2024,204093,7345,35679,1951
2,3,2019-2024,2025,239772,9296,33728,1773
3,4,2019-2025,2026,273500,11069,31955,3038


In [46]:
# Separate numerical and categorical predictors for preprocessing

numeric_features = [
    "Age",
    "Pipe length",
    "Midpoint x",
    "Midpoint y",
    "Year"
]

categorical_features = [
    "Material",
    "Surface"
]

target = "Y"

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTarget:")
print(target)

Numeric features:
['Age', 'Pipe length', 'Midpoint x', 'Midpoint y', 'Year']

Categorical features:
['Material', 'Surface']

Target:
Y


In [47]:
# Import tools for categorical encoding and preprocessing

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [48]:
# One-hot encode categorical features while keeping numerical features unchanged

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

preprocessor

,transformers,"[('categorical', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


In [49]:
# Import the baseline classifier, pipeline, and evaluation metrics

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)

In [50]:
# Combine preprocessing and logistic regression into one modeling pipeline

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

logistic_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categorical', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [51]:
# Create the first temporal training and test sets: 2019-2022 -> 2023

train_fold1 = leak_baseline[
    leak_baseline["Year"].between(2019, 2022)
].copy()

test_fold1 = leak_baseline[
    leak_baseline["Year"] == 2023
].copy()

X_train = train_fold1[
    numeric_features + categorical_features
]

y_train = train_fold1["Y"]

X_test = test_fold1[
    numeric_features + categorical_features
]

y_test = test_fold1["Y"]

print("Training rows:", len(X_train))
print("Training leaks:", y_train.sum())

print("\nTest rows:", len(X_test))
print("Test leaks:", y_test.sum())

Training rows: 166590
Training leaks: 5521

Test rows: 37503
Test leaks: 1824


In [52]:
# Train the baseline logistic regression model using 2019-2022 data

logistic_pipeline.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


c:\Users\nsim4\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [53]:
# Predict the probability of a leak for each at-risk pipe in 2023

y_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Number of predictions:", len(y_prob))
print("Minimum probability:", y_prob.min())
print("Maximum probability:", y_prob.max())
print("Mean probability:", y_prob.mean())

Number of predictions: 37503
Minimum probability: 0.030648002284025633
Maximum probability: 0.9781132300117447
Mean probability: 0.3757198663448479


In [54]:
# Evaluate the 2023 predictions using probability and classification metrics

y_pred = (y_prob >= 0.5).astype(int)

fold1_metrics = {
    "PR-AUC": average_precision_score(y_test, y_prob),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "Brier": brier_score_loss(y_test, y_prob),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred)
}

for metric, value in fold1_metrics.items():
    print(f"{metric}: {value:.4f}")

PR-AUC: 0.0659
ROC-AUC: 0.5475
Brier: 0.1910
Precision: 0.0545
Recall: 0.3136
F1: 0.0929


In [55]:
# Evaluate the logistic-regression baseline across all four temporal folds

logistic_results = []

for i, fold in enumerate(temporal_folds, start=1):

    train_data = leak_baseline[
        leak_baseline["Year"].isin(fold["train_years"])
    ].copy()

    test_data = leak_baseline[
        leak_baseline["Year"] == fold["test_year"]
    ].copy()

    X_train = train_data[
        numeric_features + categorical_features
    ]
    y_train = train_data["Y"]

    X_test = test_data[
        numeric_features + categorical_features
    ]
    y_test = test_data["Y"]

    # Fit preprocessing and model using only the training years
    logistic_pipeline.fit(X_train, y_train)

    # Predict leak probabilities for the next year
    y_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

    # Convert probabilities to classes using the default 0.5 threshold
    y_pred = (y_prob >= 0.5).astype(int)

    logistic_results.append({
        "Fold": i,
        "Train years": (
            f"{fold['train_years'][0]}-"
            f"{fold['train_years'][-1]}"
        ),
        "Test year": fold["test_year"],
        "PR-AUC": average_precision_score(y_test, y_prob),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Brier": brier_score_loss(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Mean predicted probability": y_prob.mean(),
        "Actual leak rate": y_test.mean()
    })

logistic_results = pd.DataFrame(logistic_results)

display(logistic_results)

c:\Users\nsim4\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\nsim4\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://s

,Fold,Train years,Test year,PR-AUC,ROC-AUC,Brier,Precision,Recall,F1,Mean predicted probability,Actual leak rate
0,1,2019-2022,2023,0.065903,0.547468,0.190998,0.054502,0.313596,0.092865,0.375720,0.048636
1,2,2019-2023,2024,0.041116,0.344840,0.208213,0.040081,0.173244,0.065100,0.395218,0.054682
2,3,2019-2024,2025,0.041371,0.415349,0.213073,0.030239,0.138184,0.049620,0.430187,0.052568
3,4,2019-2025,2026,0.103762,0.555849,0.223003,0.112742,0.341343,0.169500,0.445473,0.095071


In [56]:
# Import XGBoost for the nonlinear leak-prediction baseline

from xgboost import XGBClassifier

print("XGBoost imported successfully.")

XGBoost imported successfully.


In [57]:
# Build an XGBoost pipeline using the same baseline features

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

xgb_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categorical', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [58]:
# Train XGBoost on 2019-2022 and predict leak probabilities for 2023

train_fold1 = leak_baseline[
    leak_baseline["Year"].between(2019, 2022)
].copy()

test_fold1 = leak_baseline[
    leak_baseline["Year"] == 2023
].copy()

X_train = train_fold1[
    numeric_features + categorical_features
]
y_train = train_fold1["Y"]

X_test = test_fold1[
    numeric_features + categorical_features
]
y_test = test_fold1["Y"]

xgb_pipeline.fit(X_train, y_train)

xgb_prob = xgb_pipeline.predict_proba(X_test)[:, 1]

print("Number of predictions:", len(xgb_prob))
print("Minimum probability:", xgb_prob.min())
print("Maximum probability:", xgb_prob.max())
print("Mean probability:", xgb_prob.mean())
print("Actual leak rate:", y_test.mean())

Number of predictions: 37503
Minimum probability: 0.00013830973
Maximum probability: 0.84848696
Mean probability: 0.051554516
Actual leak rate: 0.0486361091112711


In [59]:
# Evaluate the 2023 XGBoost predictions and compare them with the logistic baseline

xgb_pred = (xgb_prob >= 0.5).astype(int)

xgb_fold1_metrics = {
    "PR-AUC": average_precision_score(y_test, xgb_prob),
    "ROC-AUC": roc_auc_score(y_test, xgb_prob),
    "Brier": brier_score_loss(y_test, xgb_prob),
    "Precision": precision_score(
        y_test, xgb_pred, zero_division=0
    ),
    "Recall": recall_score(y_test, xgb_pred),
    "F1": f1_score(y_test, xgb_pred)
}

comparison_fold1 = pd.DataFrame({
    "Logistic Regression": fold1_metrics,
    "XGBoost": xgb_fold1_metrics
})

display(comparison_fold1)

,Logistic Regression,XGBoost
PR-AUC,0.065903,0.330920
ROC-AUC,0.547468,0.887867
Brier,0.190998,0.036312
Precision,0.054502,0.424845
Recall,0.313596,0.375000
F1,0.092865,0.398369


In [60]:
# Evaluate the same XGBoost baseline across all temporal validation folds

xgb_results = []

for i, fold in enumerate(temporal_folds, start=1):

    train_data = leak_baseline[
        leak_baseline["Year"].isin(fold["train_years"])
    ].copy()

    test_data = leak_baseline[
        leak_baseline["Year"] == fold["test_year"]
    ].copy()

    X_train = train_data[
        numeric_features + categorical_features
    ]
    y_train = train_data["Y"]

    X_test = test_data[
        numeric_features + categorical_features
    ]
    y_test = test_data["Y"]

    # Fit preprocessing and XGBoost using only past years
    xgb_pipeline.fit(X_train, y_train)

    # Predict leak probabilities for the next unseen year
    y_prob = xgb_pipeline.predict_proba(X_test)[:, 1]

    # Use 0.5 only as a temporary classification threshold
    y_pred = (y_prob >= 0.5).astype(int)

    xgb_results.append({
        "Fold": i,
        "Train years": (
            f"{fold['train_years'][0]}-"
            f"{fold['train_years'][-1]}"
        ),
        "Test year": fold["test_year"],
        "PR-AUC": average_precision_score(y_test, y_prob),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Brier": brier_score_loss(y_test, y_prob),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Mean predicted probability": y_prob.mean(),
        "Actual leak rate": y_test.mean()
    })

xgb_results = pd.DataFrame(xgb_results)

display(xgb_results)

,Fold,Train years,Test year,PR-AUC,ROC-AUC,Brier,Precision,Recall,F1,Mean predicted probability,Actual leak rate
0,1,2019-2022,2023,0.330920,0.887867,0.036312,0.424845,0.375000,0.398369,0.051555,0.048636
1,2,2019-2023,2024,0.525517,0.901379,0.033979,0.699267,0.146591,0.242373,0.041859,0.054682
2,3,2019-2024,2025,0.250280,0.748836,0.043799,0.463768,0.144388,0.220215,0.031101,0.052568
3,4,2019-2025,2026,0.433486,0.877376,0.063528,0.486239,0.069783,0.122050,0.066851,0.095071


In [61]:
# Generate out-of-fold predictions for each temporal test year

xgb_oof_parts = []

for i, fold in enumerate(temporal_folds, start=1):

    train_data = leak_baseline[
        leak_baseline["Year"].isin(fold["train_years"])
    ].copy()

    test_data = leak_baseline[
        leak_baseline["Year"] == fold["test_year"]
    ].copy()

    X_train = train_data[
        numeric_features + categorical_features
    ]
    y_train = train_data["Y"]

    X_test = test_data[
        numeric_features + categorical_features
    ]

    # Fit using only years before the test year
    xgb_pipeline.fit(X_train, y_train)

    # Predict probabilities for the unseen test year
    test_data["predicted_probability"] = (
        xgb_pipeline.predict_proba(X_test)[:, 1]
    )

    # Record which temporal fold produced each prediction
    test_data["fold"] = i

    xgb_oof_parts.append(
        test_data[
            [
                "Pipe ID",
                "Year",
                "Y",
                "predicted_probability",
                "fold"
            ]
        ]
    )

xgb_oof = pd.concat(
    xgb_oof_parts,
    ignore_index=True
)

print("OOF rows:", len(xgb_oof))
print("OOF years:", sorted(xgb_oof["Year"].unique()))
print("Actual leaks:", xgb_oof["Y"].sum())

display(xgb_oof.head())

OOF rows: 138865
OOF years: [np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Actual leaks: 8586


,Pipe ID,Year,Y,predicted_probability,fold
0,P01001,2023,0,0.020856,1
1,P01002,2023,1,0.010684,1
2,P01003,2023,0,0.007354,1
3,P01004,2023,0,0.015491,1
4,P01005,2023,0,0.012260,1


In [62]:
# Calculate overall metrics across all out-of-fold predictions

oof_y = xgb_oof["Y"]
oof_prob = xgb_oof["predicted_probability"]

# Use 0.5 only as a temporary classification threshold
oof_pred = (oof_prob >= 0.5).astype(int)

xgb_oof_metrics = {
    "PR-AUC": average_precision_score(oof_y, oof_prob),
    "ROC-AUC": roc_auc_score(oof_y, oof_prob),
    "Brier": brier_score_loss(oof_y, oof_prob),
    "Precision": precision_score(
        oof_y, oof_pred, zero_division=0
    ),
    "Recall": recall_score(oof_y, oof_pred),
    "F1": f1_score(oof_y, oof_pred),
    "Mean predicted probability": oof_prob.mean(),
    "Actual leak rate": oof_y.mean()
}

for metric, value in xgb_oof_metrics.items():
    print(f"{metric}: {value:.4f}")

PR-AUC: 0.3756
ROC-AUC: 0.8652
Brier: 0.0438
Precision: 0.4782
Recall: 0.1675
F1: 0.2481
Mean predicted probability: 0.0476
Actual leak rate: 0.0618


## XGBoost Baseline — Temporal Validation Summary

The baseline XGBoost model was trained using the following features:

- Age
- Material
- Surface
- Pipe length
- Midpoint x
- Midpoint y
- Year

Model performance was evaluated using expanding-window temporal validation:

- 2019–2022 → 2023
- 2019–2023 → 2024
- 2019–2024 → 2025
- 2019–2025 → 2026

Predictions from the four unseen test years were combined to create 138,865 out-of-fold (OOF) predictions containing 8,586 observed leaks.

### Combined OOF Results

| Metric | Value |
|---|---:|
| PR-AUC | 0.3756 |
| ROC-AUC | 0.8652 |
| Brier Score | 0.0438 |
| Precision | 0.4782 |
| Recall | 0.1675 |
| F1 | 0.2481 |
| Mean predicted leak probability | 0.0476 |
| Actual leak rate | 0.0618 |

### Interpretation

The XGBoost baseline shows substantially better discrimination than the Logistic Regression baseline. In particular, the PR-AUC and ROC-AUC indicate that the model can meaningfully rank higher-risk pipes above lower-risk pipes.

The Brier score is relatively low, and the mean predicted probability (4.76%) is reasonably close to the overall observed leak rate (6.18%), although the model tends to underpredict the overall leak rate.

Precision, recall, and F1 shown above use a temporary threshold of 0.5. Because leak events are rare and the final optimization will use predicted probabilities rather than only hard class labels, the 0.5 threshold should not yet be treated as optimal.

The current model is intentionally a simple baseline. More informative historical and spatial features can be added later and evaluated using the same temporal validation framework.

In [63]:
# Create one pipe-level geometry table for later spatial feature engineering

pipe_geometry = pipes_clean.copy()

pipe_geometry["Pipe length"] = np.sqrt(
    (pipe_geometry["GPS x2"] - pipe_geometry["GPS x1"]) ** 2
    + (pipe_geometry["GPS y2"] - pipe_geometry["GPS y1"]) ** 2
)

pipe_geometry["Midpoint x"] = (
    pipe_geometry["GPS x1"] + pipe_geometry["GPS x2"]
) / 2

pipe_geometry["Midpoint y"] = (
    pipe_geometry["GPS y1"] + pipe_geometry["GPS y2"]
) / 2

print("Number of physical pipes:", len(pipe_geometry))

print("\nMissing midpoint coordinates:")
print(
    pipe_geometry[
        ["Midpoint x", "Midpoint y"]
    ].isna().sum()
)

display(
    pipe_geometry[
        ["Pipe ID", "Midpoint x", "Midpoint y", "Pipe length"]
    ].head()
)

Number of physical pipes: 43039

Missing midpoint coordinates:
Midpoint x    0
Midpoint y    0
dtype: int64


,Pipe ID,Midpoint x,Midpoint y,Pipe length
0,P01001,6130.220,5020.970,12.140000
1,P01002,6524.470,4174.760,59.519637
2,P01003,6527.505,4201.935,18.170003
3,P01004,2842.740,3391.535,29.750686
4,P01005,5121.825,4285.035,25.897695


pipe_geometry
1 row = physical pipe
43,039 rows
        │
        └── spatial relationships calculation


risk_set
1 row = pipe-year
305,561 rows
        │
        └── prediction observations

In [64]:
# Measure the distance from each pipe midpoint to its nearest other pipe

from sklearn.neighbors import NearestNeighbors

coords = pipe_geometry[
    ["Midpoint x", "Midpoint y"]
].to_numpy()

nearest_model = NearestNeighbors(
    n_neighbors=2,
    algorithm="kd_tree"
)

nearest_model.fit(coords)

distances, indices = nearest_model.kneighbors(coords)

# Column 0 is the pipe itself (distance = 0);
# column 1 is the nearest other pipe.
pipe_geometry["Nearest pipe distance"] = distances[:, 1]

print(
    pipe_geometry["Nearest pipe distance"].describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

count    43039.000000
mean         9.626393
std          9.333276
min          0.029155
25%          3.106614
50%          7.326404
75%         12.094331
90%         21.216197
95%         29.533939
99%         46.690041
max         73.210000
Name: Nearest pipe distance, dtype: float64


In [65]:
# Count how many other pipe midpoints fall within 25 m, 50 m, and 100 m

from sklearn.neighbors import KDTree

coords = pipe_geometry[
    ["Midpoint x", "Midpoint y"]
].to_numpy()

tree = KDTree(coords)

candidate_radii = [25, 50, 100]

neighbor_count_summary = []

for radius in candidate_radii:

    # Count all points within the radius
    counts = tree.query_radius(
        coords,
        r=radius,
        count_only=True
    )

    # Remove the pipe itself from its neighborhood count
    counts = counts - 1

    pipe_geometry[f"Neighbors within {radius}m"] = counts

    neighbor_count_summary.append({
        "Radius": radius,
        "Mean neighbors": counts.mean(),
        "Median neighbors": np.median(counts),
        "75%": np.percentile(counts, 75),
        "90%": np.percentile(counts, 90),
        "95%": np.percentile(counts, 95),
        "Max neighbors": counts.max()
    })

neighbor_count_summary = pd.DataFrame(
    neighbor_count_summary
)

display(neighbor_count_summary)

,Radius,Mean neighbors,Median neighbors,75%,90%,95%,Max neighbors
0,25,11.263226,8.0,17.0,26.0,32.0,60
1,50,31.170984,24.0,42.0,69.0,87.0,148
2,100,91.575548,75.0,127.0,185.0,218.0,360


In [67]:
# Build reusable neighbor lists for each pipe at 25 m, 50 m, and 100 m

pipe_ids = pipe_geometry["Pipe ID"].to_numpy()

neighbor_lists = {}

for radius in candidate_radii:

    # Find indices of all pipe midpoints within the radius
    neighbor_indices = tree.query_radius(
        coords,
        r=radius
    )

    # Remove each pipe itself from its own neighbor list
    neighbor_lists[radius] = [
        indices[indices != i]
        for i, indices in enumerate(neighbor_indices)
    ]

    print(
        f"{radius}m neighbor lists created "
        f"for {len(neighbor_lists[radius])} pipes."
    )

25m neighbor lists created for 43039 pipes.
50m neighbor lists created for 43039 pipes.
100m neighbor lists created for 43039 pipes.


In [68]:
# Inspect the neighboring pipes around one example pipe

example_index = 0
example_pipe = pipe_ids[example_index]

print("Example Pipe ID:", example_pipe)

for radius in candidate_radii:

    indices = neighbor_lists[radius][example_index]
    nearby_pipe_ids = pipe_ids[indices]

    print(
        f"\nNeighbors within {radius}m: "
        f"{len(nearby_pipe_ids)}"
    )

    print(nearby_pipe_ids[:10])

Example Pipe ID: P01001

Neighbors within 25m: 1
['P06375']

Neighbors within 50m: 6
['P09311' 'P01712' 'P04437' 'P06375' 'P01715' 'P01714']

Neighbors within 100m: 15
['P09311' 'P09319' 'P09318' 'P09313' 'P09312' 'P05307' 'P09310' 'P01712'
 'P04437' 'P06375']


In [71]:
# Create a lookup from each pipe to its observed first-leak year

leak_year_lookup = (
    events[
        ["Pipe ID", "Leak year"]
    ]
    .set_index("Pipe ID")["Leak year"]
    .to_dict()
)

print(
    "Number of pipes with observed leak years:",
    len(leak_year_lookup)
)

# Check a few known examples
for pipe_id in ["P01001", "P01002", "P01003"]:
    print(
        pipe_id,
        "->",
        leak_year_lookup.get(pipe_id)
    )

Number of pipes with observed leak years: 14113
P01001 -> None
P01002 -> 2023
P01003 -> 2024


In [72]:
# Count neighboring pipes that leaked strictly before each prediction year

pipe_id_to_index = {
    pipe_id: i
    for i, pipe_id in enumerate(pipe_ids)
}


def count_prior_neighbor_leaks(pipe_id, year, radius):
    """Count neighboring pipes that leaked before the given year."""

    pipe_index = pipe_id_to_index[pipe_id]

    neighbor_indices = neighbor_lists[radius][pipe_index]
    neighbor_pipe_ids = pipe_ids[neighbor_indices]

    count = 0

    for neighbor_id in neighbor_pipe_ids:

        leak_year = leak_year_lookup.get(neighbor_id)

        # Use only information available before the prediction year
        if leak_year is not None and leak_year < year:
            count += 1

    return count


for radius in candidate_radii:

    feature_name = f"Prior leaks within {radius}m"

    risk_set[feature_name] = [
        count_prior_neighbor_leaks(
            pipe_id,
            year,
            radius
        )
        for pipe_id, year in zip(
            risk_set["Pipe ID"],
            risk_set["Year"]
        )
    ]

print("Historical spatial leak features created.")

Historical spatial leak features created.


In [73]:
# Inspect the new historical spatial leak features

spatial_features = [
    "Prior leaks within 25m",
    "Prior leaks within 50m",
    "Prior leaks within 100m"
]

display(
    risk_set[
        [
            "Pipe ID",
            "Year",
            "Y"
        ] + spatial_features
    ].head(20)
)

print("\nSpatial feature summary:")

display(
    risk_set[spatial_features].describe()
)

,Pipe ID,Year,Y,Prior leaks within 25m,Prior leaks within 50m,Prior leaks within 100m
0,P01001,2019,0,0,0,0
1,P01001,2020,0,0,1,2
2,P01001,2021,0,0,2,3
3,P01001,2022,0,0,2,4
4,P01001,2023,0,0,2,4
5,P01001,2024,0,0,2,4
6,P01001,2025,0,0,2,4
7,P01001,2026,0,0,2,4
8,P01002,2019,0,0,0,0
9,P01002,2020,0,0,0,1



Spatial feature summary:


,Prior leaks within 25m,Prior leaks within 50m,Prior leaks within 100m
count,305561.000000,305561.000000,305561.000000
mean,0.597242,1.758006,5.516565
std,1.812762,4.417966,11.835146
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,1.000000
75%,0.000000,2.000000,6.000000
max,38.000000,81.000000,187.000000


In [74]:
# Manually verify which neighboring pipes contribute to P01003's 2024 history features

audit_pipe = "P01003"
audit_year = 2024

audit_index = pipe_id_to_index[audit_pipe]

print(
    f"Audit pipe: {audit_pipe}, "
    f"prediction year: {audit_year}"
)

for radius in candidate_radii:

    neighbor_indices = neighbor_lists[radius][audit_index]
    neighbor_pipe_ids = pipe_ids[neighbor_indices]

    contributing_neighbors = []

    for neighbor_id in neighbor_pipe_ids:

        leak_year = leak_year_lookup.get(neighbor_id)

        # Only leaks strictly before 2024 may contribute
        if leak_year is not None and leak_year < audit_year:
            contributing_neighbors.append(
                (neighbor_id, int(leak_year))
            )

    print(f"\n{radius}m:")
    print(
        "Number of prior leaking neighbors:",
        len(contributing_neighbors)
    )
    print(
        "Contributing neighbors:",
        contributing_neighbors
    )

Audit pipe: P01003, prediction year: 2024

25m:
Number of prior leaking neighbors: 0
Contributing neighbors: []

50m:
Number of prior leaking neighbors: 1
Contributing neighbors: [('P01002', 2023)]

100m:
Number of prior leaking neighbors: 3
Contributing neighbors: [('P01002', 2023), ('P08754', 2019), ('P03493', 2023)]


In [75]:
# Create a modeling dataset that adds historical spatial leak features
# while keeping the same missing-value handling as the baseline model

spatial_model_data = risk_set.dropna(
    subset=[
        "Age",
        "Material",
        "Surface",
        "Pipe length",
        "Midpoint x",
        "Midpoint y"
    ]
).copy()

spatial_numeric_features = [
    "Age",
    "Pipe length",
    "Midpoint x",
    "Midpoint y",
    "Year",
    "Prior leaks within 25m",
    "Prior leaks within 50m",
    "Prior leaks within 100m"
]

spatial_categorical_features = [
    "Material",
    "Surface"
]

print("Baseline rows:", len(leak_baseline))
print("Spatial model rows:", len(spatial_model_data))

print("\nSpatial features:")
for feature in spatial_features:
    print("-", feature)

Baseline rows: 305455
Spatial model rows: 305391

Spatial features:
- Prior leaks within 25m
- Prior leaks within 50m
- Prior leaks within 100m


In [76]:
# Create preprocessing for the baseline features plus spatial history features

spatial_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            spatial_categorical_features
        ),
        (
            "numeric",
            "passthrough",
            spatial_numeric_features
        )
    ]
)

spatial_xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", spatial_preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Spatial XGBoost pipeline created.")

Spatial XGBoost pipeline created.


In [77]:
# Evaluate XGBoost with spatial history using the same four temporal folds

spatial_oof_parts = []
spatial_fold_results = []

for i, fold in enumerate(temporal_folds, start=1):

    train_data = spatial_model_data[
        spatial_model_data["Year"].isin(fold["train_years"])
    ].copy()

    test_data = spatial_model_data[
        spatial_model_data["Year"] == fold["test_year"]
    ].copy()

    X_train = train_data[
        spatial_numeric_features
        + spatial_categorical_features
    ]
    y_train = train_data["Y"]

    X_test = test_data[
        spatial_numeric_features
        + spatial_categorical_features
    ]
    y_test = test_data["Y"]

    # Fit only on years before the test year
    spatial_xgb_pipeline.fit(
        X_train,
        y_train
    )

    # Predict probabilities for the unseen year
    y_prob = spatial_xgb_pipeline.predict_proba(
        X_test
    )[:, 1]

    y_pred = (y_prob >= 0.5).astype(int)

    spatial_fold_results.append({
        "Fold": i,
        "Test year": fold["test_year"],
        "PR-AUC": average_precision_score(
            y_test, y_prob
        ),
        "ROC-AUC": roc_auc_score(
            y_test, y_prob
        ),
        "Brier": brier_score_loss(
            y_test, y_prob
        ),
        "Precision": precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            y_pred
        ),
        "F1": f1_score(
            y_test,
            y_pred
        )
    })

    test_predictions = test_data[
        ["Pipe ID", "Year", "Y"]
    ].copy()

    test_predictions[
        "predicted_probability"
    ] = y_prob

    test_predictions["fold"] = i

    spatial_oof_parts.append(
        test_predictions
    )

spatial_fold_results = pd.DataFrame(
    spatial_fold_results
)

spatial_oof = pd.concat(
    spatial_oof_parts,
    ignore_index=True
)

display(spatial_fold_results)

print("\nOOF rows:", len(spatial_oof))

,Fold,Test year,PR-AUC,ROC-AUC,Brier,Precision,Recall,F1
0,1,2023,0.332024,0.888540,0.036428,0.410326,0.331140,0.366505
1,2,2024,0.471185,0.901133,0.034740,0.480769,0.012814,0.024963
2,3,2025,0.248923,0.765450,0.043518,0.427632,0.036661,0.067532
3,4,2026,0.447023,0.877869,0.062960,0.569697,0.092824,0.159638



OOF rows: 138833


In [78]:
# Compare combined OOF performance before and after adding spatial history

spatial_y = spatial_oof["Y"]
spatial_prob = spatial_oof[
    "predicted_probability"
]

spatial_pred = (
    spatial_prob >= 0.5
).astype(int)

spatial_oof_metrics = {
    "PR-AUC": average_precision_score(
        spatial_y,
        spatial_prob
    ),
    "ROC-AUC": roc_auc_score(
        spatial_y,
        spatial_prob
    ),
    "Brier": brier_score_loss(
        spatial_y,
        spatial_prob
    ),
    "Precision": precision_score(
        spatial_y,
        spatial_pred,
        zero_division=0
    ),
    "Recall": recall_score(
        spatial_y,
        spatial_pred
    ),
    "F1": f1_score(
        spatial_y,
        spatial_pred
    ),
    "Mean predicted probability":
        spatial_prob.mean(),
    "Actual leak rate":
        spatial_y.mean()
}

model_comparison = pd.DataFrame({
    "Baseline XGBoost":
        xgb_oof_metrics,
    "XGBoost + Spatial History":
        spatial_oof_metrics
})

display(model_comparison)

,Baseline XGBoost,XGBoost + Spatial History
PR-AUC,0.375631,0.372168
ROC-AUC,0.865155,0.866849
Brier,0.043794,0.043822
Precision,0.478217,0.449562
Recall,0.167482,0.113673
F1,0.248081,0.181463
Mean predicted probability,0.047616,0.047455
Actual leak rate,0.061830,0.061844


In [ ]:
# Create preprocessing for the baseline features plus spatial history features

spatial_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            spatial_categorical_features
        ),
        (
            "numeric",
            "passthrough",
            spatial_numeric_features
        )
    ]
)

spatial_xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", spatial_preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Spatial XGBoost pipeline created.")

Spatial XGBoost pipeline created.


In [79]:
# Rebuild the spatial model dataset using exactly the same rows
# as the original baseline dataset for a fair comparison

spatial_model_data = leak_baseline.copy()

spatial_model_data[
    spatial_features
] = risk_set.loc[
    spatial_model_data.index,
    spatial_features
]

print("Baseline rows:", len(leak_baseline))
print("Spatial model rows:", len(spatial_model_data))

print(
    "\nSame indices:",
    leak_baseline.index.equals(
        spatial_model_data.index
    )
)

print(
    "\nMissing spatial features:"
)

print(
    spatial_model_data[
        spatial_features
    ].isna().sum()
)

Baseline rows: 305455
Spatial model rows: 305455

Same indices: True

Missing spatial features:
Prior leaks within 25m     0
Prior leaks within 50m     0
Prior leaks within 100m    0
dtype: int64


In [80]:
# Re-run the spatial XGBoost experiment using exactly the same rows as the baseline

spatial_oof_parts_fixed = []
spatial_fold_results_fixed = []

for i, fold in enumerate(temporal_folds, start=1):

    train_data = spatial_model_data[
        spatial_model_data["Year"].isin(fold["train_years"])
    ].copy()

    test_data = spatial_model_data[
        spatial_model_data["Year"] == fold["test_year"]
    ].copy()

    X_train = train_data[
        spatial_numeric_features
        + spatial_categorical_features
    ]
    y_train = train_data["Y"]

    X_test = test_data[
        spatial_numeric_features
        + spatial_categorical_features
    ]
    y_test = test_data["Y"]

    # Train on past years only
    spatial_xgb_pipeline.fit(
        X_train,
        y_train
    )

    # Predict leak probabilities for the unseen test year
    y_prob = spatial_xgb_pipeline.predict_proba(
        X_test
    )[:, 1]

    y_pred = (y_prob >= 0.5).astype(int)

    spatial_fold_results_fixed.append({
        "Fold": i,
        "Test year": fold["test_year"],
        "PR-AUC": average_precision_score(y_test, y_prob),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Brier": brier_score_loss(y_test, y_prob),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    })

    fold_predictions = test_data[
        ["Pipe ID", "Year", "Y"]
    ].copy()

    fold_predictions[
        "predicted_probability"
    ] = y_prob

    fold_predictions["fold"] = i

    spatial_oof_parts_fixed.append(
        fold_predictions
    )


spatial_fold_results_fixed = pd.DataFrame(
    spatial_fold_results_fixed
)

spatial_oof_fixed = pd.concat(
    spatial_oof_parts_fixed,
    ignore_index=True
)

display(spatial_fold_results_fixed)

print("\nOOF rows:", len(spatial_oof_fixed))
print(
    "Actual leaks:",
    spatial_oof_fixed["Y"].sum()
)

,Fold,Test year,PR-AUC,ROC-AUC,Brier,Precision,Recall,F1
0,1,2023,0.341416,0.889270,0.035509,0.439204,0.326754,0.374725
1,2,2024,0.465337,0.901546,0.034636,0.514768,0.125064,0.201237
2,3,2025,0.243773,0.772109,0.044424,0.413408,0.041737,0.075820
3,4,2026,0.433029,0.873393,0.065666,0.483444,0.024029,0.045782



OOF rows: 138865
Actual leaks: 8586


In [81]:
# Compare baseline and spatial-history XGBoost on the identical OOF population

y_spatial = spatial_oof_fixed["Y"]
prob_spatial = spatial_oof_fixed[
    "predicted_probability"
]

pred_spatial = (
    prob_spatial >= 0.5
).astype(int)


spatial_metrics_fixed = {
    "PR-AUC": average_precision_score(
        y_spatial,
        prob_spatial
    ),
    "ROC-AUC": roc_auc_score(
        y_spatial,
        prob_spatial
    ),
    "Brier": brier_score_loss(
        y_spatial,
        prob_spatial
    ),
    "Precision": precision_score(
        y_spatial,
        pred_spatial,
        zero_division=0
    ),
    "Recall": recall_score(
        y_spatial,
        pred_spatial
    ),
    "F1": f1_score(
        y_spatial,
        pred_spatial
    ),
    "Mean predicted probability":
        prob_spatial.mean(),
    "Actual leak rate":
        y_spatial.mean()
}


fair_comparison = pd.DataFrame({
    "Baseline XGBoost":
        xgb_oof_metrics,
    "XGBoost + Spatial History":
        spatial_metrics_fixed
})

display(fair_comparison)

,Baseline XGBoost,XGBoost + Spatial History
PR-AUC,0.375631,0.375609
ROC-AUC,0.865155,0.869349
Brier,0.043794,0.044390
Precision,0.478217,0.456733
Recall,0.167482,0.114955
F1,0.248081,0.183679
Mean predicted probability,0.047616,0.044650
Actual leak rate,0.061830,0.061830


### Spatial History Experiment

Three historical spatial features were tested:

- Prior leaks within 25 m
- Prior leaks within 50 m
- Prior leaks within 100 m

For each pipe-year, only neighboring leaks that occurred strictly before the
prediction year were counted, preventing future information leakage.

The spatial-history model was evaluated using the same observations, temporal
folds, preprocessing, and XGBoost settings as the baseline model.

| Metric | Baseline XGBoost | XGBoost + Spatial History |
|---|---:|---:|
| PR-AUC | 0.375631 | 0.375609 |
| ROC-AUC | 0.865155 | 0.869349 |
| Brier | 0.043794 | 0.044390 |
| Precision | 0.478217 | 0.456733 |
| Recall | 0.167482 | 0.114955 |
| F1 | 0.248081 | 0.183679 |

The spatial features produced a small improvement in ROC-AUC but did not improve
PR-AUC and slightly worsened the Brier score. Precision, recall, and F1 at the
0.5 threshold also decreased.

**Decision:** Do not include these spatial-history features in the current
leak-prediction model. Retain the baseline XGBoost feature set.

In [84]:
# Check whether CatBoost and LightGBM are available in the current environment

import importlib.util

packages = {
    "xgboost": "XGBoost",
    "catboost": "CatBoost",
    "lightgbm": "LightGBM"
}

for package, name in packages.items():

    available = importlib.util.find_spec(package) is not None

    print(
        f"{name}:",
        "available" if available else "NOT installed"
    )

XGBoost: available
CatBoost: available
LightGBM: available


In [85]:
# Import the two additional boosting models for the ensemble experiment

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

print("CatBoost and LightGBM imported successfully.")

CatBoost and LightGBM imported successfully.


In [86]:
# Create a CatBoost pipeline using the same baseline feature set

catboost_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            CatBoostClassifier(
                iterations=300,
                depth=4,
                learning_rate=0.05,
                loss_function="Logloss",
                random_seed=42,
                verbose=False
            )
        )
    ]
)

print("CatBoost pipeline created.")

CatBoost pipeline created.


In [87]:
# Evaluate CatBoost on the same four temporal folds and save OOF probabilities

catboost_oof_parts = []
catboost_fold_results = []

for i, fold in enumerate(temporal_folds, start=1):

    train_data = leak_baseline[
        leak_baseline["Year"].isin(fold["train_years"])
    ].copy()

    test_data = leak_baseline[
        leak_baseline["Year"] == fold["test_year"]
    ].copy()

    X_train = train_data[
        numeric_features + categorical_features
    ]
    y_train = train_data["Y"]

    X_test = test_data[
        numeric_features + categorical_features
    ]
    y_test = test_data["Y"]

    # Train CatBoost using only years before the test year
    catboost_pipeline.fit(
        X_train,
        y_train
    )

    # Predict leak probabilities for the unseen test year
    y_prob = catboost_pipeline.predict_proba(
        X_test
    )[:, 1]

    # Use 0.5 only as a temporary classification threshold
    y_pred = (y_prob >= 0.5).astype(int)

    catboost_fold_results.append({
        "Fold": i,
        "Test year": fold["test_year"],
        "PR-AUC": average_precision_score(y_test, y_prob),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Brier": brier_score_loss(y_test, y_prob),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Mean predicted probability": y_prob.mean(),
        "Actual leak rate": y_test.mean()
    })

    # Save individual OOF predictions for later ensemble use
    fold_predictions = test_data[
        ["Pipe ID", "Year", "Y"]
    ].copy()

    fold_predictions[
        "catboost_probability"
    ] = y_prob

    fold_predictions["fold"] = i

    catboost_oof_parts.append(
        fold_predictions
    )


catboost_fold_results = pd.DataFrame(
    catboost_fold_results
)

catboost_oof = pd.concat(
    catboost_oof_parts,
    ignore_index=True
)

display(catboost_fold_results)

print("\nOOF rows:", len(catboost_oof))
print("Actual leaks:", catboost_oof["Y"].sum())

,Fold,Test year,PR-AUC,ROC-AUC,Brier,Precision,Recall,F1,Mean predicted probability,Actual leak rate
0,1,2023,0.342549,0.888727,0.036682,0.404255,0.250000,0.308943,0.047843,0.048636
1,2,2024,0.478558,0.903246,0.034731,0.375000,0.003075,0.006101,0.040351,0.054682
2,3,2025,0.244088,0.793522,0.045719,0.491296,0.143260,0.221834,0.035159,0.052568
3,4,2026,0.453815,0.885748,0.067164,0.545293,0.101053,0.170508,0.058654,0.095071



OOF rows: 138865
Actual leaks: 8586


In [88]:
# Calculate combined CatBoost performance across all OOF predictions

cat_y = catboost_oof["Y"]
cat_prob = catboost_oof["catboost_probability"]

cat_pred = (cat_prob >= 0.5).astype(int)

catboost_oof_metrics = {
    "PR-AUC": average_precision_score(
        cat_y, cat_prob
    ),
    "ROC-AUC": roc_auc_score(
        cat_y, cat_prob
    ),
    "Brier": brier_score_loss(
        cat_y, cat_prob
    ),
    "Precision": precision_score(
        cat_y, cat_pred, zero_division=0
    ),
    "Recall": recall_score(
        cat_y, cat_pred
    ),
    "F1": f1_score(
        cat_y, cat_pred
    ),
    "Mean predicted probability":
        cat_prob.mean(),
    "Actual leak rate":
        cat_y.mean()
}

xgb_vs_cat = pd.DataFrame({
    "XGBoost": xgb_oof_metrics,
    "CatBoost": catboost_oof_metrics
})

display(xgb_vs_cat)

,XGBoost,CatBoost
PR-AUC,0.375631,0.378810
ROC-AUC,0.865155,0.876201
Brier,0.043794,0.045390
Precision,0.478217,0.459982
Recall,0.167482,0.119147
F1,0.248081,0.189269
Mean predicted probability,0.047616,0.045325
Actual leak rate,0.061830,0.061830


In [89]:
# Create a LightGBM pipeline using the same baseline feature set

lightgbm_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LGBMClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary",
                random_state=42,
                n_jobs=-1,
                verbosity=-1
            )
        )
    ]
)

print("LightGBM pipeline created.")

LightGBM pipeline created.


In [90]:
# Evaluate LightGBM on the same four temporal folds and save OOF probabilities

lightgbm_oof_parts = []
lightgbm_fold_results = []

for i, fold in enumerate(temporal_folds, start=1):

    train_data = leak_baseline[
        leak_baseline["Year"].isin(fold["train_years"])
    ].copy()

    test_data = leak_baseline[
        leak_baseline["Year"] == fold["test_year"]
    ].copy()

    X_train = train_data[
        numeric_features + categorical_features
    ]
    y_train = train_data["Y"]

    X_test = test_data[
        numeric_features + categorical_features
    ]
    y_test = test_data["Y"]

    # Train only on years before the test year
    lightgbm_pipeline.fit(
        X_train,
        y_train
    )

    # Predict probabilities for the unseen test year
    y_prob = lightgbm_pipeline.predict_proba(
        X_test
    )[:, 1]

    y_pred = (y_prob >= 0.5).astype(int)

    lightgbm_fold_results.append({
        "Fold": i,
        "Test year": fold["test_year"],
        "PR-AUC": average_precision_score(y_test, y_prob),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Brier": brier_score_loss(y_test, y_prob),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Mean predicted probability": y_prob.mean(),
        "Actual leak rate": y_test.mean()
    })

    # Save OOF predictions for the ensemble
    fold_predictions = test_data[
        ["Pipe ID", "Year", "Y"]
    ].copy()

    fold_predictions[
        "lightgbm_probability"
    ] = y_prob

    fold_predictions["fold"] = i

    lightgbm_oof_parts.append(
        fold_predictions
    )


lightgbm_fold_results = pd.DataFrame(
    lightgbm_fold_results
)

lightgbm_oof = pd.concat(
    lightgbm_oof_parts,
    ignore_index=True
)

display(lightgbm_fold_results)

print("\nOOF rows:", len(lightgbm_oof))
print("Actual leaks:", lightgbm_oof["Y"].sum())

,Fold,Test year,PR-AUC,ROC-AUC,Brier,Precision,Recall,F1,Mean predicted probability,Actual leak rate
0,1,2023,0.345082,0.876267,0.037286,0.459810,0.583333,0.514258,0.059841,0.048636
1,2,2024,0.473926,0.899536,0.034990,0.602041,0.030241,0.057589,0.040536,0.054682
2,3,2025,0.254915,0.780095,0.044098,0.491329,0.143824,0.222513,0.029669,0.052568
3,4,2026,0.447827,0.882940,0.061363,0.500000,0.083278,0.142777,0.072352,0.095071



OOF rows: 138865
Actual leaks: 8586


In [92]:
# Merge the three OOF prediction sets using Pipe ID and Year

ensemble_oof = (
    xgb_oof[
        ["Pipe ID", "Year", "Y", "predicted_probability"]
    ]
    .rename(
        columns={
            "predicted_probability": "xgb_probability"
        }
    )
    .merge(
        catboost_oof[
            ["Pipe ID", "Year", "catboost_probability"]
        ],
        on=["Pipe ID", "Year"],
        how="inner"
    )
    .merge(
        lightgbm_oof[
            ["Pipe ID", "Year", "lightgbm_probability"]
        ],
        on=["Pipe ID", "Year"],
        how="inner"
    )
)

print("Ensemble OOF rows:", len(ensemble_oof))
print("Actual leaks:", ensemble_oof["Y"].sum())

display(ensemble_oof.head())

Ensemble OOF rows: 138865
Actual leaks: 8586


,Pipe ID,Year,Y,xgb_probability,catboost_probability,lightgbm_probability
0,P01001,2023,0,0.020856,0.016396,0.027829
1,P01002,2023,1,0.010684,0.017990,0.010208
2,P01003,2023,0,0.007354,0.010581,0.007057
3,P01004,2023,0,0.015491,0.013103,0.013520
4,P01005,2023,0,0.012260,0.015058,0.012351


In [93]:
# Check how strongly the three models' OOF probabilities are correlated

probability_columns = [
    "xgb_probability",
    "catboost_probability",
    "lightgbm_probability"
]

prediction_correlation = (
    ensemble_oof[probability_columns]
    .corr()
)

display(prediction_correlation)

,xgb_probability,catboost_probability,lightgbm_probability
xgb_probability,1.000000,0.960903,0.962286
catboost_probability,0.960903,1.000000,0.923831
lightgbm_probability,0.962286,0.923831,1.000000


In [94]:
# Create an equal-weight soft-voting ensemble from the three probabilities

ensemble_oof["ensemble_probability"] = (
    ensemble_oof["xgb_probability"]
    + ensemble_oof["catboost_probability"]
    + ensemble_oof["lightgbm_probability"]
) / 3

ensemble_oof[
    [
        "Pipe ID",
        "Year",
        "Y",
        "xgb_probability",
        "catboost_probability",
        "lightgbm_probability",
        "ensemble_probability"
    ]
].head()

,Pipe ID,Year,Y,xgb_probability,catboost_probability,lightgbm_probability,ensemble_probability
0,P01001,2023,0,0.020856,0.016396,0.027829,0.021693
1,P01002,2023,1,0.010684,0.017990,0.010208,0.012961
2,P01003,2023,0,0.007354,0.010581,0.007057,0.008330
3,P01004,2023,0,0.015491,0.013103,0.013520,0.014038
4,P01005,2023,0,0.012260,0.015058,0.012351,0.013223


In [95]:
# Compare all individual models with the equal-weight ensemble on the same OOF data

model_probabilities = {
    "XGBoost": "xgb_probability",
    "CatBoost": "catboost_probability",
    "LightGBM": "lightgbm_probability",
    "Ensemble": "ensemble_probability"
}

comparison_results = {}

y_true = ensemble_oof["Y"]

for model_name, probability_column in model_probabilities.items():

    y_prob = ensemble_oof[probability_column]
    y_pred = (y_prob >= 0.5).astype(int)

    comparison_results[model_name] = {
        "PR-AUC": average_precision_score(y_true, y_prob),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Brier": brier_score_loss(y_true, y_prob),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred
        ),
        "F1": f1_score(
            y_true, y_pred
        ),
        "Mean probability": y_prob.mean()
    }

ensemble_comparison = pd.DataFrame(
    comparison_results
)

display(ensemble_comparison)

,XGBoost,CatBoost,LightGBM,Ensemble
PR-AUC,0.375631,0.378810,0.372115,0.380553
ROC-AUC,0.865155,0.876201,0.863699,0.871555
Brier,0.043794,0.045390,0.043891,0.043825
Precision,0.478217,0.459982,0.474542,0.483543
Recall,0.167482,0.119147,0.189960,0.165968
F1,0.248081,0.189269,0.271313,0.247117
Mean probability,0.047616,0.045325,0.050432,0.047791


In [96]:
# Save the final OOF leak probabilities for later cost-model integration

final_leak_oof = ensemble_oof[
    [
        "Pipe ID",
        "Year",
        "Y",
        "xgb_probability",
        "catboost_probability",
        "lightgbm_probability",
        "ensemble_probability"
    ]
].copy()

final_leak_oof = final_leak_oof.rename(
    columns={
        "ensemble_probability": "predicted_leak_probability"
    }
)

print("Final leak OOF rows:", len(final_leak_oof))
print("Actual leaks:", final_leak_oof["Y"].sum())

display(final_leak_oof.head())

Final leak OOF rows: 138865
Actual leaks: 8586


,Pipe ID,Year,Y,xgb_probability,catboost_probability,lightgbm_probability,predicted_leak_probability
0,P01001,2023,0,0.020856,0.016396,0.027829,0.021693
1,P01002,2023,1,0.010684,0.017990,0.010208,0.012961
2,P01003,2023,0,0.007354,0.010581,0.007057,0.008330
3,P01004,2023,0,0.015491,0.013103,0.013520,0.014038
4,P01005,2023,0,0.012260,0.015058,0.012351,0.013223


### Final Leak Prediction Model

Three boosting models were evaluated using the same baseline features and
expanding-window temporal validation folds:

- XGBoost
- CatBoost
- LightGBM

An equal-weight soft-voting ensemble was constructed as:

\[
\hat{p}_{ensemble}
=
\frac{
\hat{p}_{XGB}+
\hat{p}_{CatBoost}+
\hat{p}_{LightGBM}
}{3}.
\]

The ensemble achieved the highest combined OOF PR-AUC (0.3806), compared
with 0.3756 for XGBoost, 0.3788 for CatBoost, and 0.3721 for LightGBM.

The ensemble Brier score (0.0438) remained very close to the best individual
model, indicating that the modest improvement in ranking performance did not
substantially reduce probability quality.

Because the final optimization requires leak probabilities rather than only
binary classifications, PR-AUC and Brier score were emphasized over F1 at the
temporary 0.5 threshold.

Therefore, the equal-weight ensemble is retained as the final leak-prediction
approach for the next stage.

# Cost prediction

In [97]:
# Create the event-level dataset for conditional leak-cost prediction

cost_data = events.copy()

print("Historical leak events:", len(cost_data))
print("Missing Cost:", cost_data["Cost"].isna().sum())
print("Missing Lay date:", cost_data["Lay date"].isna().sum())

display(
    cost_data[
        ["Pipe ID", "Date", "Cost", "Material", "Surface", "Age at leak"]
    ].head()
)

Historical leak events: 14113
Missing Cost: 0
Missing Lay date: 6


,Pipe ID,Date,Cost,Material,Surface,Age at leak
0,P19711,2019-01-06,2074.90,wrought iron,farmland,56.303901
1,P07628,2019-01-07,618.71,gray iron,grassland,53.774127
2,P31195,2019-04-16,1050.07,wrought iron,grassland,71.006160
3,P15715,2019-04-16,1000.81,wrought iron,structure,53.330595
4,P10415,2019-04-17,3157.83,wrought iron,swamp,65.842574


In [98]:
# Inspect the distribution and scale of historical leak costs

display(
    cost_data["Cost"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nTotal historical leak cost:",
      cost_data["Cost"].sum())

print("Minimum cost:",
      cost_data["Cost"].min())

print("Maximum cost:",
      cost_data["Cost"].max())

count    1.411300e+04
mean     2.501224e+04
std      1.560856e+05
min      1.978000e+01
50%      3.275200e+03
75%      9.246170e+03
90%      3.060028e+04
95%      7.358751e+04
99%      4.208845e+05
max      7.257576e+06
Name: Cost, dtype: float64


Total historical leak cost: 352997703.33
Minimum cost: 19.78
Maximum cost: 7257576.36


In [99]:
# Examine leak counts and cost distributions by year

cost_data["Leak year"] = cost_data["Date"].dt.year

cost_by_year = (
    cost_data
    .groupby("Leak year")["Cost"]
    .agg(
        Events="count",
        Mean="mean",
        Median="median",
        Total="sum",
        Maximum="max"
    )
)

display(cost_by_year)

,Events,Mean,Median,Total,Maximum
Leak year,,,,,
2019,1078,17248.774091,3216.365,1.859418e+07,1391513.84
2020,703,19346.948293,2196.850,1.360090e+07,2583058.15
2021,866,33499.592656,2988.505,2.901065e+07,7257576.36
2022,2874,37343.577495,6078.080,1.073254e+08,6421464.83
2023,1826,23379.635931,2630.630,4.269122e+07,4498329.32
2024,1955,14358.426005,2568.610,2.807072e+07,2814632.70
2025,1773,24058.591957,3264.990,4.265588e+07,2516372.15
2026,3038,23386.672041,3340.345,7.104871e+07,4500826.76


In [100]:
# Remove only leak events without a valid age for the baseline cost model

cost_model_data = cost_data.dropna(
    subset=["Age at leak"]
).copy()

print("Original leak events:", len(cost_data))
print("Cost-model events:", len(cost_model_data))
print("Removed events:", len(cost_data) - len(cost_model_data))

print("\nEvents by year:")
display(
    cost_model_data["Leak year"]
    .value_counts()
    .sort_index()
)

Original leak events: 14113
Cost-model events: 14107
Removed events: 6

Events by year:


Leak year
2019    1078
2020     703
2021     866
2022    2874
2023    1824
2024    1951
2025    1773
2026    3038
Name: count, dtype: int64

In [101]:
# Define expanding-window temporal folds for cost prediction

cost_temporal_folds = [
    {
        "train_years": [2019, 2020, 2021, 2022],
        "test_year": 2023
    },
    {
        "train_years": [2019, 2020, 2021, 2022, 2023],
        "test_year": 2024
    },
    {
        "train_years": [2019, 2020, 2021, 2022, 2023, 2024],
        "test_year": 2025
    },
    {
        "train_years": [2019, 2020, 2021, 2022, 2023, 2024, 2025],
        "test_year": 2026
    }
]

for i, fold in enumerate(cost_temporal_folds, start=1):

    train_mask = cost_model_data["Leak year"].isin(
        fold["train_years"]
    )

    test_mask = (
        cost_model_data["Leak year"]
        == fold["test_year"]
    )

    print(
        f"Fold {i}: "
        f"Train {min(fold['train_years'])}-{max(fold['train_years'])} "
        f"({train_mask.sum()} events) -> "
        f"Test {fold['test_year']} "
        f"({test_mask.sum()} events)"
    )

Fold 1: Train 2019-2022 (5521 events) -> Test 2023 (1824 events)
Fold 2: Train 2019-2023 (7345 events) -> Test 2024 (1951 events)
Fold 3: Train 2019-2024 (9296 events) -> Test 2025 (1773 events)
Fold 4: Train 2019-2025 (11069 events) -> Test 2026 (3038 events)


In [102]:
# Define cost features that are available before the future leak occurs

cost_numeric_features = [
    "Age at leak",
    "Pipe length",
    "Midpoint x",
    "Midpoint y",
    "Leak year"
]

cost_categorical_features = [
    "Material",
    "Surface"
]

cost_target = "Cost"

print("Numeric features:")
print(cost_numeric_features)

print("\nCategorical features:")
print(cost_categorical_features)

print("\nTarget:")
print(cost_target)

Numeric features:
['Age at leak', 'Pipe length', 'Midpoint x', 'Midpoint y', 'Leak year']

Categorical features:
['Material', 'Surface']

Target:
Cost


In [103]:
# Build the preprocessing pipeline and simple linear-regression baseline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

cost_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            cost_categorical_features
        ),
        (
            "numeric",
            "passthrough",
            cost_numeric_features
        )
    ]
)

linear_cost_pipeline = Pipeline(
    steps=[
        ("preprocessor", cost_preprocessor),
        ("model", LinearRegression())
    ]
)

print("Linear cost pipeline created.")

Linear cost pipeline created.


In [104]:
# Evaluate the linear cost baseline using expanding-window temporal validation

from sklearn.metrics import mean_squared_error, mean_absolute_error

linear_cost_results = []

for i, fold in enumerate(cost_temporal_folds, start=1):

    train_data = cost_model_data[
        cost_model_data["Leak year"].isin(
            fold["train_years"]
        )
    ].copy()

    test_data = cost_model_data[
        cost_model_data["Leak year"]
        == fold["test_year"]
    ].copy()

    X_train = train_data[
        cost_numeric_features
        + cost_categorical_features
    ]

    y_train = train_data["Cost"]

    X_test = test_data[
        cost_numeric_features
        + cost_categorical_features
    ]

    y_test = test_data["Cost"]

    # Train only on historical years
    linear_cost_pipeline.fit(
        X_train,
        y_train
    )

    # Predict conditional leak cost
    y_pred = linear_cost_pipeline.predict(
        X_test
    )

    rmse = mean_squared_error(
        y_test,
        y_pred
    ) ** 0.5

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    linear_cost_results.append({
        "Fold": i,
        "Test year": fold["test_year"],
        "RMSE": rmse,
        "MAE": mae,
        "Mean actual cost": y_test.mean(),
        "Mean predicted cost": y_pred.mean(),
        "Minimum prediction": y_pred.min(),
        "Maximum prediction": y_pred.max()
    })


linear_cost_results = pd.DataFrame(
    linear_cost_results
)

display(linear_cost_results)

,Fold,Test year,RMSE,MAE,Mean actual cost,Mean predicted cost,Minimum prediction,Maximum prediction
0,1,2023,159291.154801,36873.578928,23404.630800,32098.649436,-28214.642862,248008.237631
1,2,2024,99848.930366,21831.907975,14379.554669,18852.747800,-35905.357786,250116.327362
2,3,2025,114759.934107,29373.892111,24058.591957,22027.494880,-45146.325714,248906.004446
3,4,2026,128968.429389,28453.072785,23386.672041,16507.680325,-38425.794471,240011.215928


In [105]:
# Create an XGBoost regression model using the same baseline cost features

from xgboost import XGBRegressor

xgb_cost_pipeline = Pipeline(
    steps=[
        ("preprocessor", cost_preprocessor),
        (
            "model",
            XGBRegressor(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("XGBoost cost pipeline created.")

XGBoost cost pipeline created.


In [106]:
# Evaluate XGBoost cost regression on the same temporal folds

xgb_cost_results = []
xgb_cost_oof_parts = []

for i, fold in enumerate(cost_temporal_folds, start=1):

    train_data = cost_model_data[
        cost_model_data["Leak year"].isin(
            fold["train_years"]
        )
    ].copy()

    test_data = cost_model_data[
        cost_model_data["Leak year"]
        == fold["test_year"]
    ].copy()

    X_train = train_data[
        cost_numeric_features + cost_categorical_features
    ]
    y_train = train_data["Cost"]

    X_test = test_data[
        cost_numeric_features + cost_categorical_features
    ]
    y_test = test_data["Cost"]

    # Train using only earlier years
    xgb_cost_pipeline.fit(
        X_train,
        y_train
    )

    # Predict conditional leak costs
    y_pred = xgb_cost_pipeline.predict(
        X_test
    )

    rmse = mean_squared_error(
        y_test,
        y_pred
    ) ** 0.5

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    xgb_cost_results.append({
        "Fold": i,
        "Test year": fold["test_year"],
        "RMSE": rmse,
        "MAE": mae,
        "Mean actual cost": y_test.mean(),
        "Mean predicted cost": y_pred.mean(),
        "Minimum prediction": y_pred.min(),
        "Maximum prediction": y_pred.max()
    })

    # Save event-level OOF cost predictions
    fold_predictions = test_data[
        ["Pipe ID", "Leak year", "Cost"]
    ].copy()

    fold_predictions["predicted_cost"] = y_pred
    fold_predictions["fold"] = i

    xgb_cost_oof_parts.append(
        fold_predictions
    )


xgb_cost_results = pd.DataFrame(
    xgb_cost_results
)

xgb_cost_oof = pd.concat(
    xgb_cost_oof_parts,
    ignore_index=True
)

display(xgb_cost_results)

print("\nOOF cost events:", len(xgb_cost_oof))

,Fold,Test year,RMSE,MAE,Mean actual cost,Mean predicted cost,Minimum prediction,Maximum prediction
0,1,2023,189911.628790,40390.825537,23404.630800,31096.017578,-115476.070312,1.253993e+06
1,2,2024,116904.703896,26916.354246,14379.554669,21228.898438,-31403.478516,1.409034e+06
2,3,2025,120434.874238,29622.493959,24058.591957,24976.539062,-62246.007812,6.276491e+05
3,4,2026,165102.521904,30353.162606,23386.672041,26288.716797,-50098.429688,3.444692e+06



OOF cost events: 8586


In [107]:
# Train XGBoost on log-transformed cost and evaluate predictions on the original dollar scale

log_xgb_cost_results = []
log_xgb_cost_oof_parts = []

for i, fold in enumerate(cost_temporal_folds, start=1):

    train_data = cost_model_data[
        cost_model_data["Leak year"].isin(
            fold["train_years"]
        )
    ].copy()

    test_data = cost_model_data[
        cost_model_data["Leak year"]
        == fold["test_year"]
    ].copy()

    X_train = train_data[
        cost_numeric_features + cost_categorical_features
    ]

    X_test = test_data[
        cost_numeric_features + cost_categorical_features
    ]

    # Compress the extreme right tail during training
    y_train_log = np.log1p(
        train_data["Cost"]
    )

    y_test = test_data["Cost"]

    log_xgb_cost_pipeline = Pipeline(
        steps=[
            ("preprocessor", cost_preprocessor),
            (
                "model",
                XGBRegressor(
                    n_estimators=300,
                    max_depth=4,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="reg:squarederror",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]
    )

    log_xgb_cost_pipeline.fit(
        X_train,
        y_train_log
    )

    # Convert predictions back to dollars
    y_pred_log = log_xgb_cost_pipeline.predict(
        X_test
    )

    y_pred = np.expm1(
        y_pred_log
    )

    rmse = mean_squared_error(
        y_test,
        y_pred
    ) ** 0.5

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    log_xgb_cost_results.append({
        "Fold": i,
        "Test year": fold["test_year"],
        "RMSE": rmse,
        "MAE": mae,
        "Mean actual cost": y_test.mean(),
        "Mean predicted cost": y_pred.mean(),
        "Minimum prediction": y_pred.min(),
        "Maximum prediction": y_pred.max()
    })

    fold_predictions = test_data[
        ["Pipe ID", "Leak year", "Cost"]
    ].copy()

    fold_predictions[
        "predicted_cost"
    ] = y_pred

    fold_predictions["fold"] = i

    log_xgb_cost_oof_parts.append(
        fold_predictions
    )


log_xgb_cost_results = pd.DataFrame(
    log_xgb_cost_results
)

log_xgb_cost_oof = pd.concat(
    log_xgb_cost_oof_parts,
    ignore_index=True
)

display(log_xgb_cost_results)

,Fold,Test year,RMSE,MAE,Mean actual cost,Mean predicted cost,Minimum prediction,Maximum prediction
0,1,2023,166357.166738,20623.361058,23404.630800,7402.164062,630.620300,144281.171875
1,2,2024,105766.155553,12227.467133,14379.554669,4664.795410,1079.299316,184904.765625
2,3,2025,119476.322079,19749.587742,24058.591957,8570.979492,703.744324,152965.312500
3,4,2026,134535.829774,20179.081194,23386.672041,8904.146484,1004.598022,230344.109375


In [108]:
# Apply fold-specific smearing correction to log-XGBoost predictions

smearing_results = []
smearing_oof_parts = []

for i, fold in enumerate(cost_temporal_folds, start=1):

    train_data = cost_model_data[
        cost_model_data["Leak year"].isin(
            fold["train_years"]
        )
    ].copy()

    test_data = cost_model_data[
        cost_model_data["Leak year"]
        == fold["test_year"]
    ].copy()

    X_train = train_data[
        cost_numeric_features + cost_categorical_features
    ]

    X_test = test_data[
        cost_numeric_features + cost_categorical_features
    ]

    y_train_log = np.log1p(train_data["Cost"])
    y_test = test_data["Cost"]

    smearing_pipeline = Pipeline(
        steps=[
            ("preprocessor", cost_preprocessor),
            (
                "model",
                XGBRegressor(
                    n_estimators=300,
                    max_depth=4,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="reg:squarederror",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]
    )

    smearing_pipeline.fit(
        X_train,
        y_train_log
    )

    # Estimate the retransformation correction using training residuals
    train_pred_log = smearing_pipeline.predict(X_train)

    residual_log = (
        y_train_log.to_numpy() - train_pred_log
    )

    smearing_factor = np.mean(
        np.exp(residual_log)
    )

    # Predict the unseen test year and return to dollar scale
    test_pred_log = smearing_pipeline.predict(X_test)

    y_pred = (
        np.exp(test_pred_log) * smearing_factor
        - 1
    )

    rmse = mean_squared_error(
        y_test, y_pred
    ) ** 0.5

    mae = mean_absolute_error(
        y_test, y_pred
    )

    smearing_results.append({
        "Fold": i,
        "Test year": fold["test_year"],
        "Smearing factor": smearing_factor,
        "RMSE": rmse,
        "MAE": mae,
        "Mean actual cost": y_test.mean(),
        "Mean predicted cost": y_pred.mean(),
        "Minimum prediction": y_pred.min(),
        "Maximum prediction": y_pred.max()
    })

    fold_predictions = test_data[
        ["Pipe ID", "Leak year", "Cost"]
    ].copy()

    fold_predictions["predicted_cost"] = y_pred
    fold_predictions["fold"] = i

    smearing_oof_parts.append(
        fold_predictions
    )


smearing_results = pd.DataFrame(
    smearing_results
)

smearing_cost_oof = pd.concat(
    smearing_oof_parts,
    ignore_index=True
)

display(smearing_results)

,Fold,Test year,Smearing factor,RMSE,MAE,Mean actual cost,Mean predicted cost,Minimum prediction,Maximum prediction
0,1,2023,1.838244,163709.015829,22483.804782,23404.630800,13607.825090,1160.072520,265224.903554
1,2,2024,1.894937,101989.447316,13534.520495,14379.554669,8840.388855,2046.099306,350383.833806
2,3,2025,1.928879,115959.866655,21420.710822,24058.591957,16533.306667,1358.366296,295052.457928
3,4,2026,1.973057,131433.859443,22830.337312,23386.672041,17569.365488,1983.102679,454483.136295


In [109]:
# Generate event-level OOF predictions for the linear cost baseline

linear_cost_oof_parts = []

for i, fold in enumerate(cost_temporal_folds, start=1):

    train_data = cost_model_data[
        cost_model_data["Leak year"].isin(
            fold["train_years"]
        )
    ].copy()

    test_data = cost_model_data[
        cost_model_data["Leak year"]
        == fold["test_year"]
    ].copy()

    X_train = train_data[
        cost_numeric_features + cost_categorical_features
    ]

    y_train = train_data["Cost"]

    X_test = test_data[
        cost_numeric_features + cost_categorical_features
    ]

    linear_cost_pipeline.fit(
        X_train,
        y_train
    )

    y_pred = linear_cost_pipeline.predict(
        X_test
    )

    fold_predictions = test_data[
        ["Pipe ID", "Leak year", "Cost"]
    ].copy()

    fold_predictions["predicted_cost"] = y_pred
    fold_predictions["fold"] = i

    linear_cost_oof_parts.append(
        fold_predictions
    )


linear_cost_oof = pd.concat(
    linear_cost_oof_parts,
    ignore_index=True
)

print("Linear OOF events:", len(linear_cost_oof))

Linear OOF events: 8586


In [110]:
# Compare all cost models across the complete 2023-2026 OOF period

cost_models = {
    "Linear": linear_cost_oof,
    "Raw XGBoost": xgb_cost_oof,
    "Log XGBoost": log_xgb_cost_oof,
    "Smearing XGBoost": smearing_cost_oof
}

combined_cost_results = {}

for model_name, predictions in cost_models.items():

    y_true = predictions["Cost"]
    y_pred = predictions["predicted_cost"]

    combined_cost_results[model_name] = {
        "RMSE": mean_squared_error(
            y_true, y_pred
        ) ** 0.5,

        "MAE": mean_absolute_error(
            y_true, y_pred
        ),

        "Actual mean": y_true.mean(),

        "Predicted mean": y_pred.mean(),

        "Total actual cost": y_true.sum(),

        "Total predicted cost": y_pred.sum()
    }


combined_cost_comparison = pd.DataFrame(
    combined_cost_results
)

display(combined_cost_comparison)

,Linear,Raw XGBoost,Log XGBoost,Smearing XGBoost
RMSE,1.275170e+05,1.529955e+05,1.333158e+05,1.301777e+05
MAE,2.892753e+04,3.155372e+04,1.837793e+04,2.035334e+04
Actual mean,2.148255e+04,2.148255e+04,2.148255e+04,2.148255e+04
Predicted mean,2.149251e+04,2.588926e+04,7.552959e+03,1.453035e+04
Total actual cost,1.844492e+08,1.844492e+08,1.844492e+08,1.844492e+08
Total predicted cost,1.845347e+08,2.222852e+08,6.484970e+07,1.247576e+08


In [111]:
# Clip negative linear-regression predictions to zero and re-evaluate OOF performance

clipped_linear_cost_oof = linear_cost_oof.copy()

clipped_linear_cost_oof["predicted_cost"] = (
    clipped_linear_cost_oof["predicted_cost"]
    .clip(lower=0)
)

y_true = clipped_linear_cost_oof["Cost"]
y_pred = clipped_linear_cost_oof["predicted_cost"]

clipped_linear_results = {
    "RMSE": mean_squared_error(
        y_true, y_pred
    ) ** 0.5,

    "MAE": mean_absolute_error(
        y_true, y_pred
    ),

    "Actual mean": y_true.mean(),

    "Predicted mean": y_pred.mean(),

    "Total actual cost": y_true.sum(),

    "Total predicted cost": y_pred.sum(),

    "Minimum prediction": y_pred.min(),

    "Maximum prediction": y_pred.max()
}

display(
    pd.Series(
        clipped_linear_results,
        name="Clipped Linear"
    )
)

RMSE                    1.273419e+05
MAE                     2.726317e+04
Actual mean             2.148255e+04
Predicted mean          2.315688e+04
Total actual cost       1.844492e+08
Total predicted cost    1.988250e+08
Minimum prediction      0.000000e+00
Maximum prediction      2.501163e+05
Name: Clipped Linear, dtype: float64

In [112]:
# Compare the original linear model with its non-negative version

linear_vs_clipped = pd.DataFrame({
    "Original Linear": {
        "RMSE": mean_squared_error(
            linear_cost_oof["Cost"],
            linear_cost_oof["predicted_cost"]
        ) ** 0.5,

        "MAE": mean_absolute_error(
            linear_cost_oof["Cost"],
            linear_cost_oof["predicted_cost"]
        ),

        "Predicted mean":
            linear_cost_oof["predicted_cost"].mean(),

        "Total predicted cost":
            linear_cost_oof["predicted_cost"].sum(),

        "Minimum prediction":
            linear_cost_oof["predicted_cost"].min()
    },

    "Clipped Linear": {
        "RMSE": mean_squared_error(
            clipped_linear_cost_oof["Cost"],
            clipped_linear_cost_oof["predicted_cost"]
        ) ** 0.5,

        "MAE": mean_absolute_error(
            clipped_linear_cost_oof["Cost"],
            clipped_linear_cost_oof["predicted_cost"]
        ),

        "Predicted mean":
            clipped_linear_cost_oof["predicted_cost"].mean(),

        "Total predicted cost":
            clipped_linear_cost_oof["predicted_cost"].sum(),

        "Minimum prediction":
            clipped_linear_cost_oof["predicted_cost"].min()
    }
})

display(linear_vs_clipped)

,Original Linear,Clipped Linear
RMSE,1.275170e+05,1.273419e+05
MAE,2.892753e+04,2.726317e+04
Predicted mean,2.149251e+04,2.315688e+04
Total predicted cost,1.845347e+08,1.988250e+08
Minimum prediction,-4.514633e+04,0.000000e+00


In [113]:
# Fit, clip, and calibrate Linear Regression using training data only

calibrated_linear_results = []
calibrated_linear_oof_parts = []

for i, fold in enumerate(cost_temporal_folds, start=1):

    train_data = cost_model_data[
        cost_model_data["Leak year"].isin(
            fold["train_years"]
        )
    ].copy()

    test_data = cost_model_data[
        cost_model_data["Leak year"]
        == fold["test_year"]
    ].copy()

    X_train = train_data[
        cost_numeric_features + cost_categorical_features
    ]
    y_train = train_data["Cost"]

    X_test = test_data[
        cost_numeric_features + cost_categorical_features
    ]
    y_test = test_data["Cost"]

    # Fit the linear model using historical years only
    linear_cost_pipeline.fit(
        X_train,
        y_train
    )

    # Get training predictions and remove impossible negative costs
    train_pred = linear_cost_pipeline.predict(X_train)
    train_pred_clipped = np.clip(train_pred, 0, None)

    # Estimate the calibration factor from training data only
    calibration_factor = (
        y_train.mean()
        / train_pred_clipped.mean()
    )

    # Predict the unseen year, clip, then apply the historical calibration
    test_pred = linear_cost_pipeline.predict(X_test)
    test_pred_clipped = np.clip(test_pred, 0, None)

    y_pred = (
        test_pred_clipped
        * calibration_factor
    )

    calibrated_linear_results.append({
        "Fold": i,
        "Test year": fold["test_year"],
        "Calibration factor": calibration_factor,
        "RMSE": mean_squared_error(y_test, y_pred) ** 0.5,
        "MAE": mean_absolute_error(y_test, y_pred),
        "Mean actual cost": y_test.mean(),
        "Mean predicted cost": y_pred.mean(),
        "Minimum prediction": y_pred.min(),
        "Maximum prediction": y_pred.max()
    })

    fold_predictions = test_data[
        ["Pipe ID", "Leak year", "Cost"]
    ].copy()

    fold_predictions["predicted_cost"] = y_pred
    fold_predictions["fold"] = i

    calibrated_linear_oof_parts.append(
        fold_predictions
    )


calibrated_linear_results = pd.DataFrame(
    calibrated_linear_results
)

calibrated_linear_oof = pd.concat(
    calibrated_linear_oof_parts,
    ignore_index=True
)

display(calibrated_linear_results)

,Fold,Test year,Calibration factor,RMSE,MAE,Mean actual cost,Mean predicted cost,Minimum prediction,Maximum prediction
0,1,2023,0.941673,159393.310635,35094.792548,23404.630800,30595.109133,0.0,233542.652972
1,2,2024,0.950923,99977.531358,20481.171912,14379.554669,18557.922822,0.0,237841.292298
2,3,2025,0.954088,114264.293042,26457.904271,24058.591957,23200.827926,0.0,237478.333826
3,4,2026,0.957882,128651.781725,25332.370095,23386.672041,18405.111536,0.0,229902.500417


In [114]:
# Save the calibrated non-negative OOF cost predictions for integration with leak probabilities

final_cost_oof = calibrated_linear_oof[
    [
        "Pipe ID",
        "Leak year",
        "Cost",
        "predicted_cost"
    ]
].copy()

final_cost_oof = final_cost_oof.rename(
    columns={
        "Leak year": "Year",
        "Cost": "actual_leak_cost",
        "predicted_cost": "predicted_leak_cost"
    }
)

print("Final cost OOF rows:", len(final_cost_oof))
print(
    "Years:",
    sorted(final_cost_oof["Year"].unique())
)

display(final_cost_oof.head())

Final cost OOF rows: 8586
Years: [np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]


,Pipe ID,Year,actual_leak_cost,predicted_leak_cost
0,P01140,2023,1990.24,84060.579348
1,P39093,2023,16296.84,0.000000
2,P43557,2023,1370.38,2297.429498
3,P28881,2023,8065.28,3809.403434
4,P24481,2023,610885.15,204081.780927


# Leak × Cost Integration

In [115]:
# Predict conditional leak cost for every OOF pipe-year using training years only

all_pipe_cost_oof_parts = []

for i, fold in enumerate(cost_temporal_folds, start=1):

    # Cost model training data contains only historical leak events
    train_cost = cost_model_data[
        cost_model_data["Leak year"].isin(
            fold["train_years"]
        )
    ].copy()

    # All at-risk pipes in the unseen test year
    test_pipes = leak_baseline[
        leak_baseline["Year"]
        == fold["test_year"]
    ].copy()

    X_train = train_cost[
        cost_numeric_features + cost_categorical_features
    ].copy()

    y_train = train_cost["Cost"]

    # Rename pipe-year features to match the cost-model feature names
    test_pipes["Age at leak"] = test_pipes["Age"]
    test_pipes["Leak year"] = test_pipes["Year"]

    X_test = test_pipes[
        cost_numeric_features + cost_categorical_features
    ]

    # Fit cost model using historical leak events only
    linear_cost_pipeline.fit(
        X_train,
        y_train
    )

    # Calculate training-only non-negative calibration
    train_pred = linear_cost_pipeline.predict(X_train)
    train_pred_clipped = np.clip(train_pred, 0, None)

    calibration_factor = (
        y_train.mean()
        / train_pred_clipped.mean()
    )

    # Predict conditional cost for every at-risk pipe
    test_pred = linear_cost_pipeline.predict(X_test)

    predicted_cost = (
        np.clip(test_pred, 0, None)
        * calibration_factor
    )

    fold_predictions = test_pipes[
        ["Pipe ID", "Year", "Y"]
    ].copy()

    fold_predictions[
        "predicted_leak_cost"
    ] = predicted_cost

    fold_predictions["fold"] = i

    all_pipe_cost_oof_parts.append(
        fold_predictions
    )


all_pipe_cost_oof = pd.concat(
    all_pipe_cost_oof_parts,
    ignore_index=True
)

print(
    "All-pipe cost OOF rows:",
    len(all_pipe_cost_oof)
)

print(
    "Actual leaks:",
    all_pipe_cost_oof["Y"].sum()
)

display(all_pipe_cost_oof.head())

All-pipe cost OOF rows: 138865
Actual leaks: 8586


,Pipe ID,Year,Y,predicted_leak_cost,fold
0,P01001,2023,0,0.000000,1
1,P01002,2023,1,21964.167705,1
2,P01003,2023,0,25684.756454,1
3,P01004,2023,0,87655.456952,1
4,P01005,2023,0,26856.151431,1


In [116]:
# Merge OOF leak probabilities with conditional leak-cost predictions

expected_loss_oof = final_leak_oof.merge(
    all_pipe_cost_oof[
        [
            "Pipe ID",
            "Year",
            "predicted_leak_cost"
        ]
    ],
    on=["Pipe ID", "Year"],
    how="inner"
)

print("Merged OOF rows:", len(expected_loss_oof))
print("Actual leaks:", expected_loss_oof["Y"].sum())

display(expected_loss_oof.head())

Merged OOF rows: 138865
Actual leaks: 8586


,Pipe ID,Year,Y,xgb_probability,catboost_probability,lightgbm_probability,predicted_leak_probability,predicted_leak_cost
0,P01001,2023,0,0.020856,0.016396,0.027829,0.021693,0.000000
1,P01002,2023,1,0.010684,0.017990,0.010208,0.012961,21964.167705
2,P01003,2023,0,0.007354,0.010581,0.007057,0.008330,25684.756454
3,P01004,2023,0,0.015491,0.013103,0.013520,0.014038,87655.456952
4,P01005,2023,0,0.012260,0.015058,0.012351,0.013223,26856.151431


In [117]:
# Calculate expected leak cost for every pipe-year

expected_loss_oof["expected_leak_cost"] = (
    expected_loss_oof["predicted_leak_probability"]
    * expected_loss_oof["predicted_leak_cost"]
)

display(
    expected_loss_oof[
        [
            "Pipe ID",
            "Year",
            "Y",
            "predicted_leak_probability",
            "predicted_leak_cost",
            "expected_leak_cost"
        ]
    ].head(10)
)

,Pipe ID,Year,Y,predicted_leak_probability,predicted_leak_cost,expected_leak_cost
0,P01001,2023,0,0.021693,0.000000,0.000000
1,P01002,2023,1,0.012961,21964.167705,284.667327
2,P01003,2023,0,0.008330,25684.756454,213.965213
3,P01004,2023,0,0.014038,87655.456952,1230.502943
4,P01005,2023,0,0.013223,26856.151431,355.119774
5,P01006,2023,0,0.025479,76453.238904,1947.975238
6,P01007,2023,0,0.018369,79150.395700,1453.939419
7,P01008,2023,0,0.018591,78173.480809,1453.356602
8,P01009,2023,0,0.015429,37630.083153,580.588189
9,P01010,2023,0,0.012554,25732.764061,323.060866


In [118]:
# Attach actual leak costs and compare expected versus actual annual loss

actual_cost_lookup = final_cost_oof[
    ["Pipe ID", "Year", "actual_leak_cost"]
].copy()

expected_loss_check = expected_loss_oof.merge(
    actual_cost_lookup,
    on=["Pipe ID", "Year"],
    how="left"
)

# Non-leaking pipe-years have zero actual leak cost
expected_loss_check["actual_leak_cost"] = (
    expected_loss_check["actual_leak_cost"]
    .fillna(0)
)

annual_loss_check = (
    expected_loss_check
    .groupby("Year")
    .agg(
        Actual_leaks=("Y", "sum"),
        Actual_total_cost=("actual_leak_cost", "sum"),
        Expected_total_cost=("expected_leak_cost", "sum")
    )
)

annual_loss_check["Difference"] = (
    annual_loss_check["Expected_total_cost"]
    - annual_loss_check["Actual_total_cost"]
)

annual_loss_check["Percent_error"] = (
    annual_loss_check["Difference"]
    / annual_loss_check["Actual_total_cost"]
    * 100
)

display(annual_loss_check)

,Actual_leaks,Actual_total_cost,Expected_total_cost,Difference,Percent_error
Year,,,,,
2023,1824,42690046.58,6.243211e+07,1.974206e+07,46.245120
2024,1951,28054511.16,3.193478e+07,3.880267e+06,13.831168
2025,1773,42655883.54,1.986284e+07,-2.279304e+07,-53.434701
2026,3038,71048709.66,3.145308e+07,-3.959563e+07,-55.730265


In [119]:
# Decompose annual error into leak-frequency and conditional-cost components

diagnostic_by_year = (
    expected_loss_check
    .groupby("Year")
    .agg(
        Pipes=("Pipe ID", "count"),
        Actual_leaks=("Y", "sum"),
        Sum_predicted_probability=(
            "predicted_leak_probability", "sum"
        ),
        Actual_total_cost=(
            "actual_leak_cost", "sum"
        ),
        Expected_total_cost=(
            "expected_leak_cost", "sum"
        )
    )
)

diagnostic_by_year["Actual_leak_rate"] = (
    diagnostic_by_year["Actual_leaks"]
    / diagnostic_by_year["Pipes"]
)

diagnostic_by_year["Predicted_leak_rate"] = (
    diagnostic_by_year["Sum_predicted_probability"]
    / diagnostic_by_year["Pipes"]
)

diagnostic_by_year["Actual_cost_per_leak"] = (
    diagnostic_by_year["Actual_total_cost"]
    / diagnostic_by_year["Actual_leaks"]
)

diagnostic_by_year["Expected_cost_per_predicted_leak"] = (
    diagnostic_by_year["Expected_total_cost"]
    / diagnostic_by_year["Sum_predicted_probability"]
)

display(diagnostic_by_year)

,Pipes,Actual_leaks,Sum_predicted_probability,Actual_total_cost,Expected_total_cost,Actual_leak_rate,Predicted_leak_rate,Actual_cost_per_leak,Expected_cost_per_predicted_leak
Year,,,,,,,,,
2023,37503,1824,1990.647066,42690046.58,6.243211e+07,0.048636,0.053080,23404.630800,31362.721636
2024,35679,1951,1459.823585,28054511.16,3.193478e+07,0.054682,0.040915,14379.554669,21875.778828
2025,33728,1773,1078.488156,42655883.54,1.986284e+07,0.052568,0.031976,24058.591957,18417.299835
2026,31955,3038,2107.505810,71048709.66,3.145308e+07,0.095071,0.065952,23386.672041,14924.312571


In [120]:
# Calibrate ensemble probabilities using only earlier OOF years

calibrated_leak_oof = final_leak_oof.copy()

calibrated_leak_oof[
    "calibrated_leak_probability"
] = calibrated_leak_oof[
    "predicted_leak_probability"
]

calibration_history = []

oof_years = sorted(
    calibrated_leak_oof["Year"].unique()
)

for year in oof_years:

    historical = calibrated_leak_oof[
        calibrated_leak_oof["Year"] < year
    ]

    current = calibrated_leak_oof[
        calibrated_leak_oof["Year"] == year
    ]

    # The first OOF year has no earlier OOF calibration data
    if len(historical) == 0:
        calibration_factor = 1.0

    else:
        historical_actual_rate = (
            historical["Y"].mean()
        )

        historical_predicted_rate = (
            historical[
                "predicted_leak_probability"
            ].mean()
        )

        calibration_factor = (
            historical_actual_rate
            / historical_predicted_rate
        )

    calibrated_probability = (
        current["predicted_leak_probability"]
        * calibration_factor
    ).clip(0, 1)

    calibrated_leak_oof.loc[
        current.index,
        "calibrated_leak_probability"
    ] = calibrated_probability

    calibration_history.append({
        "Year": year,
        "Calibration factor": calibration_factor,
        "Actual leak rate":
            current["Y"].mean(),
        "Original predicted rate":
            current[
                "predicted_leak_probability"
            ].mean(),
        "Calibrated predicted rate":
            calibrated_probability.mean()
    })


calibration_history = pd.DataFrame(
    calibration_history
)

display(calibration_history)

,Year,Calibration factor,Actual leak rate,Original predicted rate,Calibrated predicted rate
0,2023,1.000000,0.048636,0.053080,0.053080
1,2024,0.916285,0.054682,0.040915,0.037490
2,2025,1.094054,0.052568,0.031976,0.034984
3,2026,1.225006,0.095071,0.065952,0.080792


In [121]:
# Compare OOF probability quality before and after temporal calibration

original_prob = calibrated_leak_oof[
    "predicted_leak_probability"
]

calibrated_prob = calibrated_leak_oof[
    "calibrated_leak_probability"
]

y_true = calibrated_leak_oof["Y"]

calibration_comparison = pd.DataFrame({
    "Original Ensemble": {
        "PR-AUC": average_precision_score(
            y_true, original_prob
        ),
        "ROC-AUC": roc_auc_score(
            y_true, original_prob
        ),
        "Brier": brier_score_loss(
            y_true, original_prob
        ),
        "Mean probability":
            original_prob.mean()
    },

    "Temporally Calibrated": {
        "PR-AUC": average_precision_score(
            y_true, calibrated_prob
        ),
        "ROC-AUC": roc_auc_score(
            y_true, calibrated_prob
        ),
        "Brier": brier_score_loss(
            y_true, calibrated_prob
        ),
        "Mean probability":
            calibrated_prob.mean()
    }
})

display(calibration_comparison)

,Original Ensemble,Temporally Calibrated
PR-AUC,0.380553,0.377448
ROC-AUC,0.871555,0.871308
Brier,0.043825,0.043710
Mean probability,0.047791,0.051056


In [122]:
# Recalculate expected financial loss using temporally calibrated leak probabilities

calibrated_expected_loss = expected_loss_oof.merge(
    calibrated_leak_oof[
        [
            "Pipe ID",
            "Year",
            "calibrated_leak_probability"
        ]
    ],
    on=["Pipe ID", "Year"],
    how="inner"
)

calibrated_expected_loss[
    "calibrated_expected_leak_cost"
] = (
    calibrated_expected_loss[
        "calibrated_leak_probability"
    ]
    * calibrated_expected_loss[
        "predicted_leak_cost"
    ]
)

# Attach actual realized leak cost
calibrated_expected_loss = (
    calibrated_expected_loss.merge(
        actual_cost_lookup,
        on=["Pipe ID", "Year"],
        how="left"
    )
)

calibrated_expected_loss[
    "actual_leak_cost"
] = (
    calibrated_expected_loss[
        "actual_leak_cost"
    ].fillna(0)
)

In [123]:
# Compare annual expected financial loss before and after probability calibration

integrated_comparison = (
    calibrated_expected_loss
    .groupby("Year")
    .agg(
        Actual_total_cost=(
            "actual_leak_cost", "sum"
        ),
        Original_expected_cost=(
            "expected_leak_cost", "sum"
        ),
        Calibrated_expected_cost=(
            "calibrated_expected_leak_cost", "sum"
        )
    )
)

integrated_comparison[
    "Original_error_pct"
] = (
    (
        integrated_comparison[
            "Original_expected_cost"
        ]
        - integrated_comparison[
            "Actual_total_cost"
        ]
    )
    / integrated_comparison[
        "Actual_total_cost"
    ]
    * 100
)

integrated_comparison[
    "Calibrated_error_pct"
] = (
    (
        integrated_comparison[
            "Calibrated_expected_cost"
        ]
        - integrated_comparison[
            "Actual_total_cost"
        ]
    )
    / integrated_comparison[
        "Actual_total_cost"
    ]
    * 100
)

display(integrated_comparison)

,Actual_total_cost,Original_expected_cost,Calibrated_expected_cost,Original_error_pct,Calibrated_error_pct
Year,,,,,
2023,42690046.58,6.243211e+07,6.243211e+07,46.245120,46.245120
2024,28054511.16,3.193478e+07,2.926136e+07,13.831168,4.301790
2025,42655883.54,1.986284e+07,2.173101e+07,-53.434701,-49.055065
2026,71048709.66,3.145308e+07,3.853019e+07,-55.730265,-45.769326


In [124]:
# Examine whether leak-cost severity differs systematically by pipe material

material_cost_summary = (
    cost_model_data
    .groupby("Material")["Cost"]
    .agg(
        Events="count",
        Mean="mean",
        Median="median",
        Maximum="max"
    )
    .sort_values("Mean", ascending=False)
)

display(material_cost_summary)

,Events,Mean,Median,Maximum
Material,,,,
brass,978,42410.811084,9400.725,1813588.52
gray iron,1113,39673.996496,7457.920,4500826.76
cast iron,3240,26986.314880,3825.620,7257576.36
wrought iron,7859,22039.602734,2690.310,4498329.32
copper,917,7306.972279,1533.390,735746.60


In [125]:
# Examine whether leak-cost severity differs systematically by surface type

surface_cost_summary = (
    cost_model_data
    .groupby("Surface")["Cost"]
    .agg(
        Events="count",
        Mean="mean",
        Median="median",
        Maximum="max"
    )
    .sort_values("Mean", ascending=False)
)

display(surface_cost_summary)

,Events,Mean,Median,Maximum
Surface,,,,
road,835,232300.236934,71591.530,6421464.83
structure,1893,48994.117517,11849.910,7257576.36
swamp,810,6404.154259,3381.130,172487.77
grassland,3816,6038.966976,2551.380,275516.16
farmland,2076,5767.722765,2385.375,259097.02
water,4677,5571.499964,2357.240,379755.55


In [126]:
# Inspect characteristics of the most expensive historical leak events

high_cost_events = (
    cost_model_data[
        [
            "Pipe ID",
            "Leak year",
            "Cost",
            "Material",
            "Surface",
            "Age at leak",
            "Pipe length",
            "Midpoint x",
            "Midpoint y"
        ]
    ]
    .sort_values(
        "Cost",
        ascending=False
    )
    .head(20)
)

display(high_cost_events)

,Pipe ID,Leak year,Cost,Material,Surface,Age at leak,Pipe length,Midpoint x,Midpoint y
1931,P01505,2021,7257576.36,cast iron,structure,42.193018,28.396422,2660.760,3445.140
3365,P08925,2022,6421464.83,cast iron,road,63.118412,51.056922,3891.075,4393.130
14011,P09126,2026,4500826.76,gray iron,road,61.708419,53.040465,4090.465,1988.450
7276,P30172,2023,4498329.32,wrought iron,road,63.164956,12.207805,1085.825,4222.900
2106,P11405,2021,3694393.85,wrought iron,road,63.600274,11.015466,4608.630,1928.935
12348,P37247,2026,3509392.32,wrought iron,road,62.444901,7.843577,1357.415,3418.990
6837,P44008,2023,3405617.73,cast iron,road,41.204654,66.720000,3009.520,4853.010
9217,P39464,2024,2814632.70,wrought iron,road,64.128679,27.924942,1086.370,4243.750
3135,P20921,2022,2638020.15,wrought iron,road,63.967146,22.795721,3745.980,1512.095
1703,P38606,2020,2583058.15,wrought iron,structure,64.342231,8.342757,3055.220,4788.650


In [127]:
# Measure how much of total historical leak cost comes from each surface

surface_cost_contribution = (
    cost_model_data
    .groupby("Surface")
    .agg(
        Events=("Cost", "size"),
        Total_cost=("Cost", "sum")
    )
)

surface_cost_contribution["Event_share_pct"] = (
    surface_cost_contribution["Events"]
    / surface_cost_contribution["Events"].sum()
    * 100
)

surface_cost_contribution["Cost_share_pct"] = (
    surface_cost_contribution["Total_cost"]
    / surface_cost_contribution["Total_cost"].sum()
    * 100
)

surface_cost_contribution = (
    surface_cost_contribution
    .sort_values("Cost_share_pct", ascending=False)
)

display(surface_cost_contribution)

,Events,Total_cost,Event_share_pct,Cost_share_pct
Surface,,,,
road,835,1.939707e+08,5.919047,54.952269
structure,1893,9.274586e+07,13.418870,26.275081
water,4677,2.605791e+07,33.153753,7.382254
grassland,3816,2.304470e+07,27.050401,6.528607
farmland,2076,1.197379e+07,14.716098,3.392198
swamp,810,5.187365e+06,5.741830,1.469590


In [128]:
# Group surfaces according to their clearly different historical cost severity

cost_model_data = cost_model_data.copy()

cost_model_data["Cost_surface_group"] = np.select(
    [
        cost_model_data["Surface"].eq("road"),
        cost_model_data["Surface"].eq("structure")
    ],
    [
        "road",
        "structure"
    ],
    default="other"
)

display(
    cost_model_data.groupby("Cost_surface_group")["Cost"]
    .agg(["count", "mean", "median", "max"])
)

,count,mean,median,max
Cost_surface_group,,,,
other,11379,5823.337791,2493.96,379755.55
road,835,232300.236934,71591.53,6421464.83
structure,1893,48994.117517,11849.91,7257576.36


In [130]:
# Evaluate separate road, structure, and other cost models using temporal folds
from sklearn.base import clone
surface_group_oof_parts = []
surface_group_results = []

for fold_num, (train_end, test_year) in enumerate(
    [(2022, 2023), (2023, 2024), (2024, 2025), (2025, 2026)],
    start=1
):
    train_fold = cost_model_data[
        cost_model_data["Leak year"] <= train_end
    ].copy()

    test_fold = cost_model_data[
        cost_model_data["Leak year"] == test_year
    ].copy()

    fold_predictions = pd.Series(
        index=test_fold.index,
        dtype=float
    )

    for group in ["road", "structure", "other"]:

        train_group = train_fold[
            train_fold["Cost_surface_group"] == group
        ].copy()

        test_group = test_fold[
            test_fold["Cost_surface_group"] == group
        ].copy()

        if len(test_group) == 0:
            continue

        group_pipeline = Pipeline([
            ("preprocessor", clone(cost_preprocessor)),
            ("model", LinearRegression())
        ])

        group_pipeline.fit(
            train_group[cost_numeric_features + cost_categorical_features],
            train_group["Cost"]
        )

        pred = group_pipeline.predict(
            test_group[cost_numeric_features + cost_categorical_features]
        )

        # Repair cost cannot be negative
        pred = np.maximum(pred, 0)

        # Training-only calibration after clipping
        train_pred = group_pipeline.predict(
            train_group[cost_numeric_features + cost_categorical_features]
        )

        train_pred = np.maximum(train_pred, 0)

        calibration_factor = (
            train_group["Cost"].mean()
            / train_pred.mean()
        )

        pred = pred * calibration_factor

        fold_predictions.loc[test_group.index] = pred

    y_true = test_fold["Cost"].values
    y_pred = fold_predictions.values

    surface_group_results.append({
        "Year": test_year,
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "Actual_mean": y_true.mean(),
        "Predicted_mean": y_pred.mean(),
        "Actual_total": y_true.sum(),
        "Predicted_total": y_pred.sum()
    })

    fold_output = test_fold[
        ["Pipe ID", "Leak year", "Cost"]
    ].copy()

    fold_output["predicted_cost"] = y_pred
    fold_output["fold"] = fold_num

    surface_group_oof_parts.append(fold_output)


surface_group_results = pd.DataFrame(surface_group_results)

surface_group_oof = pd.concat(
    surface_group_oof_parts,
    ignore_index=True
)

display(surface_group_results)

,Year,RMSE,MAE,Actual_mean,Predicted_mean,Actual_total,Predicted_total
0,2023,164650.956637,30436.805701,23404.630800,23322.885229,42690046.58,4.254094e+07
1,2024,100895.616639,19065.346529,14379.554669,18079.810474,28054511.16,3.527371e+07
2,2025,121660.279969,30851.041274,24058.591957,31705.983605,42655883.54,5.621471e+07
3,2026,131769.863944,24235.536160,23386.672041,18187.689947,71048709.66,5.525420e+07


In [131]:
# Create interpretable surface interaction features for cost prediction

cost_interaction_data = cost_model_data.copy()

cost_interaction_data["Surface_Material"] = (
    cost_interaction_data["Surface"].astype(str)
    + "_"
    + cost_interaction_data["Material"].astype(str)
)

cost_interaction_data["Road_length"] = (
    cost_interaction_data["Pipe length"]
    * cost_interaction_data["Surface"].eq("road").astype(int)
)

cost_interaction_data["Structure_length"] = (
    cost_interaction_data["Pipe length"]
    * cost_interaction_data["Surface"].eq("structure").astype(int)
)

interaction_numeric_features = [
    "Age at leak",
    "Pipe length",
    "Midpoint x",
    "Midpoint y",
    "Leak year",
    "Road_length",
    "Structure_length"
]

interaction_categorical_features = [
    "Material",
    "Surface",
    "Surface_Material"
]

display(
    cost_interaction_data[
        interaction_numeric_features
        + interaction_categorical_features
        + ["Cost"]
    ].head()
)

,Age at leak,Pipe length,Midpoint x,Midpoint y,Leak year,Road_length,Structure_length,Material,Surface,Surface_Material,Cost
0,56.303901,34.357923,1805.615,3195.370,2019,0.0,0.000000,wrought iron,farmland,farmland_wrought iron,2074.90
1,53.774127,58.111749,4034.635,2597.825,2019,0.0,0.000000,gray iron,grassland,grassland_gray iron,618.71
2,71.006160,15.095128,3007.460,2465.955,2019,0.0,0.000000,wrought iron,grassland,grassland_wrought iron,1050.07
3,53.330595,4.614152,3051.110,3480.670,2019,0.0,4.614152,wrought iron,structure,structure_wrought iron,1000.81
4,65.842574,20.392952,3582.305,4652.690,2019,0.0,0.000000,wrought iron,swamp,swamp_wrought iron,3157.83


In [132]:
# Evaluate the interaction cost model using the same temporal validation folds

interaction_preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        interaction_categorical_features
    ),
    (
        "numeric",
        "passthrough",
        interaction_numeric_features
    )
])

interaction_results = []
interaction_oof_parts = []

for fold_num, (train_end, test_year) in enumerate(
    [(2022, 2023), (2023, 2024), (2024, 2025), (2025, 2026)],
    start=1
):
    train_fold = cost_interaction_data[
        cost_interaction_data["Leak year"] <= train_end
    ].copy()

    test_fold = cost_interaction_data[
        cost_interaction_data["Leak year"] == test_year
    ].copy()

    model = Pipeline([
        ("preprocessor", clone(interaction_preprocessor)),
        ("model", LinearRegression())
    ])

    features = (
        interaction_numeric_features
        + interaction_categorical_features
    )

    model.fit(
        train_fold[features],
        train_fold["Cost"]
    )

    # Predict training data for training-only calibration
    train_pred = model.predict(
        train_fold[features]
    )

    train_pred = np.maximum(train_pred, 0)

    calibration_factor = (
        train_fold["Cost"].mean()
        / train_pred.mean()
    )

    # Predict unseen test year
    test_pred = model.predict(
        test_fold[features]
    )

    test_pred = np.maximum(test_pred, 0)
    test_pred = test_pred * calibration_factor

    y_true = test_fold["Cost"].values

    interaction_results.append({
        "Year": test_year,
        "RMSE": np.sqrt(
            mean_squared_error(y_true, test_pred)
        ),
        "MAE": mean_absolute_error(
            y_true, test_pred
        ),
        "Actual_mean": y_true.mean(),
        "Predicted_mean": test_pred.mean(),
        "Actual_total": y_true.sum(),
        "Predicted_total": test_pred.sum(),
        "Calibration_factor": calibration_factor
    })

    fold_output = test_fold[
        ["Pipe ID", "Leak year", "Cost"]
    ].copy()

    fold_output["predicted_cost"] = test_pred
    fold_output["fold"] = fold_num

    interaction_oof_parts.append(fold_output)


interaction_results = pd.DataFrame(
    interaction_results
)

interaction_oof = pd.concat(
    interaction_oof_parts,
    ignore_index=True
)

display(interaction_results)

,Year,RMSE,MAE,Actual_mean,Predicted_mean,Actual_total,Predicted_total,Calibration_factor
0,2023,158869.408173,30706.821509,23404.630800,23757.365721,42690046.58,4.333344e+07,0.988173
1,2024,98968.509023,18581.899421,14379.554669,15608.026774,28054511.16,3.045126e+07,0.979433
2,2025,115793.215217,28640.530823,24058.591957,26148.198188,42655883.54,4.636076e+07,0.980531
3,2026,129468.078436,26138.751751,23386.672041,19307.687397,71048709.66,5.865675e+07,0.983134


In [133]:
# Compare the global, surface-group, and interaction cost models on all OOF events

def summarize_cost_oof(df, prediction_column):
    y_true = df["Cost"].values
    y_pred = df[prediction_column].values

    return {
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "MAE": mean_absolute_error(
            y_true, y_pred
        ),
        "Actual_total": y_true.sum(),
        "Predicted_total": y_pred.sum(),
        "Total_error_pct": (
            (y_pred.sum() - y_true.sum())
            / y_true.sum()
            * 100
        )
    }


# Current global calibrated Linear model
global_compare = final_cost_oof.rename(
    columns={
        "actual_leak_cost": "Cost",
        "predicted_leak_cost": "predicted_cost"
    }
)

cost_model_comparison = pd.DataFrame({
    "Global Linear": summarize_cost_oof(
        global_compare,
        "predicted_cost"
    ),

    "Surface Group": summarize_cost_oof(
        surface_group_oof,
        "predicted_cost"
    ),

    "Interaction Linear": summarize_cost_oof(
        interaction_oof,
        "predicted_cost"
    )
}).T

display(cost_model_comparison)

,RMSE,MAE,Actual_total,Predicted_total,Total_error_pct
Global Linear,127361.849197,26536.369978,1.844492e+08,1.890618e+08,2.500761
Surface Group,131424.946269,25744.197497,1.844492e+08,1.892836e+08,2.621000
Interaction Linear,127621.281824,25908.655622,1.844492e+08,1.788022e+08,-3.061519


In [134]:
# Summarize the final calibrated leak + global cost OOF predictions by year

final_integrated_oof = calibrated_expected_loss.copy()

final_oof_summary = (
    final_integrated_oof
    .groupby("Year")
    .agg(
        Pipes=("Pipe ID", "count"),
        Actual_leaks=("Y", "sum"),
        Predicted_leaks=(
            "calibrated_leak_probability", "sum"
        ),
        Actual_total_cost=(
            "actual_leak_cost", "sum"
        ),
        Expected_total_cost=(
            "calibrated_expected_leak_cost", "sum"
        )
    )
)

final_oof_summary["Actual_leak_rate"] = (
    final_oof_summary["Actual_leaks"]
    / final_oof_summary["Pipes"]
)

final_oof_summary["Predicted_leak_rate"] = (
    final_oof_summary["Predicted_leaks"]
    / final_oof_summary["Pipes"]
)

final_oof_summary["Cost_error"] = (
    final_oof_summary["Expected_total_cost"]
    - final_oof_summary["Actual_total_cost"]
)

final_oof_summary["Cost_error_pct"] = (
    final_oof_summary["Cost_error"]
    / final_oof_summary["Actual_total_cost"]
    * 100
)

display(final_oof_summary)

,Pipes,Actual_leaks,Predicted_leaks,Actual_total_cost,Expected_total_cost,Actual_leak_rate,Predicted_leak_rate,Cost_error,Cost_error_pct
Year,,,,,,,,,
2023,37503,1824,1990.647066,42690046.58,6.243211e+07,0.048636,0.053080,1.974206e+07,46.245120
2024,35679,1951,1337.614419,28054511.16,2.926136e+07,0.054682,0.037490,1.206846e+06,4.301790
2025,33728,1773,1179.923900,42655883.54,2.173101e+07,0.052568,0.034984,-2.092487e+07,-49.055065
2026,31955,3038,2581.706465,71048709.66,3.853019e+07,0.095071,0.080792,-3.251852e+07,-45.769326


In [135]:
# Calculate overall diagnostics for the final integrated OOF model

actual_total = final_integrated_oof[
    "actual_leak_cost"
].sum()

expected_total = final_integrated_oof[
    "calibrated_expected_leak_cost"
].sum()

actual_leaks = final_integrated_oof["Y"].sum()

predicted_leaks = final_integrated_oof[
    "calibrated_leak_probability"
].sum()

overall_final_summary = pd.DataFrame({
    "Metric": [
        "OOF pipe-years",
        "Actual leaks",
        "Predicted leaks",
        "Actual total leak cost",
        "Expected total leak cost",
        "Total cost error",
        "Total cost error (%)"
    ],
    "Value": [
        len(final_integrated_oof),
        actual_leaks,
        predicted_leaks,
        actual_total,
        expected_total,
        expected_total - actual_total,
        (expected_total - actual_total)
        / actual_total * 100
    ]
})

display(overall_final_summary)

,Metric,Value
0,OOF pipe-years,1.388650e+05
1,Actual leaks,8.586000e+03
2,Predicted leaks,7.089892e+03
3,Actual total leak cost,1.844492e+08
4,Expected total leak cost,1.519547e+08
5,Total cost error,-3.249448e+07
6,Total cost error (%),-1.761704e+01


In [136]:
# Build the 2027 risk set using pipes with no observed leak through 2026

leaked_pipe_ids = set(events["Pipe ID"])

risk_2027 = pipes_clean[
    ~pipes_clean["Pipe ID"].isin(leaked_pipe_ids)
].copy()

risk_2027["Year"] = 2027

risk_2027["Age"] = (
    pd.Timestamp("2027-01-01")
    - risk_2027["Lay date"]
).dt.days / 365.25

risk_2027["Pipe length"] = np.sqrt(
    (risk_2027["GPS x2"] - risk_2027["GPS x1"]) ** 2
    + (risk_2027["GPS y2"] - risk_2027["GPS y1"]) ** 2
)

risk_2027["Midpoint x"] = (
    risk_2027["GPS x1"] + risk_2027["GPS x2"]
) / 2

risk_2027["Midpoint y"] = (
    risk_2027["GPS y1"] + risk_2027["GPS y2"]
) / 2

print("2027 at-risk pipes:", len(risk_2027))

display(
    risk_2027[
        [
            "Pipe ID",
            "Material",
            "Surface",
            "Age",
            "Pipe length",
            "Midpoint x",
            "Midpoint y",
            "Year"
        ]
    ].head()
)

2027 at-risk pipes: 28926


,Pipe ID,Material,Surface,Age,Pipe length,Midpoint x,Midpoint y,Year
0,P01001,copper,grassland,71.811088,12.140000,6130.220,5020.970,2027
3,P01004,cast iron,structure,44.613279,29.750686,2842.740,3391.535,2027
5,P01006,cast iron,structure,55.241615,41.926084,3734.360,5100.305,2027
6,P01007,cast iron,structure,55.238877,12.150000,3728.295,5099.050,2027
7,P01008,cast iron,structure,55.241615,22.918440,3731.335,5088.000,2027


In [137]:
# Retrain all three leak models on 2019–2026 data and predict 2027 leak probabilities

final_leak_features = numeric_features + categorical_features

X_train_final = leak_baseline[final_leak_features]
y_train_final = leak_baseline["Y"]

X_2027 = risk_2027[final_leak_features]


# XGBoost
final_xgb = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

final_xgb.fit(X_train_final, y_train_final)

xgb_prob_2027 = final_xgb.predict_proba(X_2027)[:, 1]


# CatBoost
final_cat = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", CatBoostClassifier(
        iterations=300,
        depth=4,
        learning_rate=0.05,
        loss_function="Logloss",
        random_seed=42,
        verbose=False
    ))
])

final_cat.fit(X_train_final, y_train_final)

cat_prob_2027 = final_cat.predict_proba(X_2027)[:, 1]


# LightGBM
final_lgbm = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", LGBMClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

final_lgbm.fit(X_train_final, y_train_final)

lgbm_prob_2027 = final_lgbm.predict_proba(X_2027)[:, 1]


# Equal-weight soft-voting ensemble
leak_2027 = risk_2027[
    ["Pipe ID", "Material", "Surface", "Age", "Pipe length"]
].copy()

leak_2027["xgb_probability"] = xgb_prob_2027
leak_2027["catboost_probability"] = cat_prob_2027
leak_2027["lightgbm_probability"] = lgbm_prob_2027

leak_2027["raw_ensemble_probability"] = (
    leak_2027["xgb_probability"]
    + leak_2027["catboost_probability"]
    + leak_2027["lightgbm_probability"]
) / 3

print("2027 pipes:", len(leak_2027))
print(
    "Raw predicted leaks:",
    leak_2027["raw_ensemble_probability"].sum()
)
print(
    "Mean raw leak probability:",
    leak_2027["raw_ensemble_probability"].mean()
)

display(leak_2027.head(10))

2027 pipes: 28926
Raw predicted leaks: 2689.427452777133
Mean raw leak probability: 0.09297612710976745


,Pipe ID,Material,Surface,Age,Pipe length,xgb_probability,catboost_probability,lightgbm_probability,raw_ensemble_probability
0,P01001,copper,grassland,71.811088,12.140000,0.009348,0.010553,0.009720,0.009874
3,P01004,cast iron,structure,44.613279,29.750686,0.049130,0.043365,0.046025,0.046173
5,P01006,cast iron,structure,55.241615,41.926084,0.053931,0.065595,0.054102,0.057876
6,P01007,cast iron,structure,55.238877,12.150000,0.044414,0.052682,0.043044,0.046713
7,P01008,cast iron,structure,55.241615,22.918440,0.045937,0.056613,0.042535,0.048362
8,P01009,cast iron,water,44.594114,45.490000,0.054052,0.049757,0.043974,0.049261
9,P01010,cast iron,water,55.452430,28.378337,0.047183,0.058259,0.040866,0.048770
11,P01012,cast iron,water,55.463381,5.852350,0.039356,0.046990,0.042315,0.042887
12,P01013,cast iron,water,55.463381,5.823075,0.039356,0.046990,0.042315,0.042887
13,P01014,cast iron,water,55.463381,5.744258,0.039356,0.047064,0.042315,0.042912


In [139]:
# Check 2027 cost features for missing values and data-type problems

print("Feature columns:")
print(cost_features)

print("\nMissing values:")
display(
    cost_2027[cost_features]
    .isna()
    .sum()
)

print("\nData types:")
display(
    cost_2027[cost_features]
    .dtypes
)

print("\nRows with any missing cost feature:")
missing_2027 = cost_2027[
    cost_2027[cost_features].isna().any(axis=1)
]

print("Number of affected rows:", len(missing_2027))

display(
    missing_2027[
        ["Pipe ID"] + cost_features
    ]
)

Feature columns:
['Age at leak', 'Pipe length', 'Midpoint x', 'Midpoint y', 'Leak year', 'Material', 'Surface']

Missing values:


Age at leak    9
Pipe length    0
Midpoint x     0
Midpoint y     0
Leak year      0
Material       8
Surface        0
dtype: int64


Data types:


Age at leak    float64
Pipe length    float64
Midpoint x     float64
Midpoint y     float64
Leak year        int64
Material        object
Surface         object
dtype: object


Rows with any missing cost feature:
Number of affected rows: 17


,Pipe ID,Age at leak,Pipe length,Midpoint x,Midpoint y,Leak year,Material,Surface
8985,P09986,64.862423,7.285396,580.635,2501.295,2027,NaN,water
8986,P09987,79.164956,12.810308,600.600,2493.970,2027,NaN,water
9868,P10869,NaN,6.314966,1001.250,4236.150,2027,wrought iron,water
11274,P12275,NaN,6.530926,2589.975,3206.375,2027,wrought iron,grassland
11275,P12276,NaN,14.536554,2588.975,3204.245,2027,wrought iron,grassland
11276,P12277,NaN,8.518709,2490.140,2764.880,2027,wrought iron,grassland
11277,P12278,NaN,3.622430,2490.630,2769.390,2027,wrought iron,grassland
11278,P12279,NaN,19.825955,2494.835,2778.150,2027,wrought iron,grassland
11279,P12280,NaN,33.519335,2521.795,2880.615,2027,wrought iron,grassland
11280,P12281,NaN,6.132414,2519.875,2873.730,2027,wrought iron,grassland


In [140]:
# Handle missing cost features consistently and refit the final production cost model

final_cost_train = cost_model_data.copy()
cost_2027 = risk_2027.copy()

cost_2027["Age at leak"] = cost_2027["Age"]
cost_2027["Leak year"] = 2027


# Use training information only for missing age
age_median = final_cost_train["Age at leak"].median()

final_cost_train["Age at leak"] = (
    final_cost_train["Age at leak"]
    .fillna(age_median)
)

cost_2027["Age at leak"] = (
    cost_2027["Age at leak"]
    .fillna(age_median)
)


# Represent missing material explicitly
final_cost_train["Material"] = (
    final_cost_train["Material"]
    .fillna("Unknown")
)

cost_2027["Material"] = (
    cost_2027["Material"]
    .fillna("Unknown")
)


cost_features = (
    cost_numeric_features
    + cost_categorical_features
)


# Fit the final cost model on all historical leak events
final_cost_model = Pipeline([
    ("preprocessor", clone(cost_preprocessor)),
    ("model", LinearRegression())
])

final_cost_model.fit(
    final_cost_train[cost_features],
    final_cost_train["Cost"]
)


# Calculate training-only non-negative calibration
train_cost_pred = final_cost_model.predict(
    final_cost_train[cost_features]
)

train_cost_pred = np.maximum(train_cost_pred, 0)

final_cost_calibration = (
    final_cost_train["Cost"].mean()
    / train_cost_pred.mean()
)


# Predict conditional leak cost for every 2027 surviving pipe
cost_pred_2027 = final_cost_model.predict(
    cost_2027[cost_features]
)

cost_pred_2027 = np.maximum(
    cost_pred_2027,
    0
)

cost_pred_2027 *= final_cost_calibration

cost_2027["predicted_leak_cost"] = cost_pred_2027


print("Training age median:", age_median)
print(
    "Final cost calibration factor:",
    final_cost_calibration
)
print(
    "Mean predicted cost conditional on leak:",
    cost_2027["predicted_leak_cost"].mean()
)
print(
    "Median predicted cost conditional on leak:",
    cost_2027["predicted_leak_cost"].median()
)
print(
    "Zero-cost predictions:",
    (cost_2027["predicted_leak_cost"] == 0).sum()
)
print(
    "Missing predictions:",
    cost_2027["predicted_leak_cost"].isna().sum()
)

Training age median: 63.4715947980835
Final cost calibration factor: 0.9670054441886302
Mean predicted cost conditional on leak: 18214.622943773542
Median predicted cost conditional on leak: 1255.1783708231485
Zero-cost predictions: 11904
Missing predictions: 0


In [141]:
# Merge 2027 leak probabilities and conditional cost predictions to calculate expected leak loss

prediction_2027 = leak_2027[
    [
        "Pipe ID",
        "raw_ensemble_probability"
    ]
].merge(
    cost_2027[
        [
            "Pipe ID",
            "predicted_leak_cost"
        ]
    ],
    on="Pipe ID",
    how="inner"
)

prediction_2027["expected_leak_loss"] = (
    prediction_2027["raw_ensemble_probability"]
    * prediction_2027["predicted_leak_cost"]
)

print("2027 predicted pipes:", len(prediction_2027))

print(
    "Expected number of leaks:",
    prediction_2027["raw_ensemble_probability"].sum()
)

print(
    "Total expected leak loss:",
    prediction_2027["expected_leak_loss"].sum()
)

print(
    "Mean expected leak loss per pipe:",
    prediction_2027["expected_leak_loss"].mean()
)

display(
    prediction_2027
    .sort_values(
        "expected_leak_loss",
        ascending=False
    )
    .head(10)
)

2027 predicted pipes: 28926
Expected number of leaks: 2689.427452777133
Total expected leak loss: 49451671.52484188
Mean expected leak loss per pipe: 1709.592460929333


,Pipe ID,raw_ensemble_probability,predicted_leak_cost,expected_leak_loss
17857,P27698,0.642952,229903.923405,147817.125257
17854,P27694,0.628506,225725.353837,141869.782790
17866,P27714,0.623571,223507.704302,139373.024403
23362,P35992,0.590301,229270.605813,135338.576523
15980,P24950,0.601656,223172.977652,134273.398198
10837,P17425,0.593881,222421.092502,132091.595157
10838,P17426,0.593135,222336.689860,131875.598604
10839,P17427,0.592956,222150.573357,131725.626437
28700,P43633,0.598534,218709.799621,130905.163047
4635,P08403,0.599058,218506.843873,130898.218233


# cost prediction

In [ ]:
# Store the work-unit construction cost rules

SURFACE_RATES = {
    "grassland": 3,
    "farmland": 7,
    "swamp": 12,
    "road": 17,
    "water": 32,
    "structure": 50
}

GROUNDBREAKING_COST = 100
NON_PU_PIPE_COST_PER_METER = 200


def construction_cost(
    radius,
    surface,
    non_pu_pipe_meters
):
    area = np.pi * radius**2

    surface_cost = (
        SURFACE_RATES[surface]
        * area**0.85
    )

    pipe_cost = (
        NON_PU_PIPE_COST_PER_METER
        * non_pu_pipe_meters
    )

    total_cost = (
        GROUNDBREAKING_COST
        + surface_cost
        + pipe_cost
    )

    return total_cost

In [143]:
# Build the physical pipe network as it exists at the start of 2027

network_2027 = pipes_clean.copy()

leaked_pipe_ids = set(events["Pipe ID"])

network_2027["Material_2027"] = np.where(
    network_2027["Pipe ID"].isin(leaked_pipe_ids),
    "polyurethane",
    network_2027["Material"]
)

print("Total physical pipes:", len(network_2027))

print("\nMaterial at start of 2027:")
display(
    network_2027["Material_2027"]
    .value_counts(dropna=False)
)

print(
    "\nHistorically replaced with polyurethane:",
    network_2027["Pipe ID"].isin(leaked_pipe_ids).sum()
)

Total physical pipes: 43039

Material at start of 2027:


Material_2027
wrought iron    16697
polyurethane    15791
cast iron        7390
gray iron        2150
copper            696
brass             307
NaN                 8
Name: count, dtype: int64


Historically replaced with polyurethane: 14113


In [144]:
# Add pipe geometry needed for circular work-unit construction

network_2027["Pipe length"] = np.sqrt(
    (network_2027["GPS x2"] - network_2027["GPS x1"])**2
    +
    (network_2027["GPS y2"] - network_2027["GPS y1"])**2
)

network_2027["Midpoint x"] = (
    network_2027["GPS x1"] + network_2027["GPS x2"]
) / 2

network_2027["Midpoint y"] = (
    network_2027["GPS y1"] + network_2027["GPS y2"]
) / 2


iron_materials = [
    "wrought iron",
    "cast iron",
    "gray iron"
]

network_2027["Is_iron_2027"] = (
    network_2027["Material_2027"]
    .isin(iron_materials)
)

network_2027["Is_non_PU_2027"] = (
    network_2027["Material_2027"]
    .notna()
    &
    (network_2027["Material_2027"] != "polyurethane")
)


print("Physical pipe segments:", len(network_2027))
print("Iron segments remaining:", network_2027["Is_iron_2027"].sum())
print("Non-PU segments:", network_2027["Is_non_PU_2027"].sum())

print(
    "Total remaining iron length:",
    network_2027.loc[
        network_2027["Is_iron_2027"],
        "Pipe length"
    ].sum()
)

display(
    network_2027[
        [
            "Pipe ID",
            "Material_2027",
            "Surface",
            "Pipe length",
            "Midpoint x",
            "Midpoint y",
            "Is_iron_2027",
            "Is_non_PU_2027"
        ]
    ].head()
)

Physical pipe segments: 43039
Iron segments remaining: 26237
Non-PU segments: 27240
Total remaining iron length: 440677.3230412878


,Pipe ID,Material_2027,Surface,Pipe length,Midpoint x,Midpoint y,Is_iron_2027,Is_non_PU_2027
0,P01001,copper,grassland,12.140000,6130.220,5020.970,False,True
1,P01002,polyurethane,water,59.519637,6524.470,4174.760,False,False
2,P01003,polyurethane,water,18.170003,6527.505,4201.935,False,False
3,P01004,cast iron,structure,29.750686,2842.740,3391.535,True,True
4,P01005,polyurethane,water,25.897695,5121.825,4285.035,False,False


In [145]:
# Create one baseline circular work-unit candidate for each remaining iron pipe

iron_2027 = network_2027[
    network_2027["Is_iron_2027"]
].copy()

work_units = iron_2027[
    [
        "Pipe ID",
        "Surface",
        "Pipe length",
        "Midpoint x",
        "Midpoint y"
    ]
].copy()

work_units = work_units.rename(
    columns={
        "Pipe ID": "Center Pipe ID",
        "Midpoint x": "Center x",
        "Midpoint y": "Center y"
    }
)

work_units["Radius"] = (
    work_units["Pipe length"] / 2
    + 1e-6
)

work_units["Area"] = (
    np.pi * work_units["Radius"]**2
)

print("Candidate work units:", len(work_units))

display(
    work_units.head(10)
)

Candidate work units: 26237


,Center Pipe ID,Surface,Pipe length,Center x,Center y,Radius,Area
3,P01004,structure,29.750686,2842.740,3391.535,14.875344,695.158600
5,P01006,structure,41.926084,3734.360,5100.305,20.963043,1380.570274
6,P01007,structure,12.150000,3728.295,5099.050,6.075001,115.942479
7,P01008,structure,22.918440,3731.335,5088.000,11.459221,412.534306
8,P01009,water,45.490000,2592.525,4299.010,22.745001,1625.256057
9,P01010,water,28.378337,5766.280,4269.680,14.189169,632.504792
11,P01012,water,5.852350,4877.580,3930.540,2.926176,26.899905
12,P01013,water,5.823075,4871.755,3930.545,2.911538,26.631456
13,P01014,water,5.744258,4883.315,3931.300,2.872130,25.915409
15,P01016,water,12.317824,4901.820,3939.930,6.158913,119.167560


In [146]:
# Find all physical pipe segments fully contained inside each candidate circle

from scipy.spatial import cKDTree

pipe_xy = network_2027[
    ["Midpoint x", "Midpoint y"]
].to_numpy()

pipe_tree = cKDTree(pipe_xy)

pipe_lengths = network_2027["Pipe length"].to_numpy()

x1 = network_2027["GPS x1"].to_numpy()
y1 = network_2027["GPS y1"].to_numpy()
x2 = network_2027["GPS x2"].to_numpy()
y2 = network_2027["GPS y2"].to_numpy()

pipe_ids = network_2027["Pipe ID"].to_numpy()


covered_pipe_ids = []
covered_pipe_counts = []

max_half_length = pipe_lengths.max() / 2


for _, unit in work_units.iterrows():

    cx = unit["Center x"]
    cy = unit["Center y"]
    radius = unit["Radius"]

    # First find nearby pipes that could possibly fit inside the circle
    nearby_idx = pipe_tree.query_ball_point(
        [cx, cy],
        radius + max_half_length
    )

    nearby_idx = np.asarray(nearby_idx, dtype=int)

    # Exact test: both endpoints must lie inside the circle
    d1_sq = (
        (x1[nearby_idx] - cx)**2
        + (y1[nearby_idx] - cy)**2
    )

    d2_sq = (
        (x2[nearby_idx] - cx)**2
        + (y2[nearby_idx] - cy)**2
    )

    radius_sq = radius**2

    fully_inside = (
        (d1_sq <= radius_sq)
        & (d2_sq <= radius_sq)
    )

    ids_inside = pipe_ids[
        nearby_idx[fully_inside]
    ].tolist()

    covered_pipe_ids.append(ids_inside)
    covered_pipe_counts.append(len(ids_inside))


work_units["Covered Pipe IDs"] = covered_pipe_ids
work_units["Covered Pipe Count"] = covered_pipe_counts


print(
    "Work units:",
    len(work_units)
)

print(
    "Minimum covered pipes:",
    work_units["Covered Pipe Count"].min()
)

print(
    "Maximum covered pipes:",
    work_units["Covered Pipe Count"].max()
)

print(
    "Mean covered pipes:",
    work_units["Covered Pipe Count"].mean()
)

print(
    "Work units covering only their center pipe:",
    (work_units["Covered Pipe Count"] == 1).sum()
)

display(
    work_units[
        [
            "Center Pipe ID",
            "Radius",
            "Covered Pipe Count",
            "Covered Pipe IDs"
        ]
    ]
    .sort_values(
        "Covered Pipe Count",
        ascending=False
    )
    .head(10)
)

Work units: 26237
Minimum covered pipes: 1
Maximum covered pipes: 36
Mean covered pipes: 1.3646758394633534
Work units covering only their center pipe: 22509


,Center Pipe ID,Radius,Covered Pipe Count,Covered Pipe IDs
5392,P06393,27.657697,36,"[P32751, P13222, P13225, P13221, P18856, P1335..."
8784,P09785,26.013675,35,"[P43004, P43221, P09778, P09777, P43228, P0978..."
5065,P06066,36.099771,35,"[P19613, P19627, P19612, P22719, P22724, P1962..."
3521,P04522,29.089326,34,"[P29284, P29280, P29285, P37025, P29283, P2928..."
4829,P05830,22.483602,33,"[P09689, P23173, P43534, P21620, P05830, P0968..."
4511,P05512,26.153340,32,"[P20909, P33988, P39520, P20908, P33989, P1266..."
257,P01258,33.360001,30,"[P36702, P11479, P11480, P11478, P11594, P1159..."
7629,P08630,27.695431,30,"[P31041, P31040, P38243, P31042, P14206, P3106..."
5554,P06555,30.615736,28,"[P12633, P11702, P12635, P11709, P12631, P1263..."
5393,P06394,25.907978,28,"[P32446, P32447, P32485, P13364, P13362, P3248..."


In [147]:
# Calculate the initial construction cost of each candidate work unit

pipe_info = network_2027.set_index("Pipe ID")

covered_non_pu_lengths = []
covered_non_pu_counts = []
covered_iron_counts = []

for ids in work_units["Covered Pipe IDs"]:

    covered = pipe_info.loc[ids]

    non_pu = covered[
        covered["Is_non_PU_2027"]
    ]

    iron = covered[
        covered["Is_iron_2027"]
    ]

    covered_non_pu_lengths.append(
        non_pu["Pipe length"].sum()
    )

    covered_non_pu_counts.append(
        len(non_pu)
    )

    covered_iron_counts.append(
        len(iron)
    )


work_units["Non-PU Pipe Count"] = (
    covered_non_pu_counts
)

work_units["Non-PU Pipe Meters"] = (
    covered_non_pu_lengths
)

work_units["Covered Iron Count"] = (
    covered_iron_counts
)


# Calculate each component of the professor's cost equation

work_units["Groundbreaking Cost"] = (
    GROUNDBREAKING_COST
)

work_units["Surface Cost"] = (
    work_units["Surface"].map(SURFACE_RATES)
    * work_units["Area"]**0.85
)

work_units["Pipe Cost"] = (
    NON_PU_PIPE_COST_PER_METER
    * work_units["Non-PU Pipe Meters"]
)

work_units["Construction Cost"] = (
    work_units["Groundbreaking Cost"]
    + work_units["Surface Cost"]
    + work_units["Pipe Cost"]
)


print(
    "Missing construction costs:",
    work_units["Construction Cost"].isna().sum()
)

print(
    "Mean construction cost:",
    work_units["Construction Cost"].mean()
)

print(
    "Median construction cost:",
    work_units["Construction Cost"].median()
)

print(
    "Minimum construction cost:",
    work_units["Construction Cost"].min()
)

print(
    "Maximum construction cost:",
    work_units["Construction Cost"].max()
)


display(
    work_units[
        [
            "Center Pipe ID",
            "Surface",
            "Radius",
            "Covered Pipe Count",
            "Covered Iron Count",
            "Non-PU Pipe Count",
            "Non-PU Pipe Meters",
            "Groundbreaking Cost",
            "Surface Cost",
            "Pipe Cost",
            "Construction Cost"
        ]
    ]
    .sort_values(
        "Construction Cost",
        ascending=False
    )
    .head(10)
)

Missing construction costs: 0
Mean construction cost: 7539.20254385351
Median construction cost: 3051.4090269336366
Minimum construction cost: 158.95995388676172
Maximum construction cost: 109427.31480683602


,Center Pipe ID,Surface,Radius,Covered Pipe Count,Covered Iron Count,Non-PU Pipe Count,Non-PU Pipe Meters,Groundbreaking Cost,Surface Cost,Pipe Cost,Construction Cost
6463,P07464,structure,37.440723,27,12,15,233.882774,100,62550.760009,46776.554798,109427.314807
3521,P04522,structure,29.089326,34,28,28,290.927852,100,40728.143507,58185.570495,99013.714002
42581,P43582,structure,34.453363,11,10,10,217.303605,100,54305.165451,43460.721075,97865.886526
5876,P06877,water,35.926936,5,5,5,277.745403,100,37320.002751,55549.080509,92969.083260
1080,P02081,structure,33.918405,17,3,7,181.786407,100,52879.533011,36357.281398,89336.814408
3583,P04584,structure,35.300001,3,3,3,151.486349,100,56593.221484,30297.269897,86990.491381
10152,P11153,water,34.974352,22,21,21,252.563307,100,35653.471879,50512.661474,86266.133353
7272,P08273,structure,36.268658,3,3,3,127.127317,100,59258.535495,25425.463455,84783.998950
1166,P02167,structure,36.105002,5,5,5,118.855873,100,58804.686467,23771.174627,82675.861094
4526,P05527,structure,33.365031,9,9,9,154.373631,100,51421.294953,30874.726263,82396.021216


In [148]:
# Sum the 2027 expected leak loss avoided by replacing all covered non-PU pipes

expected_loss_map = (
    prediction_2027
    .set_index("Pipe ID")["expected_leak_loss"]
    .to_dict()
)

work_unit_benefits = []

for ids in work_units["Covered Pipe IDs"]:

    benefit = sum(
        expected_loss_map.get(pipe_id, 0)
        for pipe_id in ids
    )

    work_unit_benefits.append(benefit)


work_units["Expected Leak Benefit 2027"] = (
    work_unit_benefits
)

work_units["Net Benefit 2027"] = (
    work_units["Expected Leak Benefit 2027"]
    - work_units["Construction Cost"]
)

work_units["Benefit Cost Ratio 2027"] = (
    work_units["Expected Leak Benefit 2027"]
    / work_units["Construction Cost"]
)


print(
    "Work units with positive 2027 net benefit:",
    (work_units["Net Benefit 2027"] > 0).sum()
)

print(
    "Work units with benefit-cost ratio > 1:",
    (work_units["Benefit Cost Ratio 2027"] > 1).sum()
)

print(
    "Maximum benefit-cost ratio:",
    work_units["Benefit Cost Ratio 2027"].max()
)


display(
    work_units[
        [
            "Center Pipe ID",
            "Surface",
            "Covered Pipe Count",
            "Covered Iron Count",
            "Construction Cost",
            "Expected Leak Benefit 2027",
            "Net Benefit 2027",
            "Benefit Cost Ratio 2027"
        ]
    ]
    .sort_values(
        "Benefit Cost Ratio 2027",
        ascending=False
    )
    .head(20)
)

Work units with positive 2027 net benefit: 1416
Work units with benefit-cost ratio > 1: 1416
Maximum benefit-cost ratio: 197.0182930348048


,Center Pipe ID,Surface,Covered Pipe Count,Covered Iron Count,Construction Cost,Expected Leak Benefit 2027,Net Benefit 2027,Benefit Cost Ratio 2027
18682,P19683,road,1,1,634.985406,125103.740772,124468.755367,197.018293
17357,P18358,road,1,1,569.634534,69119.824804,68550.190270,121.340650
14218,P15219,road,1,1,962.283831,115399.546990,114437.263159,119.922567
16426,P17427,road,1,1,1106.316960,131725.626437,130619.309477,119.066806
10013,P11014,road,1,1,611.678328,69127.405963,68515.727635,113.012678
19307,P20308,road,1,1,1084.436921,103339.956426,102255.519506,95.293654
16425,P17426,road,1,1,1425.835176,131875.598604,130449.763428,92.490072
18862,P19863,road,2,2,2502.388215,228473.929327,225971.541112,91.302352
16424,P17425,road,1,1,1572.153209,132091.595157,130519.441948,84.019544
14327,P15328,road,1,1,1386.132555,116343.771104,114957.638549,83.934087


In [149]:
# Inspect the prediction components behind the highest-BCR work units

top_bcr = (
    work_units
    .sort_values(
        "Benefit Cost Ratio 2027",
        ascending=False
    )
    .head(20)
)

top_pipe_ids = top_bcr[
    "Center Pipe ID"
].tolist()

top_diagnostic = (
    network_2027[
        [
            "Pipe ID",
            "Material_2027",
            "Surface",
            "Pipe length"
        ]
    ]
    .merge(
        prediction_2027[
            [
                "Pipe ID",
                "raw_ensemble_probability",
                "predicted_leak_cost",
                "expected_leak_loss"
            ]
        ],
        on="Pipe ID",
        how="inner"
    )
)

top_diagnostic = top_diagnostic[
    top_diagnostic["Pipe ID"].isin(top_pipe_ids)
]

top_diagnostic = top_diagnostic.merge(
    top_bcr[
        [
            "Center Pipe ID",
            "Construction Cost",
            "Benefit Cost Ratio 2027"
        ]
    ],
    left_on="Pipe ID",
    right_on="Center Pipe ID",
    how="left"
)

display(
    top_diagnostic[
        [
            "Pipe ID",
            "Material_2027",
            "Surface",
            "Pipe length",
            "raw_ensemble_probability",
            "predicted_leak_cost",
            "expected_leak_loss",
            "Construction Cost",
            "Benefit Cost Ratio 2027"
        ]
    ]
    .sort_values(
        "Benefit Cost Ratio 2027",
        ascending=False
    )
)

,Pipe ID,Material_2027,Surface,Pipe length,raw_ensemble_probability,predicted_leak_cost,expected_leak_loss,Construction Cost,Benefit Cost Ratio 2027
14,P19683,wrought iron,road,2.373942,0.564869,221474.081887,125103.740772,634.985406,197.018293
12,P18358,wrought iron,road,2.103188,0.312578,221127.919146,69119.824804,569.634534,121.340650
4,P15219,wrought iron,road,3.677893,0.519060,222324.263691,115399.546990,962.283831,119.922567
10,P17427,wrought iron,road,4.228499,0.592956,222150.573357,131725.626437,1106.316960,119.066806
1,P11014,wrought iron,road,2.277828,0.312578,221152.172764,69127.405963,611.678328,113.012678
18,P20308,wrought iron,road,4.145660,0.460598,224360.471547,103339.956426,1084.436921,95.293654
9,P17426,wrought iron,road,5.408752,0.593135,222336.689860,131875.598604,1425.835176,92.490072
17,P19863,wrought iron,road,5.456189,0.513537,222612.236653,114319.722443,2502.388215,91.302352
8,P17425,wrought iron,road,5.932639,0.593881,222421.092502,132091.595157,1572.153209,84.019544
5,P15328,wrought iron,road,5.264903,0.523224,222359.268907,116343.771104,1386.132555,83.934087


# year 1 simulation
\[
\$10M\ budget
\rightarrow
\text{select work unit}
\rightarrow
\text{replace fully covered non-PU pipes}
\rightarrow
\text{update network}
\rightarrow
\text{recalculate overlapping candidates}
\rightarrow\cdots
\]

In [150]:
# Initialize the Year 1 simulation state and annual budget

ANNUAL_BUDGET = 10_000_000

year_1_budget = ANNUAL_BUDGET
year_1_spent = 0

simulation_network = network_2027.copy()

simulation_network["Current Material"] = (
    simulation_network["Material_2027"]
)

simulation_network["Replaced in 2027"] = False


# Fast lookup tables used during the simulation
pipe_length_map = (
    simulation_network
    .set_index("Pipe ID")["Pipe length"]
    .to_dict()
)

pipe_surface_map = (
    simulation_network
    .set_index("Pipe ID")["Surface"]
    .to_dict()
)

current_material = (
    simulation_network
    .set_index("Pipe ID")["Current Material"]
    .to_dict()
)

expected_loss_2027 = (
    prediction_2027
    .set_index("Pipe ID")["expected_leak_loss"]
    .to_dict()
)


print("Year 1 starting budget:", year_1_budget)
print("Physical pipes:", len(simulation_network))

print(
    "Iron pipes remaining:",
    simulation_network["Current Material"]
    .isin(iron_materials)
    .sum()
)

print(
    "Starting expected leak loss:",
    sum(expected_loss_2027.values())
)

Year 1 starting budget: 10000000
Physical pipes: 43039
Iron pipes remaining: 26237
Starting expected leak loss: 49451671.52484188


In [151]:
# Recalculate a work unit's cost and benefit using the current simulation state

def evaluate_work_unit(unit, current_material, expected_loss):

    covered_ids = unit["Covered Pipe IDs"]

    # Pipes that are still non-polyurethane
    active_non_pu = [
        pipe_id
        for pipe_id in covered_ids
        if (
            pd.notna(current_material.get(pipe_id))
            and current_material.get(pipe_id) != "polyurethane"
        )
    ]

    # Total non-PU pipe length currently replaced by this work unit
    non_pu_meters = sum(
        pipe_length_map[pipe_id]
        for pipe_id in active_non_pu
    )

    # Construction cost under the current network state
    surface_cost = (
        SURFACE_RATES[unit["Surface"]]
        * unit["Area"]**0.85
    )

    pipe_cost = (
        NON_PU_PIPE_COST_PER_METER
        * non_pu_meters
    )

    total_cost = (
        GROUNDBREAKING_COST
        + surface_cost
        + pipe_cost
    )

    # Avoided 2027 expected leak loss
    benefit = sum(
        expected_loss.get(pipe_id, 0)
        for pipe_id in active_non_pu
    )

    net_benefit = benefit - total_cost

    benefit_cost_ratio = (
        benefit / total_cost
        if total_cost > 0
        else 0
    )

    return {
        "active_pipe_ids": active_non_pu,
        "active_pipe_count": len(active_non_pu),
        "non_pu_meters": non_pu_meters,
        "construction_cost": total_cost,
        "expected_benefit": benefit,
        "net_benefit": net_benefit,
        "benefit_cost_ratio": benefit_cost_ratio
    }

In [152]:
# Test the dynamic work-unit evaluation on the highest-ranked 2027 candidate

best_initial_unit = (
    work_units
    .sort_values(
        "Benefit Cost Ratio 2027",
        ascending=False
    )
    .iloc[0]
)

test_result = evaluate_work_unit(
    best_initial_unit,
    current_material,
    expected_loss_2027
)

print(
    "Center Pipe ID:",
    best_initial_unit["Center Pipe ID"]
)

print(
    "Active non-PU pipes:",
    test_result["active_pipe_count"]
)

print(
    "Construction cost:",
    test_result["construction_cost"]
)

print(
    "Expected benefit:",
    test_result["expected_benefit"]
)

print(
    "Net benefit:",
    test_result["net_benefit"]
)

print(
    "Benefit-cost ratio:",
    test_result["benefit_cost_ratio"]
)

Center Pipe ID: P19683
Active non-PU pipes: 1
Construction cost: 634.9854059000818
Expected benefit: 125103.74077244678
Net benefit: 124468.7553665467
Benefit-cost ratio: 197.0182930348048


In [153]:
# Run a baseline Year 1 strategy using dynamic benefit-cost evaluation

year_1_budget = ANNUAL_BUDGET
year_1_spent = 0

current_material = (
    network_2027
    .set_index("Pipe ID")["Material_2027"]
    .to_dict()
)

selected_work_units = []
replaced_pipe_ids = set()


# Start from highest initial benefit-cost ratio
candidate_order = (
    work_units
    .sort_values(
        "Benefit Cost Ratio 2027",
        ascending=False
    )
)


for _, unit in candidate_order.iterrows():

    result = evaluate_work_unit(
        unit,
        current_material,
        expected_loss_2027
    )

    # Skip candidates with no remaining non-PU pipes
    if result["active_pipe_count"] == 0:
        continue

    # Baseline strategy: only perform positive-net-benefit work
    if result["benefit_cost_ratio"] <= 1:
        continue

    # Skip if this work unit does not fit in the remaining budget
    if (
        year_1_spent
        + result["construction_cost"]
        > year_1_budget
    ):
        continue

    # Execute the work unit
    year_1_spent += result["construction_cost"]

    for pipe_id in result["active_pipe_ids"]:
        current_material[pipe_id] = "polyurethane"
        replaced_pipe_ids.add(pipe_id)

    selected_work_units.append({
        "Center Pipe ID": unit["Center Pipe ID"],
        "Radius": unit["Radius"],
        "Construction Cost": result["construction_cost"],
        "Expected Benefit": result["expected_benefit"],
        "Net Benefit": result["net_benefit"],
        "Benefit Cost Ratio": result["benefit_cost_ratio"],
        "Replaced Pipe Count": result["active_pipe_count"]
    })


year_1_plan = pd.DataFrame(selected_work_units)


remaining_iron = sum(
    material in iron_materials
    for material in current_material.values()
)


print("Selected work units:", len(year_1_plan))
print("2027 CAPEX:", year_1_spent)
print("Unused 2027 budget:", year_1_budget - year_1_spent)
print("Pipes replaced:", len(replaced_pipe_ids))
print("Iron pipes remaining:", remaining_iron)

print(
    "Expected leak loss avoided:",
    year_1_plan["Expected Benefit"].sum()
)

print(
    "Remaining expected 2027 leak loss:",
    sum(
        loss
        for pipe_id, loss in expected_loss_2027.items()
        if current_material.get(pipe_id) != "polyurethane"
    )
)

display(
    year_1_plan.head(20)
)

Selected work units: 1290
2027 CAPEX: 6618042.809921839
Unused 2027 budget: 3381957.1900781607
Pipes replaced: 1627
Iron pipes remaining: 24615
Expected leak loss avoided: 33697695.081827864
Remaining expected 2027 leak loss: 15735245.366291473


,Center Pipe ID,Radius,Construction Cost,Expected Benefit,Net Benefit,Benefit Cost Ratio,Replaced Pipe Count
0,P19683,1.186972,634.985406,125103.740772,124468.755367,197.018293,1
1,P18358,1.051595,569.634534,69119.824804,68550.190270,121.340650,1
2,P15219,1.838948,962.283831,115399.546990,114437.263159,119.922567,1
3,P17427,2.114250,1106.316960,131725.626437,130619.309477,119.066806,1
4,P11014,1.138915,611.678328,69127.405963,68515.727635,113.012678,1
5,P20308,2.072831,1084.436921,103339.956426,102255.519506,95.293654,1
6,P17426,2.704377,1425.835176,131875.598604,130449.763428,92.490072,1
7,P19863,2.728096,2502.388215,228473.929327,225971.541112,91.302352,2
8,P17425,2.966320,1572.153209,132091.595157,130519.441948,84.019544,1
9,P15328,2.632452,1386.132555,116343.771104,114957.638549,83.934087,1


# Year 2 = 2028
if works fine -> iterate

In [154]:
# Build the physical network state at the start of 2028

network_2028 = network_2027.copy()

network_2028["Material_2028"] = (
    network_2028["Pipe ID"]
    .map(current_material)
)

network_2028["Is_iron_2028"] = (
    network_2028["Material_2028"]
    .isin(iron_materials)
)

network_2028["Is_non_PU_2028"] = (
    network_2028["Material_2028"]
    .notna()
    &
    (network_2028["Material_2028"] != "polyurethane")
)


print(
    "Physical pipes:",
    len(network_2028)
)

print(
    "Iron pipes remaining:",
    network_2028["Is_iron_2028"].sum()
)

print(
    "Non-PU pipes remaining:",
    network_2028["Is_non_PU_2028"].sum()
)

print(
    "Polyurethane pipes:",
    (network_2028["Material_2028"] == "polyurethane").sum()
)

display(
    network_2028["Material_2028"]
    .value_counts(dropna=False)
)

Physical pipes: 43039
Iron pipes remaining: 24615
Non-PU pipes remaining: 25613
Polyurethane pipes: 17418


Material_2028
polyurethane    17418
wrought iron    15397
cast iron        7232
gray iron        1986
copper            693
brass             305
NaN                 8
Name: count, dtype: int64

In [156]:
# Build the 2028 risk set from pipes that remain non-polyurethane

risk_2028 = network_2028[
    network_2028["Is_non_PU_2028"]
].copy()

risk_2028["Year"] = 2028


# Update pipe age at the start of 2028
reference_date_2028 = pd.Timestamp("2028-01-01")

risk_2028["Age"] = (
    reference_date_2028
    - risk_2028["Lay date"]
).dt.days / 365.25


print("2028 pipes at risk:", len(risk_2028))

print(
    "Remaining iron pipes:",
    risk_2028["Material_2028"]
    .isin(iron_materials)
    .sum()
)

print(
    "Remaining copper/brass:",
    risk_2028["Material_2028"]
    .isin(["copper", "brass"])
    .sum()
)

print("\nMissing prediction features:")

display(
    risk_2028[
        [
            "Age",
            "Pipe length",
            "Midpoint x",
            "Midpoint y",
            "Material",
            "Surface"
        ]
    ]
    .isna()
    .sum()
)

2028 pipes at risk: 25613
Remaining iron pipes: 24615
Remaining copper/brass: 998

Missing prediction features:


Age            9
Pipe length    0
Midpoint x     0
Midpoint y     0
Material       0
Surface        0
dtype: int64

In [158]:
# Find trained model objects already stored in the notebook

model_names = [
    name for name, obj in globals().items()
    if (
        "xgb" in name.lower()
        or "cat" in name.lower()
        or "lgb" in name.lower()
    )
]

for name in sorted(model_names):
    obj = globals()[name]

    if hasattr(obj, "predict_proba"):
        print(
            name,
            "->",
            type(obj).__name__
        )

CatBoostClassifier -> type
LGBMClassifier -> type
XGBClassifier -> type
catboost_pipeline -> Pipeline
final_cat -> Pipeline
final_lgbm -> Pipeline
final_xgb -> Pipeline
spatial_xgb_pipeline -> Pipeline
xgb_pipeline -> Pipeline


In [159]:
# Predict 2028 leak probabilities using the trained production ensemble

risk_2028["Age"] = (
    risk_2028["Age"]
    .fillna(age_median)
)

leak_features_2028 = (
    numeric_features
    + categorical_features
)

xgb_prob_2028 = final_xgb.predict_proba(
    risk_2028[leak_features_2028]
)[:, 1]

cat_prob_2028 = final_cat.predict_proba(
    risk_2028[leak_features_2028]
)[:, 1]

lgbm_prob_2028 = final_lgbm.predict_proba(
    risk_2028[leak_features_2028]
)[:, 1]


leak_2028 = risk_2028[
    [
        "Pipe ID",
        "Material",
        "Surface",
        "Age",
        "Pipe length"
    ]
].copy()

leak_2028["xgb_probability"] = xgb_prob_2028
leak_2028["catboost_probability"] = cat_prob_2028
leak_2028["lightgbm_probability"] = lgbm_prob_2028

leak_2028["raw_ensemble_probability"] = (
    leak_2028["xgb_probability"]
    + leak_2028["catboost_probability"]
    + leak_2028["lightgbm_probability"]
) / 3


print("2028 pipes:", len(leak_2028))

print(
    "Expected number of 2028 leaks:",
    leak_2028["raw_ensemble_probability"].sum()
)

print(
    "Mean 2028 leak probability:",
    leak_2028["raw_ensemble_probability"].mean()
)

print(
    "Maximum 2028 leak probability:",
    leak_2028["raw_ensemble_probability"].max()
)

print(
    "Missing probabilities:",
    leak_2028["raw_ensemble_probability"].isna().sum()
)

display(
    leak_2028
    .sort_values(
        "raw_ensemble_probability",
        ascending=False
    )
    .head(10)
)

2028 pipes: 25613
Expected number of 2028 leaks: 2632.6541221306115
Mean 2028 leak probability: 0.10278585570337764
Maximum 2028 leak probability: 0.7369174274878555
Missing probabilities: 0


,Pipe ID,Material,Surface,Age,Pipe length,xgb_probability,catboost_probability,lightgbm_probability,raw_ensemble_probability
14408,P15409,wrought iron,grassland,62.376454,67.997565,0.694753,0.772390,0.743609,0.736917
8571,P09572,gray iron,water,62.825462,53.527032,0.662408,0.742865,0.648926,0.684733
22845,P23846,wrought iron,grassland,62.318960,27.538656,0.747525,0.506808,0.797465,0.683933
14297,P15298,wrought iron,grassland,62.370979,63.302310,0.657217,0.774824,0.598382,0.676808
8609,P09610,gray iron,water,62.830938,54.999995,0.641529,0.742808,0.643137,0.675825
8610,P09611,gray iron,water,62.830938,54.990655,0.641529,0.742808,0.634793,0.673043
8568,P09569,gray iron,water,62.825462,52.527259,0.614527,0.755646,0.626653,0.665609
7370,P08371,gray iron,water,62.841889,53.256271,0.639120,0.727716,0.628586,0.665141
6573,P07574,gray iron,grassland,62.327173,66.020628,0.624073,0.734985,0.621788,0.660282
8232,P09233,cast iron,grassland,62.313484,51.989537,0.632966,0.718818,0.628736,0.660173


In [161]:
# Find calibration-related variables already stored in the notebook

for name, value in globals().items():
    if (
        "calib" in name.lower()
        or "factor" in name.lower()
        or "smear" in name.lower()
    ):
        if isinstance(value, (int, float, np.integer, np.floating)):
            print(name, "=", value)

smearing_factor = 1.973057459427679
calibration_factor = 0.983134342974772
final_cost_calibration = 0.9670054441886302


In [162]:
# Apply the selected production cost calibration to the 2028 predictions

cost_2028["predicted_leak_cost"] = (
    np.maximum(raw_cost_2028, 0)
    * final_cost_calibration
)

print(
    "Mean predicted conditional cost:",
    cost_2028["predicted_leak_cost"].mean()
)

print(
    "Median predicted conditional cost:",
    cost_2028["predicted_leak_cost"].median()
)

print(
    "Zero-cost predictions:",
    (cost_2028["predicted_leak_cost"] == 0).sum()
)

print(
    "Missing cost predictions:",
    cost_2028["predicted_leak_cost"].isna().sum()
)

Mean predicted conditional cost: 13110.211808355034
Median predicted conditional cost: 0.0
Zero-cost predictions: 13854
Missing cost predictions: 0


In [163]:
# Combine 2028 leak probability and conditional leak cost

prediction_2028 = (
    leak_2028[
        [
            "Pipe ID",
            "raw_ensemble_probability"
        ]
    ]
    .merge(
        cost_2028[
            [
                "Pipe ID",
                "predicted_leak_cost"
            ]
        ],
        on="Pipe ID",
        how="inner"
    )
)

prediction_2028["expected_leak_loss"] = (
    prediction_2028["raw_ensemble_probability"]
    * prediction_2028["predicted_leak_cost"]
)


print("2028 predicted pipes:", len(prediction_2028))

print(
    "Expected number of leaks:",
    prediction_2028[
        "raw_ensemble_probability"
    ].sum()
)

print(
    "Total expected leak loss:",
    prediction_2028[
        "expected_leak_loss"
    ].sum()
)

print(
    "Mean expected leak loss per pipe:",
    prediction_2028[
        "expected_leak_loss"
    ].mean()
)

print(
    "Pipes with zero expected leak loss:",
    (
        prediction_2028["expected_leak_loss"] == 0
    ).sum()
)

display(
    prediction_2028
    .sort_values(
        "expected_leak_loss",
        ascending=False
    )
    .head(10)
)

2028 predicted pipes: 25613
Expected number of leaks: 2632.6541221306115
Total expected leak loss: 14828403.334066296
Mean expected leak loss per pipe: 578.9405120082105
Pipes with zero expected leak loss: 13854


,Pipe ID,raw_ensemble_probability,predicted_leak_cost,expected_leak_loss
9086,P15900,0.403302,223474.543291,90127.792903
25385,P43582,0.606104,45381.748346,27506.051840
4519,P08409,0.644450,42653.139068,27487.807490
4517,P08402,0.644450,42564.939594,27430.967355
4867,P09134,0.606744,44442.633361,26965.299807
4865,P09132,0.604575,44445.313225,26870.535698
4887,P09198,0.599697,44478.387920,26673.555091
4888,P09199,0.597429,44480.228072,26573.779178
4891,P09204,0.593965,44511.942128,26438.556066
4890,P09201,0.593245,44552.608323,26430.630434


confirms the cost extrapolation problem is now large enough that we should fix it before simulating 2028 construction or moving to 2029

In [164]:
# Define a cost model that does not use calendar year

cost_numeric_no_year = [
    "Age at leak",
    "Pipe length",
    "Midpoint x",
    "Midpoint y"
]

cost_categorical_no_year = [
    "Material",
    "Surface"
]

cost_features_no_year = (
    cost_numeric_no_year
    + cost_categorical_no_year
)

cost_preprocessor_no_year = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            cost_categorical_no_year
        ),
        (
            "num",
            "passthrough",
            cost_numeric_no_year
        )
    ]
)

cost_linear_no_year = Pipeline(
    steps=[
        (
            "preprocessor",
            cost_preprocessor_no_year
        ),
        (
            "model",
            LinearRegression()
        )
    ]
)

print("Numeric features:", cost_numeric_no_year)
print("Categorical features:", cost_categorical_no_year)
print("Leak year included:", "Leak year" in cost_features_no_year)

Numeric features: ['Age at leak', 'Pipe length', 'Midpoint x', 'Midpoint y']
Categorical features: ['Material', 'Surface']
Leak year included: False


In [166]:
# Inspect the existing temporal fold structure

print(type(cost_temporal_folds))
print()

for i, fold in enumerate(cost_temporal_folds, start=1):
    print(f"Fold {i}:")
    print(fold)
    print("Type:", type(fold))
    print()

<class 'list'>

Fold 1:
{'train_years': [2019, 2020, 2021, 2022], 'test_year': 2023}
Type: <class 'dict'>

Fold 2:
{'train_years': [2019, 2020, 2021, 2022, 2023], 'test_year': 2024}
Type: <class 'dict'>

Fold 3:
{'train_years': [2019, 2020, 2021, 2022, 2023, 2024], 'test_year': 2025}
Type: <class 'dict'>

Fold 4:
{'train_years': [2019, 2020, 2021, 2022, 2023, 2024, 2025], 'test_year': 2026}
Type: <class 'dict'>



In [167]:
# Validate the no-Year cost model using the same temporal folds

no_year_results = []
no_year_oof = []

for fold_num, fold in enumerate(
    cost_temporal_folds,
    start=1
):

    train_years = fold["train_years"]
    test_year = fold["test_year"]

    train_fold = cost_model_data[
        cost_model_data["Leak year"].isin(train_years)
    ].copy()

    test_fold = cost_model_data[
        cost_model_data["Leak year"] == test_year
    ].copy()

    model = clone(cost_linear_no_year)

    model.fit(
        train_fold[cost_features_no_year],
        train_fold["Cost"]
    )

    # Calculate calibration using training data only
    train_pred = model.predict(
        train_fold[cost_features_no_year]
    )

    train_pred_clipped = np.maximum(
        train_pred,
        0
    )

    calibration = (
        train_fold["Cost"].mean()
        / train_pred_clipped.mean()
    )

    # Predict the unseen future year
    test_pred = model.predict(
        test_fold[cost_features_no_year]
    )

    test_pred = (
        np.maximum(test_pred, 0)
        * calibration
    )

    rmse = np.sqrt(
        mean_squared_error(
            test_fold["Cost"],
            test_pred
        )
    )

    mae = mean_absolute_error(
        test_fold["Cost"],
        test_pred
    )

    no_year_results.append({
        "Fold": fold_num,
        "Test Year": test_year,
        "RMSE": rmse,
        "MAE": mae,
        "Actual Mean": test_fold["Cost"].mean(),
        "Predicted Mean": test_pred.mean(),
        "Calibration Factor": calibration
    })

    no_year_oof.append(
        pd.DataFrame({
            "Pipe ID": test_fold["Pipe ID"].values,
            "Year": test_year,
            "Actual Cost": test_fold["Cost"].values,
            "Predicted Cost": test_pred
        })
    )


no_year_results = pd.DataFrame(no_year_results)

no_year_oof = pd.concat(
    no_year_oof,
    ignore_index=True
)

display(no_year_results)

,Fold,Test Year,RMSE,MAE,Actual Mean,Predicted Mean,Calibration Factor
0,1,2023,159579.395717,37677.802948,23404.630800,34097.979508,0.942507
1,2,2024,100667.774277,26155.374286,14379.554669,25895.509670,0.951033
2,3,2025,114653.746062,29745.186189,24058.591957,31381.651309,0.958492
3,4,2026,128499.719297,28388.847701,23386.672041,27268.143402,0.966933


Removing Leak year does not destroy RMSE, but it noticeably worsens aggregate cost calibration in every validation year

In [168]:
# Calculate combined OOF performance for the no-Year cost model

no_year_rmse = np.sqrt(
    mean_squared_error(
        no_year_oof["Actual Cost"],
        no_year_oof["Predicted Cost"]
    )
)

no_year_mae = mean_absolute_error(
    no_year_oof["Actual Cost"],
    no_year_oof["Predicted Cost"]
)

actual_total = no_year_oof["Actual Cost"].sum()

predicted_total = no_year_oof["Predicted Cost"].sum()

total_error_pct = (
    (predicted_total - actual_total)
    / actual_total
    * 100
)


print("No-Year combined RMSE:", no_year_rmse)
print("No-Year combined MAE:", no_year_mae)

print(
    "Actual total cost:",
    actual_total
)

print(
    "Predicted total cost:",
    predicted_total
)

print(
    "Total cost error (%):",
    total_error_pct
)

No-Year combined RMSE: 127552.71168534651
No-Year combined MAE: 30134.752182476586
Actual total cost: 184449150.94
Predicted total cost: 251197141.4146896
Total cost error (%): 36.187746126520395


In [169]:
# Test a production-safe cost model by freezing Year at the latest observed training year

frozen_year_results = []
frozen_year_oof = []

for fold_num, fold in enumerate(
    cost_temporal_folds,
    start=1
):

    train_years = fold["train_years"]
    test_year = fold["test_year"]

    train_fold = cost_model_data[
        cost_model_data["Leak year"].isin(train_years)
    ].copy()

    test_fold = cost_model_data[
        cost_model_data["Leak year"] == test_year
    ].copy()

    model = clone(final_cost_model)

    model.fit(
        train_fold[cost_features],
        train_fold["Cost"]
    )

    # Calculate calibration using training data only
    train_pred = model.predict(
        train_fold[cost_features]
    )

    train_pred_clipped = np.maximum(
        train_pred,
        0
    )

    calibration = (
        train_fold["Cost"].mean()
        / train_pred_clipped.mean()
    )

    # Freeze calendar year at the latest observed training year
    test_for_prediction = test_fold.copy()

    test_for_prediction["Leak year"] = max(
        train_years
    )

    test_pred = model.predict(
        test_for_prediction[cost_features]
    )

    test_pred = (
        np.maximum(test_pred, 0)
        * calibration
    )

    rmse = np.sqrt(
        mean_squared_error(
            test_fold["Cost"],
            test_pred
        )
    )

    mae = mean_absolute_error(
        test_fold["Cost"],
        test_pred
    )

    frozen_year_results.append({
        "Fold": fold_num,
        "Test Year": test_year,
        "Frozen Year": max(train_years),
        "RMSE": rmse,
        "MAE": mae,
        "Actual Mean": test_fold["Cost"].mean(),
        "Predicted Mean": test_pred.mean(),
        "Calibration Factor": calibration
    })

    frozen_year_oof.append(
        pd.DataFrame({
            "Pipe ID": test_fold["Pipe ID"].values,
            "Year": test_year,
            "Actual Cost": test_fold["Cost"].values,
            "Predicted Cost": test_pred
        })
    )


frozen_year_results = pd.DataFrame(
    frozen_year_results
)

frozen_year_oof = pd.concat(
    frozen_year_oof,
    ignore_index=True
)

display(frozen_year_results)

,Fold,Test Year,Frozen Year,RMSE,MAE,Actual Mean,Predicted Mean,Calibration Factor
0,1,2023,2022,159537.539318,36829.358627,23404.630800,33004.429911,0.941673
1,2,2024,2023,100191.975343,22835.973802,14379.554669,21784.727504,0.950923
2,3,2025,2024,114339.010257,27307.621391,24058.591957,26021.379594,0.954088
3,4,2026,2025,128604.273004,25595.245596,23386.672041,20971.886432,0.957882


In [170]:
# Calculate combined OOF performance for the frozen-Year cost model

frozen_rmse = np.sqrt(
    mean_squared_error(
        frozen_year_oof["Actual Cost"],
        frozen_year_oof["Predicted Cost"]
    )
)

frozen_mae = mean_absolute_error(
    frozen_year_oof["Actual Cost"],
    frozen_year_oof["Predicted Cost"]
)

frozen_actual_total = (
    frozen_year_oof["Actual Cost"].sum()
)

frozen_predicted_total = (
    frozen_year_oof["Predicted Cost"].sum()
)

frozen_total_error_pct = (
    (frozen_predicted_total - frozen_actual_total)
    / frozen_actual_total
    * 100
)


print("Frozen-Year combined RMSE:", frozen_rmse)
print("Frozen-Year combined MAE:", frozen_mae)

print(
    "Actual total cost:",
    frozen_actual_total
)

print(
    "Predicted total cost:",
    frozen_predicted_total
)

print(
    "Total cost error (%):",
    frozen_total_error_pct
)

Frozen-Year combined RMSE: 127435.35272852344
Frozen-Year combined MAE: 27708.42113583912
Actual total cost: 184449150.94
Predicted total cost: 212550580.5212021
Total cost error (%): 15.2353260711638


In [171]:
# Recalculate 2028 cost using 2026 as the frozen calendar-year input

cost_2028_frozen = risk_2028.copy()

cost_2028_frozen["Age at leak"] = (
    cost_2028_frozen["Age"]
    .fillna(age_median)
)

# Freeze calendar year at the last observed year
cost_2028_frozen["Leak year"] = 2026

cost_2028_frozen["Material"] = (
    cost_2028_frozen["Material"]
    .fillna("Unknown")
)

raw_cost_2028_frozen = final_cost_model.predict(
    cost_2028_frozen[cost_features]
)

cost_2028_frozen["predicted_leak_cost"] = (
    np.maximum(raw_cost_2028_frozen, 0)
    * final_cost_calibration
)


print(
    "Mean predicted conditional cost:",
    cost_2028_frozen["predicted_leak_cost"].mean()
)

print(
    "Median predicted conditional cost:",
    cost_2028_frozen["predicted_leak_cost"].median()
)

print(
    "Zero-cost predictions:",
    (cost_2028_frozen["predicted_leak_cost"] == 0).sum()
)

print(
    "Zero-cost percentage:",
    (
        cost_2028_frozen["predicted_leak_cost"] == 0
    ).mean() * 100
)

print(
    "Missing predictions:",
    cost_2028_frozen["predicted_leak_cost"].isna().sum()
)

Mean predicted conditional cost: 15689.271244395637
Median predicted conditional cost: 3761.2537397195088
Zero-cost predictions: 7334
Zero-cost percentage: 28.633896849256235
Missing predictions: 0


In [172]:
# Combine 2028 leak probabilities with Frozen-Year conditional costs

prediction_2028_frozen = (
    leak_2028[
        [
            "Pipe ID",
            "raw_ensemble_probability"
        ]
    ]
    .merge(
        cost_2028_frozen[
            [
                "Pipe ID",
                "predicted_leak_cost"
            ]
        ],
        on="Pipe ID",
        how="inner"
    )
)

prediction_2028_frozen["expected_leak_loss"] = (
    prediction_2028_frozen["raw_ensemble_probability"]
    * prediction_2028_frozen["predicted_leak_cost"]
)


print(
    "2028 predicted pipes:",
    len(prediction_2028_frozen)
)

print(
    "Expected number of leaks:",
    prediction_2028_frozen[
        "raw_ensemble_probability"
    ].sum()
)

print(
    "Total expected leak loss:",
    prediction_2028_frozen[
        "expected_leak_loss"
    ].sum()
)

print(
    "Mean expected leak loss per pipe:",
    prediction_2028_frozen[
        "expected_leak_loss"
    ].mean()
)

print(
    "Pipes with zero expected leak loss:",
    (
        prediction_2028_frozen[
            "expected_leak_loss"
        ] == 0
    ).sum()
)

display(
    prediction_2028_frozen
    .sort_values(
        "expected_leak_loss",
        ascending=False
    )
    .head(10)
)

2028 predicted pipes: 25613
Expected number of leaks: 2632.6541221306115
Total expected leak loss: 23235058.286329724
Mean expected leak loss per pipe: 907.1587977327812
Pipes with zero expected leak loss: 7334


,Pipe ID,raw_ensemble_probability,predicted_leak_cost,expected_leak_loss
9086,P15900,0.403302,227769.689480,91860.035155
4519,P08409,0.644450,46948.285258,30255.813648
4517,P08402,0.644450,46860.085784,30198.973513
25385,P43582,0.606104,49676.894535,30109.356430
4867,P09134,0.606744,48737.779551,29571.353858
4865,P09132,0.604575,48740.459415,29467.274715
4887,P09198,0.599697,48773.534109,29249.341307
4888,P09199,0.597429,48775.374262,29139.824168
4891,P09204,0.593965,48807.088317,28989.724537
4890,P09201,0.593245,48847.754513,28978.706200


In [173]:
# Initialize the Year 2 budget and current network state

year_2_budget = (
    ANNUAL_BUDGET
    + (year_1_budget - year_1_spent)
)

year_2_spent = 0

current_material_2028 = (
    network_2028
    .set_index("Pipe ID")["Material_2028"]
    .to_dict()
)

expected_loss_2028 = (
    prediction_2028_frozen
    .set_index("Pipe ID")["expected_leak_loss"]
    .to_dict()
)


print("2028 available budget:", year_2_budget)

print(
    "2028 starting iron pipes:",
    sum(
        material in iron_materials
        for material in current_material_2028.values()
    )
)

print(
    "2028 starting expected leak loss:",
    sum(expected_loss_2028.values())
)

2028 available budget: 13381957.190078162
2028 starting iron pipes: 24615
2028 starting expected leak loss: 23235058.286329724


In [174]:
# Make work-unit evaluation independent of a specific simulation year

def evaluate_work_unit_dynamic(
    unit,
    current_material,
    expected_loss,
    pipe_length_map
):
    covered_ids = unit["Covered Pipe IDs"]

    # Pipes that are currently non-polyurethane
    active_non_pu = [
        pipe_id
        for pipe_id in covered_ids
        if (
            pd.notna(current_material.get(pipe_id))
            and current_material.get(pipe_id) != "polyurethane"
        )
    ]

    # Current non-PU pipe length inside the work unit
    non_pu_meters = sum(
        pipe_length_map[pipe_id]
        for pipe_id in active_non_pu
    )

    # Construction cost
    surface_cost = (
        SURFACE_RATES[unit["Surface"]]
        * unit["Area"]**0.85
    )

    pipe_cost = (
        NON_PU_PIPE_COST_PER_METER
        * non_pu_meters
    )

    construction_cost = (
        GROUNDBREAKING_COST
        + surface_cost
        + pipe_cost
    )

    # Expected leak loss avoided in the current year
    expected_benefit = sum(
        expected_loss.get(pipe_id, 0)
        for pipe_id in active_non_pu
    )

    net_benefit = (
        expected_benefit
        - construction_cost
    )

    benefit_cost_ratio = (
        expected_benefit / construction_cost
        if construction_cost > 0
        else 0
    )

    return {
        "active_pipe_ids": active_non_pu,
        "active_pipe_count": len(active_non_pu),
        "non_pu_meters": non_pu_meters,
        "construction_cost": construction_cost,
        "expected_benefit": expected_benefit,
        "net_benefit": net_benefit,
        "benefit_cost_ratio": benefit_cost_ratio
    }

In [175]:
# Evaluate every candidate under the starting 2028 network state

initial_2028_evaluations = []

for _, unit in work_units.iterrows():

    result = evaluate_work_unit_dynamic(
        unit,
        current_material_2028,
        expected_loss_2028,
        pipe_length_map
    )

    if result["active_pipe_count"] == 0:
        continue

    initial_2028_evaluations.append({
        "Center Pipe ID": unit["Center Pipe ID"],
        "Construction Cost": result["construction_cost"],
        "Expected Benefit": result["expected_benefit"],
        "Net Benefit": result["net_benefit"],
        "Benefit Cost Ratio": result["benefit_cost_ratio"],
        "Active Pipe Count": result["active_pipe_count"]
    })


initial_2028_evaluations = pd.DataFrame(
    initial_2028_evaluations
)

display(
    initial_2028_evaluations
    .sort_values(
        "Benefit Cost Ratio",
        ascending=False
    )
    .head(20)
)

print(
    "Candidates with positive net benefit:",
    (
        initial_2028_evaluations["Net Benefit"] > 0
    ).sum()
)

print(
    "Maximum 2028 BCR:",
    initial_2028_evaluations[
        "Benefit Cost Ratio"
    ].max()
)

,Center Pipe ID,Construction Cost,Expected Benefit,Net Benefit,Benefit Cost Ratio,Active Pipe Count
8370,P15900,11633.808195,96721.527843,85087.719648,8.313832,3
5566,P11345,192.714691,954.905973,762.191282,4.955024,1
9169,P17286,194.523302,953.849491,759.326189,4.903523,1
7699,P14533,401.146136,1740.853384,1339.707248,4.339699,1
7693,P14527,408.030214,1741.958944,1333.928730,4.269191,1
5467,P11174,1136.140416,4795.816824,3659.676408,4.221148,1
8457,P16142,440.968652,1782.284595,1341.315943,4.041749,1
8460,P16145,443.546832,1785.278203,1341.731371,4.025005,1
9296,P17462,1135.803832,4514.127289,3378.323458,3.974390,1
9721,P18308,1273.835360,4980.482316,3706.646957,3.909832,1


Candidates with positive net benefit: 1666
Maximum 2028 BCR: 8.313832084946878


In [176]:
# Execute the 2028 greedy replacement strategy using dynamic evaluation

year_2_spent = 0

selected_work_units_2028 = []
replaced_pipe_ids_2028 = set()


candidate_order_2028 = (
    initial_2028_evaluations
    .sort_values(
        "Benefit Cost Ratio",
        ascending=False
    )
)


for _, candidate in candidate_order_2028.iterrows():

    center_id = candidate["Center Pipe ID"]

    unit = work_units[
        work_units["Center Pipe ID"] == center_id
    ].iloc[0]

    # Recalculate after previous 2028 replacements
    result = evaluate_work_unit_dynamic(
        unit,
        current_material_2028,
        expected_loss_2028,
        pipe_length_map
    )

    # Nothing remains to replace
    if result["active_pipe_count"] == 0:
        continue

    # Only take positive-net-benefit work
    if result["benefit_cost_ratio"] <= 1:
        continue

    # Entire work unit must fit within available budget
    if (
        year_2_spent
        + result["construction_cost"]
        > year_2_budget
    ):
        continue

    # Execute the work unit
    year_2_spent += result["construction_cost"]

    for pipe_id in result["active_pipe_ids"]:
        current_material_2028[pipe_id] = "polyurethane"
        replaced_pipe_ids_2028.add(pipe_id)

    selected_work_units_2028.append({
        "Center Pipe ID": center_id,
        "Radius": unit["Radius"],
        "Construction Cost": result["construction_cost"],
        "Expected Benefit": result["expected_benefit"],
        "Net Benefit": result["net_benefit"],
        "Benefit Cost Ratio": result["benefit_cost_ratio"],
        "Replaced Pipe Count": result["active_pipe_count"]
    })


year_2_plan = pd.DataFrame(
    selected_work_units_2028
)


remaining_iron_2028 = sum(
    material in iron_materials
    for material in current_material_2028.values()
)

unused_budget_2028 = (
    year_2_budget - year_2_spent
)

remaining_expected_loss_2028 = sum(
    loss
    for pipe_id, loss in expected_loss_2028.items()
    if current_material_2028.get(pipe_id) != "polyurethane"
)


print("Selected work units:", len(year_2_plan))
print("2028 CAPEX:", year_2_spent)
print("Unused 2028 budget:", unused_budget_2028)
print("Pipes replaced:", len(replaced_pipe_ids_2028))
print("Iron pipes remaining:", remaining_iron_2028)

print(
    "Expected leak loss avoided:",
    year_2_plan["Expected Benefit"].sum()
)

print(
    "Remaining expected 2028 leak loss:",
    remaining_expected_loss_2028
)

display(
    year_2_plan.head(20)
)

Selected work units: 1556
2028 CAPEX: 4019058.100336434
Unused 2028 budget: 9362899.089741727
Pipes replaced: 1601
Iron pipes remaining: 23014
Expected leak loss avoided: 5486665.56990951
Remaining expected 2028 leak loss: 17748392.71642021


,Center Pipe ID,Radius,Construction Cost,Expected Benefit,Net Benefit,Benefit Cost Ratio,Replaced Pipe Count
0,P15900,13.437984,11633.808195,96721.527843,85087.719648,8.313832,3
1,P11345,0.228036,192.714691,954.905973,762.191282,4.955024,1
2,P17286,0.232434,194.523302,953.849491,759.326189,4.903523,1
3,P14533,0.740946,401.146136,1740.853384,1339.707248,4.339699,1
4,P14527,0.757695,408.030214,1741.958944,1333.928730,4.269191,1
5,P11174,1.938299,1136.140416,4795.816824,3659.676408,4.221148,1
6,P16142,0.837736,440.968652,1782.284595,1341.315943,4.041749,1
7,P16145,0.843994,443.546832,1785.278203,1341.731371,4.025005,1
8,P17462,1.937764,1135.803832,4514.127289,3378.323458,3.974390,1
9,P18308,2.154264,1273.835360,4980.482316,3706.646957,3.909832,1


In [177]:
# Measure remaining mandatory iron-replacement burden after 2028

remaining_iron_ids = [
    pipe_id
    for pipe_id, material in current_material_2028.items()
    if material in iron_materials
]

remaining_iron_length = sum(
    pipe_length_map[pipe_id]
    for pipe_id in remaining_iron_ids
)

initial_iron_count = 26237
initial_iron_length = 440677.323

iron_replaced_count = (
    initial_iron_count - len(remaining_iron_ids)
)

iron_replaced_length = (
    initial_iron_length - remaining_iron_length
)

years_completed = 2
years_remaining = 25 - years_completed


print("Iron pipes remaining:", len(remaining_iron_ids))

print(
    "Remaining iron length:",
    remaining_iron_length
)

print(
    "Iron pipes replaced so far:",
    iron_replaced_count
)

print(
    "Iron length replaced so far:",
    iron_replaced_length
)

print(
    "Percent of iron pipes replaced:",
    iron_replaced_count / initial_iron_count * 100
)

print(
    "Percent of iron length replaced:",
    iron_replaced_length / initial_iron_length * 100
)

print(
    "Average pipes/year needed from now:",
    len(remaining_iron_ids) / years_remaining
)

print(
    "Average iron meters/year needed from now:",
    remaining_iron_length / years_remaining
)

print(
    "Budget carried into 2029:",
    unused_budget_2028
)

Iron pipes remaining: 23014
Remaining iron length: 407792.38822889136
Iron pipes replaced so far: 3223
Iron length replaced so far: 32884.934771108616
Percent of iron pipes replaced: 12.284178831421274
Percent of iron length replaced: 7.4623614728431615
Average pipes/year needed from now: 1000.6086956521739
Average iron meters/year needed from now: 17730.103836038754
Budget carried into 2029: 9362899.089741727


In [178]:
# Carry the post-2028 physical network state into 2029

network_2029 = network_2028.copy()

network_2029["Material_2029"] = (
    network_2029["Pipe ID"]
    .map(current_material_2028)
)

network_2029["Is_iron_2029"] = (
    network_2029["Material_2029"]
    .isin(iron_materials)
)

network_2029["Is_non_PU_2029"] = (
    network_2029["Material_2029"].notna()
    & (network_2029["Material_2029"] != "polyurethane")
)


print("Physical pipes:", len(network_2029))

print(
    "Iron remaining:",
    network_2029["Is_iron_2029"].sum()
)

print(
    "Non-PU remaining:",
    network_2029["Is_non_PU_2029"].sum()
)

print(
    "Polyurethane:",
    (network_2029["Material_2029"] == "polyurethane").sum()
)

display(
    network_2029["Material_2029"]
    .value_counts(dropna=False)
)

Physical pipes: 43039
Iron remaining: 23014
Non-PU remaining: 24012
Polyurethane: 19019


Material_2029
polyurethane    19019
wrought iron    13854
cast iron        7174
gray iron        1986
copper            693
brass             305
NaN                 8
Name: count, dtype: int64

In [179]:
# Build the 2029 risk set from pipes that are still known non-polyurethane

risk_2029 = network_2029[
    network_2029["Is_non_PU_2029"]
].copy()

risk_2029["Year"] = 2029

risk_2029["Age"] = (
    pd.Timestamp("2029-01-01")
    - pd.to_datetime(risk_2029["Lay date"])
).dt.days / 365.25

risk_2029["Material"] = risk_2029["Material_2029"]

# Apply the same missing-age rule used for future prediction
risk_2029["Age"] = (
    risk_2029["Age"]
    .fillna(age_median)
)


print("2029 at-risk pipes:", len(risk_2029))

print(
    "Iron pipes:",
    risk_2029["Material"].isin(iron_materials).sum()
)

print(
    "Copper/brass pipes:",
    risk_2029["Material"].isin(["copper", "brass"]).sum()
)

print(
    "\nMissing prediction features:"
)

print(
    risk_2029[
        numeric_features + categorical_features
    ].isna().sum()
)

2029 at-risk pipes: 24012
Iron pipes: 23014
Copper/brass pipes: 998

Missing prediction features:
Age            0
Pipe length    0
Midpoint x     0
Midpoint y     0
Year           0
Material       0
Surface        0
dtype: int64


In [180]:
# Predict 2029 leak probabilities using the fixed production ensemble

X_2029 = risk_2029[
    numeric_features + categorical_features
]

risk_2029["xgb_probability"] = (
    final_xgb.predict_proba(X_2029)[:, 1]
)

risk_2029["catboost_probability"] = (
    final_cat.predict_proba(X_2029)[:, 1]
)

risk_2029["lightgbm_probability"] = (
    final_lgbm.predict_proba(X_2029)[:, 1]
)

risk_2029["raw_ensemble_probability"] = (
    risk_2029[
        [
            "xgb_probability",
            "catboost_probability",
            "lightgbm_probability"
        ]
    ].mean(axis=1)
)


leak_2029 = risk_2029[
    [
        "Pipe ID",
        "Material",
        "Surface",
        "Age",
        "Pipe length",
        "xgb_probability",
        "catboost_probability",
        "lightgbm_probability",
        "raw_ensemble_probability"
    ]
].copy()


print(
    "2029 predicted pipes:",
    len(leak_2029)
)

print(
    "Expected number of leaks:",
    leak_2029["raw_ensemble_probability"].sum()
)

print(
    "Mean leak probability:",
    leak_2029["raw_ensemble_probability"].mean()
)

print(
    "Maximum leak probability:",
    leak_2029["raw_ensemble_probability"].max()
)

print(
    "Missing probabilities:",
    leak_2029["raw_ensemble_probability"].isna().sum()
)

2029 predicted pipes: 24012
Expected number of leaks: 1301.4846057006375
Mean leak probability: 0.054201424525263935
Maximum leak probability: 0.6309073066439773
Missing probabilities: 0


In [181]:
# Separate the effect of population change from the effect of Year

risk_2029_year_check = risk_2029.copy()

# Same surviving pipes, but pretend the model input year is 2028
risk_2029_as_2028 = risk_2029_year_check.copy()
risk_2029_as_2028["Year"] = 2028

X_same_pipes_2028 = risk_2029_as_2028[
    numeric_features + categorical_features
]

xgb_same_2028 = final_xgb.predict_proba(
    X_same_pipes_2028
)[:, 1]

cat_same_2028 = final_cat.predict_proba(
    X_same_pipes_2028
)[:, 1]

lgbm_same_2028 = final_lgbm.predict_proba(
    X_same_pipes_2028
)[:, 1]

ensemble_same_2028 = (
    xgb_same_2028
    + cat_same_2028
    + lgbm_same_2028
) / 3


print(
    "Same 2029 surviving pipes with Year=2028:"
)

print(
    "Expected leaks:",
    ensemble_same_2028.sum()
)

print(
    "Mean probability:",
    ensemble_same_2028.mean()
)

print(
    "\nSame pipes with Year=2029:"
)

print(
    "Expected leaks:",
    leak_2029[
        "raw_ensemble_probability"
    ].sum()
)

print(
    "Mean probability:",
    leak_2029[
        "raw_ensemble_probability"
    ].mean()
)

Same 2029 surviving pipes with Year=2028:
Expected leaks: 1301.4846057006375
Mean probability: 0.054201424525263935

Same pipes with Year=2029:
Expected leaks: 1301.4846057006375
Mean probability: 0.054201424525263935


In [182]:
# Predict 2029 conditional leak costs with calendar year frozen at 2026

cost_2029_frozen = risk_2029.copy()

cost_2029_frozen["Age at leak"] = (
    cost_2029_frozen["Age"]
    .fillna(age_median)
)

# Freeze calendar year at the last observed training year
cost_2029_frozen["Leak year"] = 2026

cost_2029_frozen["Material"] = (
    cost_2029_frozen["Material"]
    .fillna("Unknown")
)

raw_cost_2029_frozen = final_cost_model.predict(
    cost_2029_frozen[cost_features]
)

cost_2029_frozen["predicted_leak_cost"] = (
    np.maximum(raw_cost_2029_frozen, 0)
    * final_cost_calibration
)


print(
    "Mean predicted conditional cost:",
    cost_2029_frozen["predicted_leak_cost"].mean()
)

print(
    "Median predicted conditional cost:",
    cost_2029_frozen["predicted_leak_cost"].median()
)

print(
    "Zero-cost predictions:",
    (
        cost_2029_frozen["predicted_leak_cost"] == 0
    ).sum()
)

print(
    "Zero-cost percentage:",
    (
        cost_2029_frozen["predicted_leak_cost"] == 0
    ).mean() * 100
)

print(
    "Missing predictions:",
    cost_2029_frozen[
        "predicted_leak_cost"
    ].isna().sum()
)

Mean predicted conditional cost: 15634.262004312319
Median predicted conditional cost: 3732.200945883473
Zero-cost predictions: 6939
Zero-cost percentage: 28.898050974512746
Missing predictions: 0


In [183]:
# Combine 2029 leak probabilities with Frozen-Year conditional costs

prediction_2029 = (
    leak_2029[
        [
            "Pipe ID",
            "raw_ensemble_probability"
        ]
    ]
    .merge(
        cost_2029_frozen[
            [
                "Pipe ID",
                "predicted_leak_cost"
            ]
        ],
        on="Pipe ID",
        how="inner"
    )
)

prediction_2029["expected_leak_loss"] = (
    prediction_2029["raw_ensemble_probability"]
    * prediction_2029["predicted_leak_cost"]
)


print(
    "2029 predicted pipes:",
    len(prediction_2029)
)

print(
    "Expected number of leaks:",
    prediction_2029[
        "raw_ensemble_probability"
    ].sum()
)

print(
    "Total expected leak loss:",
    prediction_2029[
        "expected_leak_loss"
    ].sum()
)

print(
    "Mean expected leak loss per pipe:",
    prediction_2029[
        "expected_leak_loss"
    ].mean()
)

print(
    "Pipes with zero expected leak loss:",
    (
        prediction_2029[
            "expected_leak_loss"
        ] == 0
    ).sum()
)

display(
    prediction_2029
    .sort_values(
        "expected_leak_loss",
        ascending=False
    )
    .head(10)
)

2029 predicted pipes: 24012
Expected number of leaks: 1301.4846057006375
Total expected leak loss: 14223153.86785346
Mean expected leak loss per pipe: 592.3352435387915
Pipes with zero expected leak loss: 6939


,Pipe ID,raw_ensemble_probability,predicted_leak_cost,expected_leak_loss
20812,P38620,0.529855,45878.175777,24308.765439
2332,P04692,0.508685,42921.878489,21833.736493
2402,P04799,0.473501,45329.069487,21463.367494
4846,P09131,0.427601,47200.493346,20182.961022
4843,P09128,0.427601,47083.767044,20133.048781
4848,P09133,0.410821,48125.213687,19770.852616
4845,P09130,0.406970,47996.467710,19533.118436
3152,P05958,0.086686,221312.675666,19184.675238
4839,P09122,0.396457,48106.874155,19072.329898
1447,P03331,0.086976,218384.665460,18994.186691


In [184]:
# Initialize the 2029 budget and dynamically evaluate candidate work units

year_3_budget = ANNUAL_BUDGET + unused_budget_2028

current_material_2029 = (
    network_2029
    .set_index("Pipe ID")["Material_2029"]
    .to_dict()
)

expected_loss_2029 = (
    prediction_2029
    .set_index("Pipe ID")["expected_leak_loss"]
    .to_dict()
)

initial_2029_evaluations = []

for _, unit in work_units.iterrows():

    result = evaluate_work_unit_dynamic(
        unit,
        current_material_2029,
        expected_loss_2029,
        pipe_length_map
    )

    if result["active_pipe_count"] == 0:
        continue

    initial_2029_evaluations.append({
        "Center Pipe ID": unit["Center Pipe ID"],
        "Construction Cost": result["construction_cost"],
        "Expected Benefit": result["expected_benefit"],
        "Net Benefit": result["net_benefit"],
        "Benefit Cost Ratio": result["benefit_cost_ratio"],
        "Active Pipe Count": result["active_pipe_count"]
    })


initial_2029_evaluations = pd.DataFrame(
    initial_2029_evaluations
)

print("2029 available budget:", year_3_budget)

print(
    "Candidates with positive net benefit:",
    (initial_2029_evaluations["Net Benefit"] > 0).sum()
)

print(
    "Maximum 2029 BCR:",
    initial_2029_evaluations["Benefit Cost Ratio"].max()
)

display(
    initial_2029_evaluations
    .sort_values("Benefit Cost Ratio", ascending=False)
    .head(20)
)

2029 available budget: 19362899.08974173
Candidates with positive net benefit: 70
Maximum 2029 BCR: 5.214369120359678


,Center Pipe ID,Construction Cost,Expected Benefit,Net Benefit,Benefit Cost Ratio,Active Pipe Count
19872,P38620,4661.880446,24308.765439,19646.884994,5.214369,1
10389,P20983,836.420906,1812.443068,976.022162,2.166903,1
9849,P19571,443.303303,880.884938,437.581634,1.987093,1
8112,P16067,1220.869727,1939.696697,718.826970,1.588783,1
4680,P10070,2546.459872,3605.451490,1058.991618,1.415868,2
16136,P32560,4634.644466,6427.347791,1792.703325,1.386805,1
7878,P15495,1231.608451,1576.668764,345.060313,1.280170,1
22532,P43208,2738.571805,3478.413203,739.841397,1.270156,1
4682,P10072,1452.394301,1810.124556,357.730255,1.246304,1
7867,P15473,1287.595037,1576.836357,289.241320,1.224637,1


In [185]:
# Estimate the standalone replacement burden for all remaining iron pipes

remaining_iron_2029 = [
    pipe_id
    for pipe_id, material in current_material_2029.items()
    if material in iron_materials
]

mandatory_burden_rows = []

for pipe_id in remaining_iron_2029:

    unit_match = work_units[
        work_units["Center Pipe ID"] == pipe_id
    ]

    if len(unit_match) == 0:
        continue

    unit = unit_match.iloc[0]

    result = evaluate_work_unit_dynamic(
        unit,
        current_material_2029,
        expected_loss_2029,
        pipe_length_map
    )

    mandatory_burden_rows.append({
        "Pipe ID": pipe_id,
        "Construction Cost": result["construction_cost"],
        "Pipe Length": pipe_length_map[pipe_id],
        "Expected Benefit 2029": result["expected_benefit"],
        "BCR 2029": result["benefit_cost_ratio"]
    })


mandatory_burden_2029 = pd.DataFrame(
    mandatory_burden_rows
)


print(
    "Remaining mandatory iron pipes:",
    len(mandatory_burden_2029)
)

print(
    "Standalone construction-cost sum:",
    mandatory_burden_2029[
        "Construction Cost"
    ].sum()
)

print(
    "Mean standalone cost per iron pipe:",
    mandatory_burden_2029[
        "Construction Cost"
    ].mean()
)

print(
    "Median standalone cost per iron pipe:",
    mandatory_burden_2029[
        "Construction Cost"
    ].median()
)

print(
    "Remaining 23-year nominal budget:",
    year_3_budget + ANNUAL_BUDGET * 22
)

Remaining mandatory iron pipes: 23014
Standalone construction-cost sum: 185133998.50195044
Mean standalone cost per iron pipe: 8044.407686710282
Median standalone cost per iron pipe: 3219.0880029272475
Remaining 23-year nominal budget: 239362899.08974174


In [186]:
# Compare actual iron replacement progress with the 25-year requirement

total_iron_count = 26237
total_iron_length = 440677.323

years_completed = 2

target_count_replaced = (
    total_iron_count
    * years_completed / 25
)

target_length_replaced = (
    total_iron_length
    * years_completed / 25
)

actual_count_replaced = (
    total_iron_count
    - len(remaining_iron_2029)
)

actual_length_replaced = (
    total_iron_length
    - sum(
        pipe_length_map[pipe_id]
        for pipe_id in remaining_iron_2029
    )
)

count_gap = (
    actual_count_replaced
    - target_count_replaced
)

length_gap = (
    actual_length_replaced
    - target_length_replaced
)


print("Years completed:", years_completed)

print("\n--- Pipe count progress ---")
print("Target replaced:", target_count_replaced)
print("Actual replaced:", actual_count_replaced)
print("Ahead (+) / behind (-):", count_gap)

print("\n--- Pipe length progress ---")
print("Target meters replaced:", target_length_replaced)
print("Actual meters replaced:", actual_length_replaced)
print("Ahead (+) / behind (-) meters:", length_gap)

print(
    "\nRequired average meters/year from 2029 onward:",
    sum(
        pipe_length_map[pipe_id]
        for pipe_id in remaining_iron_2029
    ) / 23
)

Years completed: 2

--- Pipe count progress ---
Target replaced: 2098.96
Actual replaced: 3223
Ahead (+) / behind (-): 1124.04

--- Pipe length progress ---
Target meters replaced: 35254.18584
Actual meters replaced: 32884.934771108616
Ahead (+) / behind (-) meters: -2369.2510688913826

Required average meters/year from 2029 onward: 17730.103836038754


In [187]:
# Calculate how much iron length should be replaced during 2029

years_completed_after_2029 = 3

target_cumulative_length_2029 = (
    total_iron_length
    * years_completed_after_2029
    / 25
)

iron_length_replaced_before_2029 = (
    total_iron_length
    - sum(
        pipe_length_map[pipe_id]
        for pipe_id in remaining_iron_2029
    )
)

required_iron_length_2029 = max(
    0,
    target_cumulative_length_2029
    - iron_length_replaced_before_2029
)


print(
    "Target cumulative iron length replaced after 2029:",
    target_cumulative_length_2029
)

print(
    "Iron length already replaced before 2029:",
    iron_length_replaced_before_2029
)

print(
    "Minimum iron length to replace during 2029:",
    required_iron_length_2029
)

print(
    "2029 available budget:",
    year_3_budget
)

Target cumulative iron length replaced after 2029: 52881.27876
Iron length already replaced before 2029: 32884.934771108616
Minimum iron length to replace during 2029: 19996.343988891385
2029 available budget: 19362899.08974173


In [188]:
# Extend the dynamic evaluator to track mandatory iron replacement separately

def evaluate_work_unit_dynamic(
    unit,
    current_material,
    expected_loss,
    pipe_length_map
):
    covered_ids = unit["Covered Pipe IDs"]

    # All currently active non-PU pipes covered by this work unit
    active_non_pu = [
        pipe_id
        for pipe_id in covered_ids
        if (
            pd.notna(current_material.get(pipe_id))
            and current_material.get(pipe_id) != "polyurethane"
        )
    ]

    # Mandatory iron pipes among the active covered pipes
    active_iron = [
        pipe_id
        for pipe_id in active_non_pu
        if current_material.get(pipe_id) in iron_materials
    ]

    non_pu_meters = sum(
        pipe_length_map[pipe_id]
        for pipe_id in active_non_pu
    )

    iron_meters = sum(
        pipe_length_map[pipe_id]
        for pipe_id in active_iron
    )

    surface_cost = (
        SURFACE_RATES[unit["Surface"]]
        * unit["Area"]**0.85
    )

    pipe_cost = (
        NON_PU_PIPE_COST_PER_METER
        * non_pu_meters
    )

    construction_cost = (
        GROUNDBREAKING_COST
        + surface_cost
        + pipe_cost
    )

    expected_benefit = sum(
        expected_loss.get(pipe_id, 0)
        for pipe_id in active_non_pu
    )

    net_benefit = (
        expected_benefit
        - construction_cost
    )

    benefit_cost_ratio = (
        expected_benefit / construction_cost
        if construction_cost > 0
        else 0
    )

    return {
        "active_pipe_ids": active_non_pu,
        "active_pipe_count": len(active_non_pu),

        "iron_pipe_ids": active_iron,
        "iron_pipe_count": len(active_iron),
        "iron_meters": iron_meters,

        "non_pu_meters": non_pu_meters,

        "construction_cost": construction_cost,
        "expected_benefit": expected_benefit,
        "net_benefit": net_benefit,
        "benefit_cost_ratio": benefit_cost_ratio
    }

In [189]:
# Verify that iron meters never exceed total non-PU meters

iron_tracking_check = []

for _, unit in work_units.head(100).iterrows():

    result = evaluate_work_unit_dynamic(
        unit,
        current_material_2029,
        expected_loss_2029,
        pipe_length_map
    )

    iron_tracking_check.append({
        "Center Pipe ID": unit["Center Pipe ID"],
        "Active Pipes": result["active_pipe_count"],
        "Iron Pipes": result["iron_pipe_count"],
        "Non-PU Meters": result["non_pu_meters"],
        "Iron Meters": result["iron_meters"]
    })


iron_tracking_check = pd.DataFrame(
    iron_tracking_check
)

print(
    "Any iron meters > non-PU meters:",
    (
        iron_tracking_check["Iron Meters"]
        > iron_tracking_check["Non-PU Meters"]
    ).any()
)

print(
    "Any iron pipe count > active pipe count:",
    (
        iron_tracking_check["Iron Pipes"]
        > iron_tracking_check["Active Pipes"]
    ).any()
)

display(
    iron_tracking_check.head(10)
)

Any iron meters > non-PU meters: False
Any iron pipe count > active pipe count: False


,Center Pipe ID,Active Pipes,Iron Pipes,Non-PU Meters,Iron Meters
0,P01004,1,1,29.750686,29.750686
1,P01006,2,2,54.076084,54.076084
2,P01007,1,1,12.150000,12.150000
3,P01008,1,1,22.918440,22.918440
4,P01009,1,1,45.490000,45.490000
5,P01010,1,1,28.378337,28.378337
6,P01012,1,1,5.852350,5.852350
7,P01013,1,1,5.823075,5.823075
8,P01014,1,1,5.744258,5.744258
9,P01016,1,1,12.317824,12.317824


In [190]:
# Execute profitable work first, then satisfy the 2029 mandatory iron-length target

year_3_spent = 0
iron_meters_replaced_2029 = 0

selected_work_units_2029 = []
replaced_pipe_ids_2029 = set()
replaced_iron_ids_2029 = set()

candidate_order_2029 = (
    initial_2029_evaluations
    .sort_values(
        "Benefit Cost Ratio",
        ascending=False
    )
)

# Phase 1: economically profitable work units
for _, candidate in candidate_order_2029.iterrows():

    if candidate["Benefit Cost Ratio"] <= 1:
        continue

    center_id = candidate["Center Pipe ID"]

    unit = work_units[
        work_units["Center Pipe ID"] == center_id
    ].iloc[0]

    result = evaluate_work_unit_dynamic(
        unit,
        current_material_2029,
        expected_loss_2029,
        pipe_length_map
    )

    if result["active_pipe_count"] == 0:
        continue

    if result["benefit_cost_ratio"] <= 1:
        continue

    if (
        year_3_spent + result["construction_cost"]
        > year_3_budget
    ):
        continue

    year_3_spent += result["construction_cost"]
    iron_meters_replaced_2029 += result["iron_meters"]

    for pipe_id in result["active_pipe_ids"]:
        current_material_2029[pipe_id] = "polyurethane"
        replaced_pipe_ids_2029.add(pipe_id)

    for pipe_id in result["iron_pipe_ids"]:
        replaced_iron_ids_2029.add(pipe_id)

    selected_work_units_2029.append({
        "Phase": "Economic",
        "Center Pipe ID": center_id,
        "Construction Cost": result["construction_cost"],
        "Expected Benefit": result["expected_benefit"],
        "Benefit Cost Ratio": result["benefit_cost_ratio"],
        "Iron Meters": result["iron_meters"],
        "Replaced Pipe Count": result["active_pipe_count"]
    })


# Phase 2: continue until the mandatory iron-length target is reached
for _, candidate in candidate_order_2029.iterrows():

    if iron_meters_replaced_2029 >= required_iron_length_2029:
        break

    center_id = candidate["Center Pipe ID"]

    unit = work_units[
        work_units["Center Pipe ID"] == center_id
    ].iloc[0]

    result = evaluate_work_unit_dynamic(
        unit,
        current_material_2029,
        expected_loss_2029,
        pipe_length_map
    )

    if result["active_pipe_count"] == 0:
        continue

    # Mandatory phase must actually replace iron
    if result["iron_pipe_count"] == 0:
        continue

    if (
        year_3_spent + result["construction_cost"]
        > year_3_budget
    ):
        continue

    year_3_spent += result["construction_cost"]
    iron_meters_replaced_2029 += result["iron_meters"]

    for pipe_id in result["active_pipe_ids"]:
        current_material_2029[pipe_id] = "polyurethane"
        replaced_pipe_ids_2029.add(pipe_id)

    for pipe_id in result["iron_pipe_ids"]:
        replaced_iron_ids_2029.add(pipe_id)

    selected_work_units_2029.append({
        "Phase": "Mandatory",
        "Center Pipe ID": center_id,
        "Construction Cost": result["construction_cost"],
        "Expected Benefit": result["expected_benefit"],
        "Benefit Cost Ratio": result["benefit_cost_ratio"],
        "Iron Meters": result["iron_meters"],
        "Replaced Pipe Count": result["active_pipe_count"]
    })


year_3_plan = pd.DataFrame(selected_work_units_2029)

unused_budget_2029 = (
    year_3_budget - year_3_spent
)

remaining_iron_after_2029 = sum(
    material in iron_materials
    for material in current_material_2029.values()
)


print("Selected work units:", len(year_3_plan))
print("2029 CAPEX:", year_3_spent)
print("Unused 2029 budget:", unused_budget_2029)

print(
    "Total pipes replaced:",
    len(replaced_pipe_ids_2029)
)

print(
    "Iron pipes replaced:",
    len(replaced_iron_ids_2029)
)

print(
    "Iron meters replaced:",
    iron_meters_replaced_2029
)

print(
    "Required iron meters:",
    required_iron_length_2029
)

print(
    "Mandatory target reached:",
    iron_meters_replaced_2029 >= required_iron_length_2029
)

print(
    "Iron pipes remaining:",
    remaining_iron_after_2029
)

print("\nWork units by phase:")
print(year_3_plan["Phase"].value_counts())

print(
    "\nExpected leak loss avoided:",
    year_3_plan["Expected Benefit"].sum()
)

Selected work units: 1417
2029 CAPEX: 6108640.6092955265
Unused 2029 budget: 13254258.480446203
Total pipes replaced: 1562
Iron pipes replaced: 1560
Iron meters replaced: 20002.645355841883
Required iron meters: 19996.343988891385
Mandatory target reached: True
Iron pipes remaining: 21454

Work units by phase:
Phase
Mandatory    1349
Economic       68
Name: count, dtype: int64

Expected leak loss avoided: 4491031.766834844


# iteration

In [191]:
# Create one reusable function for annual leak and conditional-cost prediction

def predict_future_year(
    network_state,
    material_column,
    year
):
    # Keep currently known non-PU pipes
    risk = network_state[
        network_state[material_column].notna()
        & (network_state[material_column] != "polyurethane")
    ].copy()

    # Build current-year model features
    risk["Year"] = year

    risk["Age"] = (
        pd.Timestamp(f"{year}-01-01")
        - pd.to_datetime(risk["Lay date"])
    ).dt.days / 365.25

    risk["Age"] = risk["Age"].fillna(age_median)

    risk["Material"] = risk[material_column]

    # Leak prediction
    X = risk[
        numeric_features + categorical_features
    ]

    risk["xgb_probability"] = (
        final_xgb.predict_proba(X)[:, 1]
    )

    risk["catboost_probability"] = (
        final_cat.predict_proba(X)[:, 1]
    )

    risk["lightgbm_probability"] = (
        final_lgbm.predict_proba(X)[:, 1]
    )

    risk["predicted_leak_probability"] = (
        risk[
            [
                "xgb_probability",
                "catboost_probability",
                "lightgbm_probability"
            ]
        ].mean(axis=1)
    )

    # Conditional cost prediction
    risk["Age at leak"] = risk["Age"]

    # Freeze calendar year at the last observed year
    risk["Leak year"] = 2026

    raw_cost = final_cost_model.predict(
        risk[cost_features]
    )

    risk["predicted_leak_cost"] = (
        np.maximum(raw_cost, 0)
        * final_cost_calibration
    )

    # Expected leak loss
    risk["expected_leak_loss"] = (
        risk["predicted_leak_probability"]
        * risk["predicted_leak_cost"]
    )

    return risk

In [192]:
# Check that the reusable prediction function reproduces the manual 2029 results

test_prediction_2029 = predict_future_year(
    network_2029,
    "Material_2029",
    2029
)

print(
    "Pipes:",
    len(test_prediction_2029)
)

print(
    "Expected leaks:",
    test_prediction_2029[
        "predicted_leak_probability"
    ].sum()
)

print(
    "Total expected leak loss:",
    test_prediction_2029[
        "expected_leak_loss"
    ].sum()
)

print(
    "Mean conditional cost:",
    test_prediction_2029[
        "predicted_leak_cost"
    ].mean()
)

Pipes: 24012
Expected leaks: 1301.4846057006375
Total expected leak loss: 14223153.86785346
Mean conditional cost: 15634.262004312319


In [204]:
# Create a reusable two-phase construction planner for each future year

def execute_yearly_construction(
    current_material,
    prediction,
    available_budget,
    year_number
):
    current_material = current_material.copy()

    expected_loss = (
        prediction
        .set_index("Pipe ID")["expected_leak_loss"]
        .to_dict()
    )

    # Cumulative mandatory iron-length target
    target_cumulative_length = (
        total_iron_length
        * year_number
        / 25
    )

    current_remaining_iron_length = sum(
        pipe_length_map[pipe_id]
        for pipe_id, material in current_material.items()
        if material in iron_materials
    )

    iron_length_already_replaced = (
        total_iron_length
        - current_remaining_iron_length
    )

    required_iron_length_this_year = max(
        0,
        target_cumulative_length
        - iron_length_already_replaced
    )

    # Evaluate all candidates at the start of the year
    candidate_rows = []

    for _, unit in work_units.iterrows():

        result = evaluate_work_unit_dynamic(
            unit,
            current_material,
            expected_loss,
            pipe_length_map
        )

        if result["active_pipe_count"] == 0:
            continue

        candidate_rows.append({
            "Center Pipe ID": unit["Center Pipe ID"],
            "Initial BCR": result["benefit_cost_ratio"]
        })

    candidates = (
        pd.DataFrame(candidate_rows)
        .sort_values("Initial BCR", ascending=False)
    )

    spent = 0
    iron_meters_replaced = 0

    replaced_pipe_ids = set()
    replaced_iron_ids = set()
    selected_units = []

    # Phase 1: economically profitable work
    for _, candidate in candidates.iterrows():

        if candidate["Initial BCR"] <= 1:
            continue

        center_id = candidate["Center Pipe ID"]

        unit = work_units[
            work_units["Center Pipe ID"] == center_id
        ].iloc[0]

        result = evaluate_work_unit_dynamic(
            unit,
            current_material,
            expected_loss,
            pipe_length_map
        )

        if result["active_pipe_count"] == 0:
            continue

        if result["benefit_cost_ratio"] <= 1:
            continue

        if (
            spent + result["construction_cost"]
            > available_budget
        ):
            continue

        spent += result["construction_cost"]
        iron_meters_replaced += result["iron_meters"]

        for pipe_id in result["active_pipe_ids"]:
            current_material[pipe_id] = "polyurethane"
            replaced_pipe_ids.add(pipe_id)

        for pipe_id in result["iron_pipe_ids"]:
            replaced_iron_ids.add(pipe_id)

        selected_units.append({
            "Phase": "Economic",
            "Center Pipe ID": center_id,
            "Construction Cost": result["construction_cost"],
            "Expected Benefit": result["expected_benefit"],
            "BCR": result["benefit_cost_ratio"],
            "Iron Meters": result["iron_meters"]
        })

    # Phase 2: satisfy mandatory iron-length progress
    for _, candidate in candidates.iterrows():

        if (
            iron_meters_replaced
            >= required_iron_length_this_year
        ):
            break

        center_id = candidate["Center Pipe ID"]

        unit = work_units[
            work_units["Center Pipe ID"] == center_id
        ].iloc[0]

        result = evaluate_work_unit_dynamic(
            unit,
            current_material,
            expected_loss,
            pipe_length_map
        )

        if result["active_pipe_count"] == 0:
            continue

        if result["iron_pipe_count"] == 0:
            continue

        if (
            spent + result["construction_cost"]
            > available_budget
        ):
            continue

        spent += result["construction_cost"]
        iron_meters_replaced += result["iron_meters"]

        for pipe_id in result["active_pipe_ids"]:
            current_material[pipe_id] = "polyurethane"
            replaced_pipe_ids.add(pipe_id)

        for pipe_id in result["iron_pipe_ids"]:
            replaced_iron_ids.add(pipe_id)

        selected_units.append({
            "Phase": "Mandatory",
            "Center Pipe ID": center_id,
            "Construction Cost": result["construction_cost"],
            "Expected Benefit": result["expected_benefit"],
            "BCR": result["benefit_cost_ratio"],
            "Iron Meters": result["iron_meters"]
        })

    plan = pd.DataFrame(selected_units)

    unused_budget = available_budget - spent

    remaining_iron = sum(
        material in iron_materials
        for material in current_material.values()
    )

    remaining_iron_length = sum(
        pipe_length_map[pipe_id]
        for pipe_id, material in current_material.items()
        if material in iron_materials
    )

    expected_loss_avoided = (
        plan["Expected Benefit"].sum()
        if len(plan) > 0
        else 0
    )

    return {
        "material": current_material,
        "plan": plan,
        "spent": spent,
        "unused_budget": unused_budget,
        "pipes_replaced": len(replaced_pipe_ids),
        "iron_pipes_replaced": len(replaced_iron_ids),
        "iron_meters_replaced": iron_meters_replaced,
        "required_iron_meters": required_iron_length_this_year,
        "mandatory_target_reached": (
            iron_meters_replaced
            >= required_iron_length_this_year - 1e-6
            or remaining_iron == 0
        ),
        "remaining_iron": remaining_iron,
        "remaining_iron_length": remaining_iron_length,
        "expected_loss_avoided": expected_loss_avoided
    }

In [194]:
# Recreate the pre-construction 2029 material state and validate the yearly planner

test_material_2029 = (
    network_2029
    .set_index("Pipe ID")["Material_2029"]
    .to_dict()
)

test_result_2029 = execute_yearly_construction(
    current_material=test_material_2029,
    prediction=test_prediction_2029,
    available_budget=year_3_budget,
    year_number=3
)

print("Selected work units:", len(test_result_2029["plan"]))
print("CAPEX:", test_result_2029["spent"])
print("Unused budget:", test_result_2029["unused_budget"])
print("Total pipes replaced:", test_result_2029["pipes_replaced"])
print("Iron pipes replaced:", test_result_2029["iron_pipes_replaced"])
print("Iron meters replaced:", test_result_2029["iron_meters_replaced"])
print("Required iron meters:", test_result_2029["required_iron_meters"])
print("Target reached:", test_result_2029["mandatory_target_reached"])
print("Iron remaining:", test_result_2029["remaining_iron"])
print("Expected loss avoided:", test_result_2029["expected_loss_avoided"])

print("\nWork units by phase:")
print(test_result_2029["plan"]["Phase"].value_counts())

Selected work units: 1417
CAPEX: 6108640.6092955265
Unused budget: 13254258.480446203
Total pipes replaced: 1562
Iron pipes replaced: 1560
Iron meters replaced: 20002.645355841883
Required iron meters: 19996.343988891385
Target reached: True
Iron remaining: 21454
Expected loss avoided: 4491031.766834844

Work units by phase:
Phase
Mandatory    1349
Economic       68
Name: count, dtype: int64


In [195]:
# Build the 2030 network state from the completed 2029 construction plan

network_2030 = network_2029.copy()

network_2030["Material_2030"] = (
    network_2030["Pipe ID"]
    .map(test_result_2029["material"])
)

print("Physical pipes:", len(network_2030))

print(
    "Iron remaining:",
    network_2030["Material_2030"]
    .isin(iron_materials)
    .sum()
)

print(
    "Non-PU remaining:",
    (
        network_2030["Material_2030"].notna()
        & (network_2030["Material_2030"] != "polyurethane")
    ).sum()
)

print(
    "PU pipes:",
    (
        network_2030["Material_2030"]
        == "polyurethane"
    ).sum()
)

print("\nMaterial counts:")
print(
    network_2030["Material_2030"]
    .value_counts(dropna=False)
)

Physical pipes: 43039
Iron remaining: 21454
Non-PU remaining: 22450
PU pipes: 20581

Material counts:
Material_2030
polyurethane    20581
wrought iron    12462
cast iron        7032
gray iron        1960
copper            693
brass             303
NaN                 8
Name: count, dtype: int64


In [196]:
# Predict 2030 leak probabilities, conditional costs, and expected leak losses

prediction_2030 = predict_future_year(
    network_2030,
    "Material_2030",
    2030
)

print("2030 predicted pipes:", len(prediction_2030))

print(
    "Expected number of leaks:",
    prediction_2030[
        "predicted_leak_probability"
    ].sum()
)

print(
    "Mean leak probability:",
    prediction_2030[
        "predicted_leak_probability"
    ].mean()
)

print(
    "Mean conditional leak cost:",
    prediction_2030[
        "predicted_leak_cost"
    ].mean()
)

print(
    "Total expected leak loss:",
    prediction_2030[
        "expected_leak_loss"
    ].sum()
)

print(
    "Mean expected leak loss per pipe:",
    prediction_2030[
        "expected_leak_loss"
    ].mean()
)

2030 predicted pipes: 22450
Expected number of leaks: 543.078602569698
Mean leak probability: 0.02419058363339412
Mean conditional leak cost: 12613.103393986245
Total expected leak loss: 7034931.965790049
Mean expected leak loss per pipe: 313.3599984761715


In [197]:
# Add the new annual budget to the unused 2029 carryover

budget_2030 = (
    test_result_2029["unused_budget"]
    + ANNUAL_BUDGET
)

print("2030 available budget:", budget_2030)

2030 available budget: 23254258.480446205


In [198]:
# Execute the validated two-phase construction strategy for 2030

result_2030 = execute_yearly_construction(
    current_material=(
        network_2030
        .set_index("Pipe ID")["Material_2030"]
        .to_dict()
    ),
    prediction=prediction_2030,
    available_budget=budget_2030,
    year_number=4
)

print("Selected work units:", len(result_2030["plan"]))
print("2030 CAPEX:", result_2030["spent"])
print("Unused 2030 budget:", result_2030["unused_budget"])

print(
    "Total pipes replaced:",
    result_2030["pipes_replaced"]
)

print(
    "Iron pipes replaced:",
    result_2030["iron_pipes_replaced"]
)

print(
    "Iron meters replaced:",
    result_2030["iron_meters_replaced"]
)

print(
    "Required iron meters:",
    result_2030["required_iron_meters"]
)

print(
    "Mandatory target reached:",
    result_2030["mandatory_target_reached"]
)

print(
    "Iron pipes remaining:",
    result_2030["remaining_iron"]
)

print(
    "Iron length remaining:",
    result_2030["remaining_iron_length"]
)

print(
    "Expected leak loss avoided:",
    result_2030["expected_loss_avoided"]
)

print("\nWork units by phase:")
print(
    result_2030["plan"]["Phase"]
    .value_counts()
)

Selected work units: 1071
2030 CAPEX: 6802579.084559818
Unused 2030 budget: 16451679.395886388
Total pipes replaced: 1355
Iron pipes replaced: 1347
Iron meters replaced: 17627.398770728345
Required iron meters: 17620.791553049537
Mandatory target reached: True
Iron pipes remaining: 20107
Iron length remaining: 370162.34410232113
Expected leak loss avoided: 2203165.202462628

Work units by phase:
Phase
Mandatory    1067
Economic        4
Name: count, dtype: int64


# 2031-2051

In [206]:
# Simulate the remaining years using the validated prediction and construction functions

simulation_results = []

current_network = network_2030.copy()
current_material = result_2030["material"]
carryover = result_2030["unused_budget"]

for year in range(2031, 2052):

    year_number = year - 2026

    material_column = f"Material_{year}"

    # Build this year's physical network state
    current_network[material_column] = (
        current_network["Pipe ID"]
        .map(current_material)
    )

    # Predict leak probability, conditional cost, and expected loss
    prediction = predict_future_year(
        current_network,
        material_column,
        year
    )

    # Annual budget plus unused carryover
    available_budget = (
        ANNUAL_BUDGET + carryover
    )

    # Execute economic + mandatory construction
    result = execute_yearly_construction(
        current_material=current_material,
        prediction=prediction,
        available_budget=available_budget,
        year_number=year_number
    )

    economic_units = (
        (result["plan"]["Phase"] == "Economic").sum()
        if len(result["plan"]) > 0
        else 0
    )

    mandatory_units = (
        (result["plan"]["Phase"] == "Mandatory").sum()
        if len(result["plan"]) > 0
        else 0
    )

    total_expected_loss = (
        prediction["expected_leak_loss"].sum()
    )

    simulation_results.append({
        "Year": year,
        "Year Number": year_number,
        "At-Risk Pipes": len(prediction),
        "Expected Leaks":
            prediction["predicted_leak_probability"].sum(),
        "Expected Leak Loss": total_expected_loss,
        "Available Budget": available_budget,
        "CAPEX": result["spent"],
        "Unused Budget": result["unused_budget"],
        "Work Units": len(result["plan"]),
        "Economic Units": economic_units,
        "Mandatory Units": mandatory_units,
        "Pipes Replaced": result["pipes_replaced"],
        "Iron Pipes Replaced": result["iron_pipes_replaced"],
        "Iron Meters Replaced": result["iron_meters_replaced"],
        "Required Iron Meters": result["required_iron_meters"],
        "Target Reached": result["mandatory_target_reached"],
        "Iron Remaining": result["remaining_iron"],
        "Iron Length Remaining": result["remaining_iron_length"],
        "Expected Loss Avoided": result["expected_loss_avoided"]
    })

    # Carry updated state into the next year
    current_material = result["material"]
    carryover = result["unused_budget"]


simulation_results = pd.DataFrame(
    simulation_results
)

display(simulation_results)

,Year,Year Number,At-Risk Pipes,Expected Leaks,Expected Leak Loss,Available Budget,CAPEX,Unused Budget,Work Units,Economic Units,Mandatory Units,Pipes Replaced,Iron Pipes Replaced,Iron Meters Replaced,Required Iron Meters,Target Reached,Iron Remaining,Iron Length Remaining,Expected Loss Avoided
0,2031,5,21095,430.400449,4.482933e+06,2.645168e+07,8.692892e+06,1.775879e+07,1098,1,1097,1227,1216,17631.229287,17620.485702,True,18891,352531.114815,1.237499e+06
1,2032,6,19868,396.785477,3.357457e+06,2.775879e+07,8.943413e+06,1.881537e+07,926,1,925,1036,1019,17627.682326,17616.349335,True,17872,334903.432489,9.224804e+05
2,2033,7,18832,715.035245,5.763720e+06,2.881537e+07,1.268520e+07,1.613017e+07,589,0,589,657,641,17640.823671,17615.759929,True,17231,317262.608818,4.001646e+06
3,2034,8,18175,802.557736,1.957173e+06,2.613017e+07,8.136490e+06,1.799368e+07,1161,0,1161,1231,1227,17604.687781,17602.029178,True,16004,299657.921036,5.589116e+05
4,2035,9,16944,632.099286,1.647108e+06,2.799368e+07,6.180366e+06,2.181331e+07,1458,0,1458,1510,1495,17641.361938,17624.434316,True,14509,282016.559099,3.067341e+05
5,2036,10,15434,346.057399,1.420738e+06,3.181331e+07,6.475387e+06,2.533793e+07,1456,0,1456,1480,1476,17615.727833,17610.165299,True,13033,264400.831266,2.641232e+05
6,2037,11,13954,262.803142,1.212194e+06,3.533793e+07,5.625647e+06,2.971228e+07,1485,0,1485,1510,1505,17640.553000,17621.530386,True,11528,246760.278265,1.915262e+05
7,2038,12,12444,227.024708,1.124761e+06,3.971228e+07,5.792725e+06,3.391955e+07,1317,0,1317,1353,1340,17613.485307,17608.070305,True,10188,229146.792958,1.785159e+05
8,2039,13,11091,207.846139,1.116589e+06,4.391955e+07,6.880684e+06,3.703887e+07,1165,0,1165,1198,1189,17631.714042,17621.677918,True,8999,211515.078916,1.832119e+05
9,2040,14,9893,190.953442,9.362394e+05,4.703887e+07,5.898568e+06,4.114030e+07,1186,0,1186,1199,1198,17622.535508,17617.056796,True,7801,193892.543408,1.208177e+05


In [207]:
# Verify the key budget and mandatory-replacement constraints

print(
    "All annual mandatory targets reached:",
    simulation_results["Target Reached"].all()
)

print(
    "Final iron pipes remaining:",
    simulation_results.iloc[-1]["Iron Remaining"]
)

print(
    "Final iron length remaining:",
    simulation_results.iloc[-1]["Iron Length Remaining"]
)

print(
    "Any year spent more than available budget:",
    (
        simulation_results["CAPEX"]
        > simulation_results["Available Budget"]
    ).any()
)

print(
    "Final unused budget:",
    simulation_results.iloc[-1]["Unused Budget"]
)

print(
    "Total CAPEX, 2031-2051:",
    simulation_results["CAPEX"].sum()
)

print(
    "Total expected leak loss, 2031-2051:",
    simulation_results["Expected Leak Loss"].sum()
)

print(
    "Total expected loss avoided, 2031-2051:",
    simulation_results["Expected Loss Avoided"].sum()
)

All annual mandatory targets reached: True
Final iron pipes remaining: 0
Final iron length remaining: 0.0
Any year spent more than available budget: False
Final unused budget: 67964311.45022152
Total CAPEX, 2031-2051: 158487367.94566485
Total expected leak loss, 2031-2051: 29605332.12339019
Total expected loss avoided, 2031-2051: 8377588.070254312


In [208]:
# Inspect the final-year floating-point difference

final_year = simulation_results.iloc[-1]

difference = (
    final_year["Iron Meters Replaced"]
    - final_year["Required Iron Meters"]
)

print(
    "Iron meters replaced:",
    repr(final_year["Iron Meters Replaced"])
)

print(
    "Required iron meters:",
    repr(final_year["Required Iron Meters"])
)

print(
    "Difference:",
    difference
)

print(
    "Iron remaining:",
    final_year["Iron Remaining"]
)

print(
    "Iron length remaining:",
    final_year["Iron Length Remaining"]
)

Iron meters replaced: np.float64(17590.074823657367)
Required iron meters: np.float64(17590.0748236574)
Difference: -3.2741809263825417e-11
Iron remaining: 0
Iron length remaining: 0.0


# save

In [209]:
import os

print("Current working directory:")
print(os.getcwd())

Current working directory:
e:\User\nsim\Documents\User\nsim\Documents\stat440\STAT440-star-kingdom-project\notebooks\Amy_Sim


In [210]:
from pathlib import Path

output_dir = Path.cwd()

# Save annual simulation summary
simulation_results.to_csv(
    output_dir / "star_kingdom_25yr_baseline.csv",
    index=False
)

# Save final material state
final_material_df = pd.DataFrame({
    "Pipe ID": list(current_material.keys()),
    "Final Material": list(current_material.values())
})

final_material_df.to_csv(
    output_dir / "star_kingdom_final_material_state.csv",
    index=False
)

print("Files saved successfully:")
print(output_dir / "star_kingdom_25yr_baseline.csv")
print(output_dir / "star_kingdom_final_material_state.csv")

Files saved successfully:
e:\User\nsim\Documents\User\nsim\Documents\stat440\STAT440-star-kingdom-project\notebooks\Amy_Sim\star_kingdom_25yr_baseline.csv
e:\User\nsim\Documents\User\nsim\Documents\stat440\STAT440-star-kingdom-project\notebooks\Amy_Sim\star_kingdom_final_material_state.csv


In [211]:
from pathlib import Path

# Current project folder
project_dir = Path.cwd()

# Create subfolders
(project_dir / "results").mkdir(exist_ok=True)
(project_dir / "submissions").mkdir(exist_ok=True)

print("Project directory:", project_dir)
print("\nCreated folders:")

for folder in ["results", "submissions"]:
    print("✓", project_dir / folder)

Project directory: e:\User\nsim\Documents\User\nsim\Documents\stat440\STAT440-star-kingdom-project\notebooks\Amy_Sim

Created folders:
✓ e:\User\nsim\Documents\User\nsim\Documents\stat440\STAT440-star-kingdom-project\notebooks\Amy_Sim\results
✓ e:\User\nsim\Documents\User\nsim\Documents\stat440\STAT440-star-kingdom-project\notebooks\Amy_Sim\submissions


In [212]:
# Find existing DataFrames containing selected work-unit records

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = set(obj.columns)

        if (
            "Center Pipe ID" in cols
            or {"Center x", "Center y", "Radius"}.issubset(cols)
        ):
            print(f"\n{name}")
            print("Rows:", len(obj))
            print("Columns:", list(obj.columns))


work_units
Rows: 26237
Columns: ['Center Pipe ID', 'Surface', 'Pipe length', 'Center x', 'Center y', 'Radius', 'Area', 'Covered Pipe IDs', 'Covered Pipe Count', 'Non-PU Pipe Count', 'Non-PU Pipe Meters', 'Covered Iron Count', 'Groundbreaking Cost', 'Surface Cost', 'Pipe Cost', 'Construction Cost', 'Expected Leak Benefit 2027', 'Net Benefit 2027', 'Benefit Cost Ratio 2027']

top_bcr
Rows: 20
Columns: ['Center Pipe ID', 'Surface', 'Pipe length', 'Center x', 'Center y', 'Radius', 'Area', 'Covered Pipe IDs', 'Covered Pipe Count', 'Non-PU Pipe Count', 'Non-PU Pipe Meters', 'Covered Iron Count', 'Groundbreaking Cost', 'Surface Cost', 'Pipe Cost', 'Construction Cost', 'Expected Leak Benefit 2027', 'Net Benefit 2027', 'Benefit Cost Ratio 2027']

top_diagnostic
Rows: 20
Columns: ['Pipe ID', 'Material_2027', 'Surface', 'Pipe length', 'raw_ensemble_probability', 'predicted_leak_cost', 'expected_leak_loss', 'Center Pipe ID', 'Construction Cost', 'Benefit Cost Ratio 2027']

candidate_order
Rows: 2

In [213]:
all_work_unit_plans = []

# Preserve the already completed 2030 plan
all_work_unit_plans.append(
    result_2030["plan"].assign(Year=2030)
)

In [214]:
# Preserve every selected work unit
all_work_unit_plans.append(
    result["plan"].assign(Year=year)
)

In [217]:
for name, obj in sorted(globals().items()):
    if isinstance(obj, pd.DataFrame) and "plan" in name.lower():
        print(
            f"{name}: {len(obj):,} rows | "
            f"{list(obj.columns)}"
        )

year_1_plan: 1,290 rows | ['Center Pipe ID', 'Radius', 'Construction Cost', 'Expected Benefit', 'Net Benefit', 'Benefit Cost Ratio', 'Replaced Pipe Count']
year_2_plan: 1,556 rows | ['Center Pipe ID', 'Radius', 'Construction Cost', 'Expected Benefit', 'Net Benefit', 'Benefit Cost Ratio', 'Replaced Pipe Count']
year_3_plan: 1,417 rows | ['Phase', 'Center Pipe ID', 'Construction Cost', 'Expected Benefit', 'Benefit Cost Ratio', 'Iron Meters', 'Replaced Pipe Count']


In [219]:
from pathlib import Path

# Recreate the submission folder path
project_dir = Path.cwd()
submissions_dir = project_dir / "submissions"

# Create folder if it does not exist
submissions_dir.mkdir(parents=True, exist_ok=True)

# Combine all work-unit records
baseline_01_work_units = pd.concat(
    all_work_unit_plans,
    ignore_index=True
)

# Save CSV
output_file = (
    submissions_dir / "baseline_01_work_units.csv"
)

baseline_01_work_units.to_csv(
    output_file,
    index=False
)

print("Saved successfully!")
print("Rows:", len(baseline_01_work_units))
print("Location:", output_file.resolve())

Saved successfully!
Rows: 1936
Location: E:\User\nsim\Documents\User\nsim\Documents\stat440\STAT440-star-kingdom-project\notebooks\Amy_Sim\submissions\baseline_01_work_units.csv


In [220]:
# Prepare the geometry lookup
geometry = work_units[
    ["Center Pipe ID", "Center x", "Center y", "Radius"]
].copy()

# Collect the four annual plans
annual_plans = {
    2027: year_1_plan,
    2028: year_2_plan,
    2029: year_3_plan,
    2030: result_2030["plan"]
}

combined_plans = []

for year, plan in annual_plans.items():

    temp = plan.copy()

    # Remove existing Radius if present
    if "Radius" in temp.columns:
        temp = temp.drop(columns=["Radius"])

    # Attach coordinates and radius
    temp = temp.merge(
        geometry,
        on="Center Pipe ID",
        how="left",
        validate="many_to_one"
    )

    temp["Year"] = year

    combined_plans.append(temp)

baseline_01_2027_2030 = pd.concat(
    combined_plans,
    ignore_index=True
)

# Check missing geometry
print(
    "Missing coordinates:",
    baseline_01_2027_2030[
        ["Center x", "Center y", "Radius"]
    ].isna().any(axis=1).sum()
)

print("\nWork units by year:")
print(
    baseline_01_2027_2030.groupby("Year").size()
)

display(baseline_01_2027_2030.head())

Missing coordinates: 0

Work units by year:
Year
2027    1290
2028    1556
2029    1417
2030    1071
dtype: int64


,Center Pipe ID,Construction Cost,Expected Benefit,Net Benefit,Benefit Cost Ratio,Replaced Pipe Count,Center x,Center y,Radius,Year,Phase,Iron Meters,BCR
0,P19683,634.985406,125103.740772,124468.755367,197.018293,1.0,1784.040,2800.980,1.186972,2027,NaN,NaN,NaN
1,P18358,569.634534,69119.824804,68550.190270,121.340650,1.0,1359.095,5417.305,1.051595,2027,NaN,NaN,NaN
2,P15219,962.283831,115399.546990,114437.263159,119.922567,1.0,1486.920,3253.255,1.838948,2027,NaN,NaN,NaN
3,P17427,1106.316960,131725.626437,130619.309477,119.066806,1.0,1628.065,2652.035,2.114250,2027,NaN,NaN,NaN
4,P11014,611.678328,69127.405963,68515.727635,113.012678,1.0,1360.355,5416.460,1.138915,2027,NaN,NaN,NaN


In [221]:
baseline_01_2027_2030.to_csv(
    submissions_dir / "baseline_01_work_units_2027_2030.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [222]:
print("Number of saved annual plans:", len(all_work_unit_plans))

for i, plan in enumerate(all_work_unit_plans):
    print(
        i,
        "Rows:", len(plan),
        "Columns:", list(plan.columns)
    )
    if "Year" in plan.columns:
        print("Year:", plan["Year"].unique())

Number of saved annual plans: 2
0 Rows: 1071 Columns: ['Phase', 'Center Pipe ID', 'Construction Cost', 'Expected Benefit', 'BCR', 'Iron Meters', 'Year']
Year: [2030]
1 Rows: 865 Columns: ['Phase', 'Center Pipe ID', 'Construction Cost', 'Expected Benefit', 'BCR', 'Iron Meters', 'Year']
Year: [2051]


In [223]:
# Re-run 2031-2051 simulation and preserve every annual plan

simulation_results = []
plans_2031_2051 = []

current_network = network_2030.copy()
current_material = result_2030["material"].copy()
carryover = result_2030["unused_budget"]

for year in range(2031, 2052):

    year_number = year - 2026
    material_column = f"Material_{year}"

    current_network[material_column] = (
        current_network["Pipe ID"]
        .map(current_material)
    )

    prediction = predict_future_year(
        current_network,
        material_column,
        year
    )

    available_budget = ANNUAL_BUDGET + carryover

    result = execute_yearly_construction(
        current_material=current_material,
        prediction=prediction,
        available_budget=available_budget,
        year_number=year_number
    )

    # IMPORTANT: Save the plan INSIDE the loop
    yearly_plan = result["plan"].copy()
    yearly_plan["Year"] = year

    plans_2031_2051.append(yearly_plan)

    economic_units = (
        (result["plan"]["Phase"] == "Economic").sum()
        if len(result["plan"]) > 0 else 0
    )

    mandatory_units = (
        (result["plan"]["Phase"] == "Mandatory").sum()
        if len(result["plan"]) > 0 else 0
    )

    simulation_results.append({
        "Year": year,
        "Year Number": year_number,
        "At-Risk Pipes": len(prediction),
        "Expected Leaks":
            prediction["predicted_leak_probability"].sum(),
        "Expected Leak Loss":
            prediction["expected_leak_loss"].sum(),
        "Available Budget": available_budget,
        "CAPEX": result["spent"],
        "Unused Budget": result["unused_budget"],
        "Work Units": len(result["plan"]),
        "Economic Units": economic_units,
        "Mandatory Units": mandatory_units,
        "Pipes Replaced": result["pipes_replaced"],
        "Iron Pipes Replaced": result["iron_pipes_replaced"],
        "Iron Meters Replaced": result["iron_meters_replaced"],
        "Required Iron Meters": result["required_iron_meters"],
        "Target Reached": result["mandatory_target_reached"],
        "Iron Remaining": result["remaining_iron"],
        "Iron Length Remaining": result["remaining_iron_length"],
        "Expected Loss Avoided": result["expected_loss_avoided"]
    })

    current_material = result["material"]
    carryover = result["unused_budget"]


simulation_results = pd.DataFrame(simulation_results)

print("Years captured:", len(plans_2031_2051))
print("Total work units:", sum(len(p) for p in plans_2031_2051))

print("\nWork units by year:")
for plan in plans_2031_2051:
    print(
        int(plan["Year"].iloc[0]),
        len(plan)
    )

Years captured: 21
Total work units: 18976

Work units by year:
2031 1098
2032 926
2033 589
2034 1161
2035 1458
2036 1456
2037 1485
2038 1317
2039 1165
2040 1186
2041 1036
2042 634
2043 569
2044 624
2045 654
2046 565
2047 542
2048 522
2049 523
2050 601
2051 865


In [224]:
# Geometry lookup
geometry = work_units[
    ["Center Pipe ID", "Center x", "Center y", "Radius"]
].drop_duplicates("Center Pipe ID")

# Combine 2031-2051 plans with geometry
plans_2031_2051_with_geometry = []

for plan in plans_2031_2051:

    temp = plan.copy()

    temp = temp.merge(
        geometry,
        on="Center Pipe ID",
        how="left",
        validate="many_to_one"
    )

    plans_2031_2051_with_geometry.append(temp)

# Combine all future years
baseline_01_2027_2051 = pd.concat(
    [
        baseline_01_2027_2030,
        *plans_2031_2051_with_geometry
    ],
    ignore_index=True,
    sort=False
)

# Validation
print("Total work units:", len(baseline_01_2027_2051))

print("\nWork units by year:")
print(
    baseline_01_2027_2051.groupby("Year").size()
)

print(
    "\nMissing geometry:",
    baseline_01_2027_2051[
        ["Center x", "Center y", "Radius"]
    ].isna().any(axis=1).sum()
)

print(
    "\nDuplicate year + center:",
    baseline_01_2027_2051.duplicated(
        ["Year", "Center Pipe ID"]
    ).sum()
)

# Save complete 25-year archive
output_file = (
    submissions_dir / "baseline_01_work_units_2027_2051.csv"
)

baseline_01_2027_2051.to_csv(
    output_file,
    index=False
)

print("\nSaved:", output_file.resolve())

Total work units: 24310

Work units by year:
Year
2027    1290
2028    1556
2029    1417
2030    1071
2031    1098
2032     926
2033     589
2034    1161
2035    1458
2036    1456
2037    1485
2038    1317
2039    1165
2040    1186
2041    1036
2042     634
2043     569
2044     624
2045     654
2046     565
2047     542
2048     522
2049     523
2050     601
2051     865
dtype: int64

Missing geometry: 0

Duplicate year + center: 0

Saved: E:\User\nsim\Documents\User\nsim\Documents\stat440\STAT440-star-kingdom-project\notebooks\Amy_Sim\submissions\baseline_01_work_units_2027_2051.csv


In [225]:
# Final validation of Submission 01 archive

df = baseline_01_2027_2051

print("Total rows:", len(df))
print("Unique years:", df["Year"].nunique())
print("Year range:", df["Year"].min(), "-", df["Year"].max())

print(
    "Missing coordinates:",
    df[["Center x", "Center y", "Radius"]]
    .isna().any(axis=1).sum()
)

print(
    "Duplicate work units:",
    df.duplicated(
        ["Year", "Center Pipe ID"]
    ).sum()
)

print(
    "Invalid radius:",
    (df["Radius"] <= 0).sum()
)

print(
    "Annual work-unit totals:",
    df.groupby("Year").size().sum()
)

# Verify saved file
saved_file = (
    submissions_dir / "baseline_01_work_units_2027_2051.csv"
)

print("\nFile exists:", saved_file.exists())
print("File size (MB):", round(saved_file.stat().st_size / 1e6, 2))

Total rows: 24310
Unique years: 25
Year range: 2027 - 2051
Missing coordinates: 0
Duplicate work units: 0
Invalid radius: 0
Annual work-unit totals: 24310

File exists: True
File size (MB): 3.39


In [226]:
# Prepare Submission 01
submission_01 = baseline_01_2027_2051.copy()

# Check repeated centers across years
print(
    "Repeated Center Pipe IDs:",
    submission_01["Center Pipe ID"].duplicated().sum()
)

# Keep unique construction circles
submission_01 = submission_01.drop_duplicates(
    subset=["Center Pipe ID"]
)

# Sort by year
submission_01 = submission_01.sort_values(
    ["Year", "Center Pipe ID"]
)

# Generate required coordinate-radius format
submission_lines = [
    f"({row['Center x']:.9f}, {row['Center y']:.9f})\t{row['Radius']:.2f}"
    for _, row in submission_01.iterrows()
]

# Save submission file
submission_file = submissions_dir / "submission_01.txt"

with open(submission_file, "w") as f:
    f.write("\n".join(submission_lines))

print("Submission saved:", submission_file.resolve())
print("Total circles:", len(submission_lines))
print("\nFirst 10 lines:")

for line in submission_lines[:10]:
    print(line)

Repeated Center Pipe IDs: 0
Submission saved: E:\User\nsim\Documents\User\nsim\Documents\stat440\STAT440-star-kingdom-project\notebooks\Amy_Sim\submissions\submission_01.txt
Total circles: 24310

First 10 lines:
(3731.325000000, 5422.615000000)	10.64
(3596.360000000, 5429.760000000)	7.58
(3582.715000000, 5436.390000000)	8.97
(3167.990000000, 4746.780000000)	8.34
(3353.740000000, 5096.060000000)	7.58
(3534.210000000, 2233.540000000)	12.44
(3482.645000000, 5269.715000000)	9.10
(3467.465000000, 6854.610000000)	12.84
(4589.590000000, 3952.015000000)	14.21
(3634.255000000, 5261.695000000)	11.45


In [228]:
import csv

submission_file = submissions_dir / "submission_01.csv"

with open(
    submission_file,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(
        f,
        quoting=csv.QUOTE_MINIMAL
    )

    for _, row in submission_01.iterrows():

        coordinate = (
            f"({row['Center x']:.9f}, "
            f"{row['Center y']:.9f})"
        )

        radius = f"{row['Radius']:.9f}"

        writer.writerow([coordinate, radius])

print("Saved:", submission_file.resolve())
print("Total work units:", len(submission_01))

with open(submission_file, "r") as f:
    for _ in range(10):
        print(f.readline().rstrip())

Saved: E:\User\nsim\Documents\User\nsim\Documents\stat440\STAT440-star-kingdom-project\notebooks\Amy_Sim\submissions\submission_01.csv
Total work units: 24310
"(3731.325000000, 5422.615000000)",10.637164626
"(3596.360000000, 5429.760000000)",7.580027385
"(3582.715000000, 5436.390000000)",8.970860769
"(3167.990000000, 4746.780000000)",8.340006995
"(3353.740000000, 5096.060000000)",7.580001000
"(3534.210000000, 2233.540000000)",12.441242899
"(3482.645000000, 5269.715000000)",9.095002374
"(3467.465000000, 6854.610000000)",12.840001974
"(4589.590000000, 3952.015000000)",14.207003675
"(3634.255000000, 5261.695000000)",11.454399718


In [229]:
# Preserve original annual execution order
submission_01 = baseline_01_2027_2051.copy()

# Do NOT sort by Center Pipe ID
# Do NOT remove work units unless duplicates are verified

# Create exact competition CSV
import csv

submission_file = submissions_dir / "submission_01.csv"

with open(
    submission_file,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    for _, row in submission_01.iterrows():

        coordinate = (
            f"({row['Center x']:.9f}, "
            f"{row['Center y']:.9f})"
        )

        radius = f"{row['Radius']:.9f}"

        writer.writerow([coordinate, radius])

print("Submission 01 saved:", submission_file.resolve())
print("Total work units:", len(submission_01))

print("\nFirst 10 lines:")

with open(submission_file, "r") as f:
    for _ in range(10):
        print(f.readline().rstrip())

Submission 01 saved: E:\User\nsim\Documents\User\nsim\Documents\stat440\STAT440-star-kingdom-project\notebooks\Amy_Sim\submissions\submission_01.csv
Total work units: 24310

First 10 lines:
"(1784.040000000, 2800.980000000)",1.186971935
"(1359.095000000, 5417.305000000)",1.051595028
"(1486.920000000, 3253.255000000)",1.838947709
"(1628.065000000, 2652.035000000)",2.114250276
"(1360.355000000, 5416.460000000)",1.138914956
"(745.855000000, 3168.290000000)",2.072831191
"(1626.335000000, 2655.535000000)",2.704377083
"(1533.140000000, 2906.350000000)",2.728095573
"(1626.465000000, 2649.895000000)",2.966320268
"(1542.170000000, 3424.830000000)",2.632452329


In [230]:
import numpy as np
from scipy.spatial import cKDTree

# Use the exact radii written to the submission CSV
circles = submission_01[
    ["Center x", "Center y", "Radius"]
].copy()

circles["Radius"] = circles["Radius"].round(9)

cx = circles["Center x"].to_numpy()
cy = circles["Center y"].to_numpy()
r = circles["Radius"].to_numpy()

# Build spatial index
circle_tree = cKDTree(
    np.column_stack([cx, cy])
)

# Target iron pipes remaining at the beginning of 2027
iron_pipes = iron_2027.copy()

px1 = iron_pipes["GPS x1"].to_numpy()
py1 = iron_pipes["GPS y1"].to_numpy()
px2 = iron_pipes["GPS x2"].to_numpy()
py2 = iron_pipes["GPS y2"].to_numpy()

pipe_ids = iron_pipes["Pipe ID"].to_numpy()

pipe_lengths = np.sqrt(
    (px2 - px1)**2 +
    (py2 - py1)**2
)

mid_x = (px1 + px2) / 2
mid_y = (py1 + py2) / 2

max_radius = r.max()
max_half_length = pipe_lengths.max() / 2

uncovered_pipes = []

for i in range(len(iron_pipes)):

    # Find circles that could possibly contain this pipe
    search_radius = max_radius + pipe_lengths[i] / 2

    candidates = circle_tree.query_ball_point(
        [mid_x[i], mid_y[i]],
        search_radius
    )

    if not candidates:
        uncovered_pipes.append(pipe_ids[i])
        continue

    candidates = np.asarray(candidates, dtype=int)

    # Check both endpoints against each candidate circle
    d1_sq = (
        (px1[i] - cx[candidates])**2
        + (py1[i] - cy[candidates])**2
    )

    d2_sq = (
        (px2[i] - cx[candidates])**2
        + (py2[i] - cy[candidates])**2
    )

    fully_inside = (
        (d1_sq <= r[candidates]**2)
        & (d2_sq <= r[candidates]**2)
    )

    if not fully_inside.any():
        uncovered_pipes.append(pipe_ids[i])


print("Total iron pipes:", len(iron_pipes))
print("Covered iron pipes:",
      len(iron_pipes) - len(uncovered_pipes))
print("Uncovered iron pipes:", len(uncovered_pipes))

print(
    "Coverage percentage:",
    round(
        100 * (1 - len(uncovered_pipes) / len(iron_pipes)),
        6
    ),
    "%"
)

if uncovered_pipes:
    print("\nFirst uncovered Pipe IDs:")
    print(uncovered_pipes[:20])
else:
    print("\nSUCCESS: All iron pipes are fully covered!")

Total iron pipes: 26237
Covered iron pipes: 26237
Uncovered iron pipes: 0
Coverage percentage: 100.0 %

SUCCESS: All iron pipes are fully covered!


In [231]:
import inspect

print(inspect.getsource(evaluate_work_unit_dynamic))

def evaluate_work_unit_dynamic(
    unit,
    current_material,
    expected_loss,
    pipe_length_map
):
    covered_ids = unit["Covered Pipe IDs"]

    # All currently active non-PU pipes covered by this work unit
    active_non_pu = [
        pipe_id
        for pipe_id in covered_ids
        if (
            pd.notna(current_material.get(pipe_id))
            and current_material.get(pipe_id) != "polyurethane"
        )
    ]

    # Mandatory iron pipes among the active covered pipes
    active_iron = [
        pipe_id
        for pipe_id in active_non_pu
        if current_material.get(pipe_id) in iron_materials
    ]

    non_pu_meters = sum(
        pipe_length_map[pipe_id]
        for pipe_id in active_non_pu
    )

    iron_meters = sum(
        pipe_length_map[pipe_id]
        for pipe_id in active_iron
    )

    surface_cost = (
        SURFACE_RATES[unit["Surface"]]
        * unit["Area"]**0.85
    )

    pipe_cost = (
        NON_PU_PIPE_COST_PER_METER
        * non_pu_me

In [232]:
# Build a surface lookup for all physical pipes
pipe_surface_map = (
    network_2027
    .set_index("Pipe ID")["Surface"]
    .to_dict()
)

# Calculate the distinct surface types in every candidate circle
work_units["Contained Surfaces"] = (
    work_units["Covered Pipe IDs"].apply(
        lambda ids: sorted({
            pipe_surface_map[pid]
            for pid in ids
            if pd.notna(pipe_surface_map.get(pid))
        })
    )
)

work_units["Surface Rate Count"] = (
    work_units["Contained Surfaces"].apply(len)
)

print("Work units:", len(work_units))

print(
    "Units with multiple surface types:",
    (work_units["Surface Rate Count"] > 1).sum()
)

print(
    "Units with only one surface type:",
    (work_units["Surface Rate Count"] == 1).sum()
)

display(
    work_units[
        [
            "Center Pipe ID",
            "Surface",
            "Contained Surfaces",
            "Surface Rate Count",
            "Area"
        ]
    ]
    .sort_values("Surface Rate Count", ascending=False)
    .head(10)
)

Work units: 26237
Units with multiple surface types: 819
Units with only one surface type: 25418


,Center Pipe ID,Surface,Contained Surfaces,Surface Rate Count,Area
257,P01258,road,"[road, structure, swamp, water]",4,3496.246001
258,P01259,road,"[road, structure, swamp, water]",4,2216.185261
19835,P20836,road,"[road, structure, water]",3,155.200533
7925,P08926,road,"[grassland, road, swamp]",3,2047.383397
197,P01198,road,"[grassland, road, structure]",3,2647.748608
8259,P09260,water,"[road, structure, water]",3,2314.653145
7206,P08207,water,"[grassland, road, water]",3,2587.188753
6649,P07650,road,"[grassland, road, water]",3,1889.457806
4958,P05959,grassland,"[grassland, structure, swamp]",3,2648.963462
256,P01257,road,"[road, structure, swamp]",3,2216.185182


In [234]:
def evaluate_work_unit_dynamic(
    unit,
    current_material,
    expected_loss,
    pipe_length_map
):
    covered_ids = unit["Covered Pipe IDs"]

    # All active non-PU pipes
    active_non_pu = [
        pipe_id
        for pipe_id in covered_ids
        if (
            pd.notna(current_material.get(pipe_id))
            and current_material.get(pipe_id) != "polyurethane"
        )
    ]

    # Active iron pipes
    active_iron = [
        pipe_id
        for pipe_id in active_non_pu
        if current_material.get(pipe_id) in iron_materials
    ]

    non_pu_meters = sum(
        pipe_length_map[pipe_id]
        for pipe_id in active_non_pu
    )

    iron_meters = sum(
        pipe_length_map[pipe_id]
        for pipe_id in active_iron
    )

    # Surface rate: all distinct surfaces in the circle
    contained_surfaces = {
        pipe_surface_map[pipe_id]
        for pipe_id in covered_ids
        if pd.notna(pipe_surface_map.get(pipe_id))
    }

    surface_rate = sum(
        SURFACE_RATES[surface]
        for surface in contained_surfaces
    )

    # Confirmed Formula B
    area = unit["Area"]

    surface_cost = (
        surface_rate * area**0.85
    )

    pipe_cost = (
        NON_PU_PIPE_COST_PER_METER
        * non_pu_meters
    )

    # Official construction cost, rounded to cents
    construction_cost = round(
        GROUNDBREAKING_COST
        + surface_cost
        + pipe_cost,
        2
    )

    expected_benefit = sum(
        expected_loss.get(pipe_id, 0)
        for pipe_id in active_non_pu
    )

    net_benefit = (
        expected_benefit - construction_cost
    )

    benefit_cost_ratio = (
        expected_benefit / construction_cost
        if construction_cost > 0
        else 0
    )

    return {
        "active_pipe_ids": active_non_pu,
        "active_pipe_count": len(active_non_pu),

        "iron_pipe_ids": active_iron,
        "iron_pipe_count": len(active_iron),
        "iron_meters": iron_meters,

        "non_pu_meters": non_pu_meters,

        "surface_rate": surface_rate,
        "contained_surfaces": sorted(contained_surfaces),

        "construction_cost": construction_cost,
        "expected_benefit": expected_benefit,
        "net_benefit": net_benefit,
        "benefit_cost_ratio": benefit_cost_ratio
    }

In [235]:
# Compare old and corrected surface costs

surface_diagnostic = work_units.copy()

surface_diagnostic["Old Surface Rate"] = (
    surface_diagnostic["Surface"].map(SURFACE_RATES)
)

surface_diagnostic["Correct Surface Rate"] = (
    surface_diagnostic["Contained Surfaces"].apply(
        lambda surfaces: sum(
            SURFACE_RATES[s] for s in surfaces
        )
    )
)

surface_diagnostic["Old Surface Cost"] = (
    surface_diagnostic["Old Surface Rate"]
    * surface_diagnostic["Area"]**0.85
)

surface_diagnostic["Correct Surface Cost"] = (
    surface_diagnostic["Correct Surface Rate"]
    * surface_diagnostic["Area"]**0.85
)

surface_diagnostic["Additional Cost"] = (
    surface_diagnostic["Correct Surface Cost"]
    - surface_diagnostic["Old Surface Cost"]
)

print("Total candidate work units:",
      len(surface_diagnostic))

print("Affected candidates:",
      (surface_diagnostic["Additional Cost"] > 0).sum())

print("Total additional surface cost across candidates:",
      round(surface_diagnostic["Additional Cost"].sum(), 2))

print("Maximum additional cost:",
      round(surface_diagnostic["Additional Cost"].max(), 2))

print("\nTop 15 affected candidates:")

display(
    surface_diagnostic[
        [
            "Center Pipe ID",
            "Contained Surfaces",
            "Old Surface Rate",
            "Correct Surface Rate",
            "Area",
            "Additional Cost"
        ]
    ]
    .sort_values("Additional Cost", ascending=False)
    .head(15)
)

Total candidate work units: 26237
Affected candidates: 819
Total additional surface cost across candidates: 8109582.45
Maximum additional cost: 98721.1

Top 15 affected candidates:


,Center Pipe ID,Contained Surfaces,Old Surface Rate,Correct Surface Rate,Area,Additional Cost
42930,P43931,"[road, structure, water]",17,99,4209.502065,98721.096456
42927,P43928,"[road, structure, water]",17,99,4209.501986,98721.094890
42928,P43929,"[road, structure, water]",17,99,4209.501986,98721.094890
257,P01258,"[road, structure, swamp, water]",17,111,3496.246001,96647.258397
43003,P44004,"[road, structure, water]",17,99,3496.246001,84309.310517
43006,P44007,"[road, structure, water]",17,99,3496.246001,84309.310517
5642,P06643,"[road, structure, swamp]",17,79,4196.063509,74440.183392
5173,P06174,"[road, structure, swamp]",17,79,3921.870557,70284.726093
7576,P08577,"[grassland, structure, swamp]",3,65,3840.572420,69044.367344
4368,P05369,"[road, structure, water]",17,99,2627.422503,66132.399447


In [236]:
# Identify selected work units affected by corrected surface rates

selected_ids = set(
    submission_01["Center Pipe ID"]
)

selected_diagnostic = surface_diagnostic[
    surface_diagnostic["Center Pipe ID"].isin(selected_ids)
].copy()

affected_selected = selected_diagnostic[
    selected_diagnostic["Additional Cost"] > 0
]

print(
    "Selected work units:",
    len(submission_01)
)

print(
    "Selected units with multiple surfaces:",
    len(affected_selected)
)

print(
    "Additional cost across selected units:",
    round(
        affected_selected["Additional Cost"].sum(),
        2
    )
)

print(
    "Maximum additional selected-unit cost:",
    round(
        affected_selected["Additional Cost"].max(),
        2
    )
)

display(
    affected_selected[
        [
            "Center Pipe ID",
            "Contained Surfaces",
            "Old Surface Rate",
            "Correct Surface Rate",
            "Additional Cost"
        ]
    ]
    .sort_values("Additional Cost", ascending=False)
    .head(15)
)

Selected work units: 24310
Selected units with multiple surfaces: 782
Additional cost across selected units: 7947104.2
Maximum additional selected-unit cost: 98721.1


,Center Pipe ID,Contained Surfaces,Old Surface Rate,Correct Surface Rate,Additional Cost
42930,P43931,"[road, structure, water]",17,99,98721.096456
42928,P43929,"[road, structure, water]",17,99,98721.094890
42927,P43928,"[road, structure, water]",17,99,98721.094890
257,P01258,"[road, structure, swamp, water]",17,111,96647.258397
43006,P44007,"[road, structure, water]",17,99,84309.310517
43003,P44004,"[road, structure, water]",17,99,84309.310517
5642,P06643,"[road, structure, swamp]",17,79,74440.183392
5173,P06174,"[road, structure, swamp]",17,79,70284.726093
7576,P08577,"[grassland, structure, swamp]",3,65,69044.367344
4368,P05369,"[road, structure, water]",17,99,66132.399447


In [237]:
def evaluate_work_unit_dynamic(
    unit,
    current_material,
    expected_loss,
    pipe_length_map
):
    covered_ids = unit["Covered Pipe IDs"]

    # All currently active non-polyurethane pipes
    active_non_pu = [
        pipe_id
        for pipe_id in covered_ids
        if (
            pd.notna(current_material.get(pipe_id))
            and current_material.get(pipe_id) != "polyurethane"
        )
    ]

    # Mandatory iron pipes
    active_iron = [
        pipe_id
        for pipe_id in active_non_pu
        if current_material.get(pipe_id) in iron_materials
    ]

    # Active pipe lengths
    non_pu_meters = sum(
        pipe_length_map[pipe_id]
        for pipe_id in active_non_pu
    )

    iron_meters = sum(
        pipe_length_map[pipe_id]
        for pipe_id in active_iron
    )

    # All distinct surfaces in the original work-unit geometry
    contained_surfaces = {
        pipe_surface_map[pipe_id]
        for pipe_id in covered_ids
        if pd.notna(pipe_surface_map.get(pipe_id))
    }

    # Corrected surface rate
    surface_rate = sum(
        SURFACE_RATES[surface]
        for surface in contained_surfaces
    )

    # Professor-confirmed Formula B
    surface_cost = (
        surface_rate * unit["Area"]**0.85
    )

    pipe_cost = (
        NON_PU_PIPE_COST_PER_METER
        * non_pu_meters
    )

    # Official construction cost, rounded to cents
    construction_cost = round(
        GROUNDBREAKING_COST
        + surface_cost
        + pipe_cost,
        2
    )

    # Expected leak benefit
    expected_benefit = sum(
        expected_loss.get(pipe_id, 0)
        for pipe_id in active_non_pu
    )

    net_benefit = (
        expected_benefit - construction_cost
    )

    benefit_cost_ratio = (
        expected_benefit / construction_cost
        if construction_cost > 0
        else 0
    )

    return {
        "active_pipe_ids": active_non_pu,
        "active_pipe_count": len(active_non_pu),

        "iron_pipe_ids": active_iron,
        "iron_pipe_count": len(active_iron),
        "iron_meters": iron_meters,

        "non_pu_meters": non_pu_meters,

        "surface_rate": surface_rate,
        "contained_surfaces": sorted(contained_surfaces),

        "construction_cost": construction_cost,
        "expected_benefit": expected_benefit,
        "net_benefit": net_benefit,
        "benefit_cost_ratio": benefit_cost_ratio
    }

In [239]:
# Find existing prediction and construction-result variables

for name, obj in globals().copy().items():

    if any(word in name.lower() for word in [
        "result",
        "prediction",
        "material",
        "risk_2027",
        "leak_2027"
    ]):
        if not name.startswith("_"):
            print(
                f"{name}: {type(obj).__name__}"
            )

logistic_results: DataFrame
xgb_results: DataFrame
spatial_fold_results: DataFrame
test_predictions: DataFrame
spatial_fold_results_fixed: DataFrame
fold_predictions: Series
catboost_fold_results: DataFrame
lightgbm_fold_results: DataFrame
prediction_correlation: DataFrame
comparison_results: dict
linear_cost_results: DataFrame
xgb_cost_results: DataFrame
log_xgb_cost_results: DataFrame
smearing_results: DataFrame
combined_cost_results: dict
predictions: DataFrame
clipped_linear_results: dict
calibrated_linear_results: DataFrame
material_cost_summary: DataFrame
surface_group_results: DataFrame
interaction_results: DataFrame
risk_2027: DataFrame
leak_2027: DataFrame
prediction_2027: DataFrame
iron_materials: list
current_material: dict
test_result: dict
result: dict
prediction_2028: DataFrame
no_year_results: DataFrame
frozen_year_results: DataFrame
test_for_prediction: DataFrame
prediction_2028_frozen: DataFrame
current_material_2028: dict
prediction_2029: DataFrame
current_material_20

In [240]:
# Inspect relevant 2027 variables

for name in [
    "risk_2027",
    "leak_2027",
    "prediction_2027",
    "final_material_df",
    "network_2027"
]:
    if name in globals():
        obj = globals()[name]

        print(f"\n{name}")
        print("Type:", type(obj).__name__)

        if isinstance(obj, pd.DataFrame):
            print("Shape:", obj.shape)
            print("Columns:", obj.columns.tolist())
            display(obj.head(2))


risk_2027
Type: DataFrame
Shape: (28926, 13)
Columns: ['Pipe ID', 'Lay date', 'Material', 'Surface', 'GPS x1', 'GPS y1', 'GPS x2', 'GPS y2', 'Year', 'Age', 'Pipe length', 'Midpoint x', 'Midpoint y']


,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2,Year,Age,Pipe length,Midpoint x,Midpoint y
0,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,2027,71.811088,12.140000,6130.22,5020.970
3,P01004,1982-05-22,cast iron,structure,2836.68,3377.95,2848.80,3405.12,2027,44.613279,29.750686,2842.74,3391.535



leak_2027
Type: DataFrame
Shape: (28926, 9)
Columns: ['Pipe ID', 'Material', 'Surface', 'Age', 'Pipe length', 'xgb_probability', 'catboost_probability', 'lightgbm_probability', 'raw_ensemble_probability']


,Pipe ID,Material,Surface,Age,Pipe length,xgb_probability,catboost_probability,lightgbm_probability,raw_ensemble_probability
0,P01001,copper,grassland,71.811088,12.140000,0.009348,0.010553,0.009720,0.009874
3,P01004,cast iron,structure,44.613279,29.750686,0.049130,0.043365,0.046025,0.046173



prediction_2027
Type: DataFrame
Shape: (28926, 4)
Columns: ['Pipe ID', 'raw_ensemble_probability', 'predicted_leak_cost', 'expected_leak_loss']


,Pipe ID,raw_ensemble_probability,predicted_leak_cost,expected_leak_loss
0,P01001,0.009874,0.000000,0.000000
1,P01004,0.046173,37033.575821,1709.969038



final_material_df
Type: DataFrame
Shape: (43039, 2)
Columns: ['Pipe ID', 'Final Material']


,Pipe ID,Final Material
0,P01001,copper
1,P01002,polyurethane



network_2027
Type: DataFrame
Shape: (43039, 14)
Columns: ['Pipe ID', 'Lay date', 'Material', 'Surface', 'GPS x1', 'GPS y1', 'GPS x2', 'GPS y2', 'Material_2027', 'Pipe length', 'Midpoint x', 'Midpoint y', 'Is_iron_2027', 'Is_non_PU_2027']


,Pipe ID,Lay date,Material,Surface,GPS x1,GPS y1,GPS x2,GPS y2,Material_2027,Pipe length,Midpoint x,Midpoint y,Is_iron_2027,Is_non_PU_2027
0,P01001,1955-03-11,copper,grassland,6136.29,5020.97,6124.15,5020.97,copper,12.140000,6130.22,5020.97,False,True
1,P01002,1971-09-19,cast iron,water,6512.35,4147.58,6536.59,4201.94,polyurethane,59.519637,6524.47,4174.76,False,False


In [241]:
# Submission 02: 2027 validation inputs

test_material = (
    network_2027
    .set_index("Pipe ID")["Material_2027"]
    .to_dict()
)

test_expected_loss = (
    prediction_2027
    .set_index("Pipe ID")["expected_leak_loss"]
    .to_dict()
)

# Original surface information for all physical pipes
pipe_surface_map = (
    network_2027
    .set_index("Pipe ID")["Surface"]
    .to_dict()
)

print("Material records:", len(test_material))
print("Expected-loss records:", len(test_expected_loss))
print("Surface records:", len(pipe_surface_map))

print(
    "2027 iron pipes:",
    sum(m in iron_materials for m in test_material.values())
)

Material records: 43039
Expected-loss records: 28926
Surface records: 43039
2027 iron pipes: 26237


In [242]:
test_ids = ["P43931", "P01258", "P06643"]

for center_id in test_ids:

    unit = work_units[
        work_units["Center Pipe ID"] == center_id
    ].iloc[0]

    result = evaluate_work_unit_dynamic(
        unit,
        test_material,
        test_expected_loss,
        pipe_length_map
    )

    print("\nCenter:", center_id)
    print("Surfaces:", result["contained_surfaces"])
    print("Surface rate:", result["surface_rate"])
    print("Construction cost:", result["construction_cost"])
    print("Expected benefit:", result["expected_benefit"])
    print("BCR:", result["benefit_cost_ratio"])


Center: P43931
Surfaces: ['road', 'structure', 'water']
Surface rate: 99
Construction cost: 143508.28
Expected benefit: 117693.31955924531
BCR: 0.8201151847074282

Center: P01258
Surfaces: ['road', 'structure', 'swamp', 'water']
Surface rate: 111
Construction cost: 177159.21
Expected benefit: 21439.603964698817
BCR: 0.12101885058473008

Center: P06643
Surfaces: ['road', 'structure', 'swamp']
Surface rate: 79
Construction cost: 124905.52
Expected benefit: 7807.778423993063
BCR: 0.0625094745531908


In [243]:
# Submission 02: Corrected candidate evaluation

corrected_candidate_rows = []

for _, unit in work_units.iterrows():

    result = evaluate_work_unit_dynamic(
        unit,
        test_material,
        test_expected_loss,
        pipe_length_map
    )

    corrected_candidate_rows.append({
        "Center Pipe ID": unit["Center Pipe ID"],
        "Center x": unit["Center x"],
        "Center y": unit["Center y"],
        "Radius": unit["Radius"],
        "Area": unit["Area"],
        "Contained Surfaces": result["contained_surfaces"],
        "Surface Rate": result["surface_rate"],
        "Active Pipe Count": result["active_pipe_count"],
        "Iron Pipe Count": result["iron_pipe_count"],
        "Iron Meters": result["iron_meters"],
        "Construction Cost": result["construction_cost"],
        "Expected Benefit": result["expected_benefit"],
        "Net Benefit": result["net_benefit"],
        "BCR": result["benefit_cost_ratio"]
    })

baseline_02_candidates = pd.DataFrame(
    corrected_candidate_rows
)

# Save the complete candidate evaluation
baseline_02_candidates.to_csv(
    submissions_dir / "baseline_02_candidate_evaluations.csv",
    index=False
)

# Summary
print("Total candidates:", len(baseline_02_candidates))

print(
    "Candidates with BCR > 1:",
    (baseline_02_candidates["BCR"] > 1).sum()
)

print(
    "Candidates with BCR <= 1:",
    (baseline_02_candidates["BCR"] <= 1).sum()
)

print(
    "Candidates covering iron pipes:",
    (baseline_02_candidates["Iron Pipe Count"] > 0).sum()
)

print(
    "Total corrected standalone construction cost:",
    round(baseline_02_candidates["Construction Cost"].sum(), 2)
)

print(
    "Maximum BCR:",
    baseline_02_candidates["BCR"].max()
)

print("\nTop 20 corrected candidates:")

display(
    baseline_02_candidates
    .sort_values("BCR", ascending=False)
    .head(20)
)

Total candidates: 26237
Candidates with BCR > 1: 1400
Candidates with BCR <= 1: 24837
Candidates covering iron pipes: 26237
Total corrected standalone construction cost: 205915639.39
Maximum BCR: 197.01686762381576

Top 20 corrected candidates:


,Center Pipe ID,Center x,Center y,Radius,Area,Contained Surfaces,Surface Rate,Active Pipe Count,Iron Pipe Count,Iron Meters,Construction Cost,Expected Benefit,Net Benefit,BCR
11213,P19683,1784.040,2800.980,1.186972,4.426197,[road],17,1,1,2.373942,634.99,125103.740772,124468.750772,197.016868
10385,P18358,1359.095,5417.305,1.051595,3.474137,[road],17,1,1,2.103188,569.63,69119.824804,68550.194804,121.341616
8434,P15219,1486.920,3253.255,1.838948,10.624014,[road],17,1,1,3.677893,962.28,115399.546990,114437.266990,119.923044
9822,P17427,1628.065,2652.035,2.114250,14.043090,[road],17,1,1,4.228499,1106.32,131725.626437,130619.306437,119.066478
5576,P11014,1360.355,5416.460,1.138915,4.075046,[road],17,1,1,2.277828,611.68,69127.405963,68515.725963,113.012369
11463,P20308,745.855,3168.290,2.072831,13.498259,[road],17,1,1,4.145660,1084.44,103339.956426,102255.516426,95.293383
9821,P17426,1626.335,2655.535,2.704377,22.976526,[road],17,1,1,5.408752,1425.84,131875.598604,130449.758604,92.489759
11245,P19863,1533.140,2906.350,2.728096,23.381320,[road],17,2,2,10.773265,2502.39,228473.929327,225971.539327,91.302287
9820,P17425,1626.465,2649.895,2.966320,27.643049,[road],17,1,1,5.932639,1572.15,132091.595157,130519.445157,84.019715
8481,P15328,1542.170,3424.830,2.632452,21.770625,[road],17,1,1,5.264903,1386.13,116343.771104,114957.641104,83.934242


In [244]:
import inspect

# Find existing planning and construction functions

for name, obj in globals().copy().items():

    if callable(obj) and any(
        word in name.lower()
        for word in [
            "plan",
            "construct",
            "execute",
            "optimize",
            "select",
            "work_unit"
        ]
    ):

        try:
            print(f"\n{'='*60}")
            print("FUNCTION:", name)
            print(inspect.signature(obj))
        except Exception:
            pass


FUNCTION: construction_cost
(radius, surface, non_pu_pipe_meters)

FUNCTION: evaluate_work_unit
(unit, current_material, expected_loss)

FUNCTION: evaluate_work_unit_dynamic
(unit, current_material, expected_loss, pipe_length_map)

FUNCTION: execute_yearly_construction
(current_material, prediction, available_budget, year_number)


In [245]:
# Check existing annual plans and results

for name in [
    "year_1_plan",
    "year_2_plan",
    "year_3_plan",
    "year_4_plan",
    "result_2028",
    "result_2029",
    "result_2030",
    "simulation_results",
    "execute_yearly_construction",
    "work_units"
]:
    if name in globals():
        obj = globals()[name]

        print(
            name,
            type(obj).__name__,
            getattr(obj, "shape", "")
        )

year_1_plan DataFrame (1290, 7)
year_2_plan DataFrame (1556, 7)
year_3_plan DataFrame (1417, 7)
result_2030 dict 
simulation_results DataFrame (21, 19)
execute_yearly_construction function 
work_units DataFrame (26237, 21)


In [246]:
import inspect

print(inspect.getsource(construction_cost))
print("\n--- evaluate_work_unit ---")
print(inspect.getsource(evaluate_work_unit))
print("\n--- execute_yearly_construction ---")
print(inspect.getsource(execute_yearly_construction))

def construction_cost(
    radius,
    surface,
    non_pu_pipe_meters
):
    area = np.pi * radius**2

    surface_cost = (
        SURFACE_RATES[surface]
        * area**0.85
    )

    pipe_cost = (
        NON_PU_PIPE_COST_PER_METER
        * non_pu_pipe_meters
    )

    total_cost = (
        GROUNDBREAKING_COST
        + surface_cost
        + pipe_cost
    )

    return total_cost


--- evaluate_work_unit ---
def evaluate_work_unit(unit, current_material, expected_loss):

    covered_ids = unit["Covered Pipe IDs"]

    # Pipes that are still non-polyurethane
    active_non_pu = [
        pipe_id
        for pipe_id in covered_ids
        if (
            pd.notna(current_material.get(pipe_id))
            and current_material.get(pipe_id) != "polyurethane"
        )
    ]

    # Total non-PU pipe length currently replaced by this work unit
    non_pu_meters = sum(
        pipe_length_map[pipe_id]
        for pipe_id in active_non_pu
    )

    # Construction cost under the curr

In [247]:
def construction_cost(
    radius,
    contained_surfaces,
    non_pu_pipe_meters
):
    area = np.pi * radius**2

    surface_rate = sum(
        SURFACE_RATES[surface]
        for surface in set(contained_surfaces)
    )

    surface_cost = (
        surface_rate * area**0.85
    )

    pipe_cost = (
        NON_PU_PIPE_COST_PER_METER
        * non_pu_pipe_meters
    )

    total_cost = (
        GROUNDBREAKING_COST
        + surface_cost
        + pipe_cost
    )

    return round(total_cost, 2)

In [248]:
def evaluate_work_unit_dynamic(
    unit,
    current_material,
    expected_loss,
    pipe_length_map
):
    covered_ids = unit["Covered Pipe IDs"]

    # Active non-polyurethane pipes
    active_non_pu = [
        pid for pid in covered_ids
        if (
            pd.notna(current_material.get(pid))
            and current_material.get(pid) != "polyurethane"
        )
    ]

    # Active iron pipes
    active_iron = [
        pid for pid in active_non_pu
        if current_material.get(pid) in iron_materials
    ]

    non_pu_meters = sum(
        pipe_length_map[pid]
        for pid in active_non_pu
    )

    iron_meters = sum(
        pipe_length_map[pid]
        for pid in active_iron
    )

    # Original geometry: all pipes wholly inside the circle
    contained_surfaces = {
        pipe_surface_map[pid]
        for pid in covered_ids
        if pd.notna(pipe_surface_map.get(pid))
    }

    construction = construction_cost(
        radius=unit["Radius"],
        contained_surfaces=contained_surfaces,
        non_pu_pipe_meters=non_pu_meters
    )

    expected_benefit = sum(
        expected_loss.get(pid, 0)
        for pid in active_non_pu
    )

    net_benefit = expected_benefit - construction

    bcr = (
        expected_benefit / construction
        if construction > 0
        else 0
    )

    return {
        "active_pipe_ids": active_non_pu,
        "active_pipe_count": len(active_non_pu),

        "iron_pipe_ids": active_iron,
        "iron_pipe_count": len(active_iron),
        "iron_meters": iron_meters,

        "non_pu_meters": non_pu_meters,

        "contained_surfaces": sorted(contained_surfaces),

        "construction_cost": construction,
        "expected_benefit": expected_benefit,
        "net_benefit": net_benefit,
        "benefit_cost_ratio": bcr
    }

In [249]:
import ast
import inspect

functions_to_check = [
    construction_cost,
    evaluate_work_unit,
    evaluate_work_unit_dynamic,
    execute_yearly_construction
]

for func in functions_to_check:
    print(f"\n{func.__name__}")
    print(inspect.getsource(func))


construction_cost
def construction_cost(
    radius,
    contained_surfaces,
    non_pu_pipe_meters
):
    area = np.pi * radius**2

    surface_rate = sum(
        SURFACE_RATES[surface]
        for surface in set(contained_surfaces)
    )

    surface_cost = (
        surface_rate * area**0.85
    )

    pipe_cost = (
        NON_PU_PIPE_COST_PER_METER
        * non_pu_pipe_meters
    )

    total_cost = (
        GROUNDBREAKING_COST
        + surface_cost
        + pipe_cost
    )

    return round(total_cost, 2)


evaluate_work_unit
def evaluate_work_unit(unit, current_material, expected_loss):

    covered_ids = unit["Covered Pipe IDs"]

    # Pipes that are still non-polyurethane
    active_non_pu = [
        pipe_id
        for pipe_id in covered_ids
        if (
            pd.notna(current_material.get(pipe_id))
            and current_material.get(pipe_id) != "polyurethane"
        )
    ]

    # Total non-PU pipe length currently replaced by this work unit
    non_pu_meters

In [250]:
# Find notebook variables that contain old construction calculations
# and inspect the current function dependencies.

print("Current corrected functions loaded:")
print("construction_cost:", construction_cost.__code__.co_firstlineno)
print("evaluate_work_unit_dynamic:",
      evaluate_work_unit_dynamic.__code__.co_firstlineno)
print("execute_yearly_construction:",
      execute_yearly_construction.__code__.co_firstlineno)

Current corrected functions loaded:
construction_cost: 1
evaluate_work_unit_dynamic: 1
execute_yearly_construction: 3


In [251]:
def evaluate_work_unit(unit, current_material, expected_loss):

    result = evaluate_work_unit_dynamic(
        unit,
        current_material,
        expected_loss,
        pipe_length_map
    )

    return {
        "active_pipe_ids": result["active_pipe_ids"],
        "active_pipe_count": result["active_pipe_count"],
        "non_pu_meters": result["non_pu_meters"],
        "construction_cost": result["construction_cost"],
        "expected_benefit": result["expected_benefit"],
        "net_benefit": result["net_benefit"],
        "benefit_cost_ratio": result["benefit_cost_ratio"]
    }

In [252]:
test_ids = ["P43931", "P01258", "P06643"]

for center_id in test_ids:

    unit = work_units[
        work_units["Center Pipe ID"] == center_id
    ].iloc[0]

    result = evaluate_work_unit_dynamic(
        unit,
        test_material,
        test_expected_loss,
        pipe_length_map
    )

    print(
        center_id,
        " | Rate:", result["contained_surfaces"],
        " | Cost:", result["construction_cost"],
        " | BCR:", round(result["benefit_cost_ratio"], 6)
    )

P43931  | Rate: ['road', 'structure', 'water']  | Cost: 143508.28  | BCR: 0.820115
P01258  | Rate: ['road', 'structure', 'swamp', 'water']  | Cost: 177159.21  | BCR: 0.121019
P06643  | Rate: ['road', 'structure', 'swamp']  | Cost: 124905.52  | BCR: 0.062509


In [253]:
# ==========================================
# SUBMISSION 02 — 2027 CONSTRUCTION PLAN
# ==========================================

# Initial 2027 material state
material_2027_s02 = (
    network_2027
    .set_index("Pipe ID")["Material_2027"]
    .to_dict()
)

# 2027 expected leak predictions
prediction_2027_s02 = prediction_2027.copy()

# Annual budget for 2027
available_budget_2027_s02 = ANNUAL_BUDGET

# Execute corrected construction planner
result_2027_s02 = execute_yearly_construction(
    current_material=material_2027_s02,
    prediction=prediction_2027_s02,
    available_budget=available_budget_2027_s02,
    year_number=1
)

# Save 2027 plan
baseline_02_year_2027 = result_2027_s02["plan"].copy()

print("2027 CAPEX:", result_2027_s02["spent"])
print("Unused budget:", result_2027_s02["unused_budget"])
print("Selected work units:", len(baseline_02_year_2027))
print("Iron pipes replaced:", result_2027_s02["iron_pipes_replaced"])
print("Iron meters replaced:", result_2027_s02["iron_meters_replaced"])
print("Required iron meters:", result_2027_s02["required_iron_meters"])
print("Mandatory target reached:", result_2027_s02["mandatory_target_reached"])
print("Iron remaining:", result_2027_s02["remaining_iron"])

display(baseline_02_year_2027.head(10))

2027 CAPEX: 6852083.840000018
Unused budget: 3147916.1599999825
Selected work units: 1286
Iron pipes replaced: 1588
Iron meters replaced: 18876.303509822217
Required iron meters: 17627.0929612878
Mandatory target reached: True
Iron remaining: 24649


,Phase,Center Pipe ID,Construction Cost,Expected Benefit,BCR,Iron Meters
0,Economic,P19683,634.99,125103.740772,197.016868,2.373942
1,Economic,P18358,569.63,69119.824804,121.341616,2.103188
2,Economic,P15219,962.28,115399.546990,119.923044,3.677893
3,Economic,P17427,1106.32,131725.626437,119.066478,4.228499
4,Economic,P11014,611.68,69127.405963,113.012369,2.277828
5,Economic,P20308,1084.44,103339.956426,95.293383,4.145660
6,Economic,P17426,1425.84,131875.598604,92.489759,5.408752
7,Economic,P19863,2502.39,228473.929327,91.302287,10.773265
8,Economic,P17425,1572.15,132091.595157,84.019715,5.932639
9,Economic,P15328,1386.13,116343.771104,83.934242,5.264903


In [254]:
# ==========================================
# SUBMISSION 02 — 2028 CONSTRUCTION PLAN
# ==========================================

# Updated material state after 2027
current_material_2028_s02 = (
    result_2027_s02["material"].copy()
)

# Build the 2028 network state
network_2028_s02 = network_2027.copy()

network_2028_s02["Material_2028"] = (
    network_2028_s02["Pipe ID"]
    .map(current_material_2028_s02)
)

# Predict 2028 leak probability and expected loss
prediction_2028_s02 = predict_future_year(
    network_2028_s02,
    "Material_2028",
    2028
)

# Carry unused budget from 2027
carryover_2028_s02 = (
    result_2027_s02["unused_budget"]
)

available_budget_2028_s02 = (
    ANNUAL_BUDGET + carryover_2028_s02
)

# Execute corrected 2028 construction plan
result_2028_s02 = execute_yearly_construction(
    current_material=current_material_2028_s02,
    prediction=prediction_2028_s02,
    available_budget=available_budget_2028_s02,
    year_number=2
)

baseline_02_year_2028 = (
    result_2028_s02["plan"].copy()
)

print("2028 predicted pipes:", len(prediction_2028_s02))
print(
    "Expected leaks:",
    prediction_2028_s02["predicted_leak_probability"].sum()
    if "predicted_leak_probability" in prediction_2028_s02.columns
    else "Check prediction columns"
)

print("Available budget:", available_budget_2028_s02)
print("2028 CAPEX:", result_2028_s02["spent"])
print("Unused budget:", result_2028_s02["unused_budget"])
print("Selected work units:", len(baseline_02_year_2028))
print("Iron pipes replaced:", result_2028_s02["iron_pipes_replaced"])
print("Iron meters replaced:", result_2028_s02["iron_meters_replaced"])
print("Required iron meters:", result_2028_s02["required_iron_meters"])
print("Mandatory target reached:", result_2028_s02["mandatory_target_reached"])
print("Iron remaining:", result_2028_s02["remaining_iron"])

display(baseline_02_year_2028.head(10))

2028 predicted pipes: 25647
Expected leaks: 2635.031958026889
Available budget: 13147916.159999982
2028 CAPEX: 4939645.090000002
Unused budget: 8208271.06999998
Selected work units: 1790
Iron pipes replaced: 1834
Iron meters replaced: 16380.245029468051
Required iron meters: 16377.882371465625
Mandatory target reached: True
Iron remaining: 22815


,Phase,Center Pipe ID,Construction Cost,Expected Benefit,BCR,Iron Meters
0,Economic,P15900,13167.87,96721.527843,7.345268,39.041122
1,Economic,P11345,192.71,954.905973,4.955145,0.456070
2,Economic,P17286,194.52,953.849491,4.903606,0.464866
3,Economic,P14533,401.15,1740.853384,4.339657,1.481891
4,Economic,P14527,408.03,1741.958944,4.269193,1.515388
5,Economic,P11174,1136.14,4795.816824,4.221150,3.876596
6,Economic,P16142,440.97,1782.284595,4.041737,1.675470
7,Economic,P16145,443.55,1785.278203,4.024976,1.687987
8,Economic,P17462,1135.80,4514.127289,3.974403,3.875526
9,Economic,P18308,1273.84,4980.482316,3.909818,4.308526


In [255]:
# ==========================================
# SUBMISSION 02 — 2029 CONSTRUCTION PLAN
# ==========================================

# Updated material state after 2028
current_material_2029_s02 = (
    result_2028_s02["material"].copy()
)

# Build 2029 network state
network_2029_s02 = network_2027.copy()

network_2029_s02["Material_2029"] = (
    network_2029_s02["Pipe ID"]
    .map(current_material_2029_s02)
)

# Generate 2029 predictions
prediction_2029_s02 = predict_future_year(
    network_2029_s02,
    "Material_2029",
    2029
)

# Carryover from 2028
carryover_2029_s02 = (
    result_2028_s02["unused_budget"]
)

available_budget_2029_s02 = (
    ANNUAL_BUDGET + carryover_2029_s02
)

# Execute corrected construction planner
result_2029_s02 = execute_yearly_construction(
    current_material=current_material_2029_s02,
    prediction=prediction_2029_s02,
    available_budget=available_budget_2029_s02,
    year_number=3
)

baseline_02_year_2029 = (
    result_2029_s02["plan"].copy()
)

print("2029 predicted pipes:", len(prediction_2029_s02))
print("Available budget:", available_budget_2029_s02)
print("2029 CAPEX:", result_2029_s02["spent"])
print("Unused budget:", result_2029_s02["unused_budget"])
print("Selected work units:", len(baseline_02_year_2029))
print("Iron pipes replaced:", result_2029_s02["iron_pipes_replaced"])
print("Iron meters replaced:", result_2029_s02["iron_meters_replaced"])
print("Required iron meters:", result_2029_s02["required_iron_meters"])
print("Mandatory target reached:", result_2029_s02["mandatory_target_reached"])
print("Iron remaining:", result_2029_s02["remaining_iron"])

display(baseline_02_year_2029.head(10))

2029 predicted pipes: 23813
Available budget: 18208271.069999978
2029 CAPEX: 5503241.920000005
Unused budget: 12705029.149999972
Selected work units: 1339
Iron pipes replaced: 1425
Iron meters replaced: 17629.251727753108
Required iron meters: 17624.730261997596
Mandatory target reached: True
Iron remaining: 21390


,Phase,Center Pipe ID,Construction Cost,Expected Benefit,BCR,Iron Meters
0,Economic,P38620,4661.88,24308.765439,5.214370,10.933590
1,Economic,P19571,443.30,880.884938,1.987108,1.649758
2,Economic,P16067,1220.87,1939.696697,1.588782,5.390232
3,Economic,P10070,2546.46,3605.451490,1.415868,9.207155
4,Economic,P32560,4634.64,6427.347791,1.386806,10.885702
5,Economic,P15473,1287.60,1576.836357,1.224632,3.889113
6,Economic,P20984,1487.47,1802.075030,1.211503,4.405145
7,Economic,P16691,1283.49,1520.684845,1.184805,3.878273
8,Economic,P10098,1330.24,1575.637080,1.184476,4.001062
9,Economic,P18625,1295.00,1520.508073,1.174138,3.908619


In [256]:
# ==========================================
# SUBMISSION 02 — 2030 CONSTRUCTION PLAN
# ==========================================

# Updated material state after 2029
current_material_2030_s02 = (
    result_2029_s02["material"].copy()
)

# Build 2030 network state
network_2030_s02 = network_2027.copy()

network_2030_s02["Material_2030"] = (
    network_2030_s02["Pipe ID"]
    .map(current_material_2030_s02)
)

# Generate 2030 predictions
prediction_2030_s02 = predict_future_year(
    network_2030_s02,
    "Material_2030",
    2030
)

# Budget plus carryover
carryover_2030_s02 = (
    result_2029_s02["unused_budget"]
)

available_budget_2030_s02 = (
    ANNUAL_BUDGET + carryover_2030_s02
)

# Execute corrected construction planner
result_2030_s02 = execute_yearly_construction(
    current_material=current_material_2030_s02,
    prediction=prediction_2030_s02,
    available_budget=available_budget_2030_s02,
    year_number=4
)

baseline_02_year_2030 = (
    result_2030_s02["plan"].copy()
)

print("2030 predicted pipes:", len(prediction_2030_s02))
print("Available budget:", available_budget_2030_s02)
print("2030 CAPEX:", result_2030_s02["spent"])
print("Unused budget:", result_2030_s02["unused_budget"])
print("Selected work units:", len(baseline_02_year_2030))
print("Iron pipes replaced:", result_2030_s02["iron_pipes_replaced"])
print("Iron meters replaced:", result_2030_s02["iron_meters_replaced"])
print("Required iron meters:", result_2030_s02["required_iron_meters"])
print("Mandatory target reached:", result_2030_s02["mandatory_target_reached"])
print("Iron remaining:", result_2030_s02["remaining_iron"])

display(baseline_02_year_2030.head(10))

2030 predicted pipes: 22386
Available budget: 22705029.149999972
2030 CAPEX: 7831264.04
Unused budget: 14873765.109999973
Selected work units: 1133
Iron pipes replaced: 1373
Iron meters replaced: 17630.366253552776
Required iron meters: 17622.571454244448
Mandatory target reached: True
Iron remaining: 20017


,Phase,Center Pipe ID,Construction Cost,Expected Benefit,BCR,Iron Meters
0,Economic,P23868,1493.05,3393.740049,2.273025,6.658588
1,Economic,P14543,1128.78,1768.218028,1.566486,4.958155
2,Economic,P23182,5461.17,7213.712836,1.320910,12.299309
3,Economic,P10561,1602.97,2041.252607,1.273419,5.297405
4,Mandatory,P39920,5065.12,4553.854967,0.899062,11.631836
5,Mandatory,P42507,2745.52,2395.933258,0.872670,8.386000
6,Mandatory,P26416,1467.07,1278.118017,0.871205,6.201331
7,Mandatory,P14326,1383.45,1125.477581,0.813530,4.139299
8,Mandatory,P39202,5515.47,4427.304956,0.802707,17.816588
9,Mandatory,P39632,2588.90,1974.622116,0.762726,11.650687


In [257]:
# Submission 02 — Save annual plans checkpoint

baseline_02_annual_plans = {
    2027: baseline_02_year_2027.copy(),
    2028: baseline_02_year_2028.copy(),
    2029: baseline_02_year_2029.copy(),
    2030: baseline_02_year_2030.copy()
}

print("Submission 02 annual plans saved in memory:")

for year, plan in baseline_02_annual_plans.items():
    print(year, len(plan), "work units")

Submission 02 annual plans saved in memory:
2027 1286 work units
2028 1790 work units
2029 1339 work units
2030 1133 work units


In [258]:

# ==========================================
# SUBMISSION 02 — SIMULATE 2031–2051
# ==========================================

simulation_results_s02 = []

# Continue from the validated 2030 state
current_material_s02 = (
    result_2030_s02["material"].copy()
)

carryover_s02 = result_2030_s02["unused_budget"]

# Continue for the remaining 21 years
for year in range(2031, 2052):

    year_number = year - 2026
    material_column = f"Material_{year}"

    # Build current year's network
    current_network_s02 = network_2027.copy()

    current_network_s02[material_column] = (
        current_network_s02["Pipe ID"]
        .map(current_material_s02)
    )

    # Predict future leaks and expected costs
    prediction_s02 = predict_future_year(
        current_network_s02,
        material_column,
        year
    )

    # Annual budget plus carryover
    available_budget_s02 = (
        ANNUAL_BUDGET + carryover_s02
    )

    # Execute corrected construction planner
    result_s02 = execute_yearly_construction(
        current_material=current_material_s02,
        prediction=prediction_s02,
        available_budget=available_budget_s02,
        year_number=year_number
    )

    # Save annual plan
    annual_plan_s02 = result_s02["plan"].copy()
    annual_plan_s02["Year"] = year

    baseline_02_annual_plans[year] = annual_plan_s02

    # Record annual results
    simulation_results_s02.append({
        "Year": year,
        "Year Number": year_number,
        "At-Risk Pipes": len(prediction_s02),
        "Expected Leaks":
            prediction_s02["predicted_leak_probability"].sum(),
        "Expected Leak Loss":
            prediction_s02["expected_leak_loss"].sum(),
        "Available Budget": available_budget_s02,
        "CAPEX": result_s02["spent"],
        "Unused Budget": result_s02["unused_budget"],
        "Work Units": len(annual_plan_s02),
        "Economic Units":
            (annual_plan_s02["Phase"] == "Economic").sum(),
        "Mandatory Units":
            (annual_plan_s02["Phase"] == "Mandatory").sum(),
        "Iron Pipes Replaced":
            result_s02["iron_pipes_replaced"],
        "Iron Meters Replaced":
            result_s02["iron_meters_replaced"],
        "Required Iron Meters":
            result_s02["required_iron_meters"],
        "Target Reached":
            result_s02["mandatory_target_reached"],
        "Iron Remaining":
            result_s02["remaining_iron"],
        "Iron Length Remaining":
            result_s02["remaining_iron_length"],
        "Expected Loss Avoided":
            result_s02["expected_loss_avoided"]
    })

    # Update state for next year
    current_material_s02 = result_s02["material"].copy()
    carryover_s02 = result_s02["unused_budget"]


# Convert results to DataFrame
simulation_results_s02 = pd.DataFrame(
    simulation_results_s02
)

display(simulation_results_s02)


,Year,Year Number,At-Risk Pipes,Expected Leaks,Expected Leak Loss,Available Budget,CAPEX,Unused Budget,Work Units,Economic Units,Mandatory Units,Iron Pipes Replaced,Iron Meters Replaced,Required Iron Meters,Target Reached,Iron Remaining,Iron Length Remaining,Expected Loss Avoided
0,2031,5,21007,427.853584,4.586774e+06,24873765.11,9265163.02,15608602.09,1038,1,1037,1181,17644.294950,17619.298121,True,18836,352516.861570,1.311030e+06
1,2032,6,19815,396.703532,3.400699e+06,25608602.09,9268539.20,16340062.89,956,1,955,1038,17611.174672,17602.096090,True,17798,334905.686898,9.535630e+05
2,2033,7,18760,714.870936,5.734596e+06,26340062.89,13093638.57,13246424.32,586,0,586,640,17641.388176,17618.014338,True,17158,317264.298722,3.961097e+06
3,2034,8,18106,803.865619,1.968757e+06,23246424.32,8037777.82,15208646.50,1223,0,1223,1287,17613.727135,17603.719082,True,15871,299650.571587,5.421571e+05
4,2035,9,16816,634.100081,1.677492e+06,25208646.50,6700498.85,18508147.65,1422,0,1422,1464,17622.270661,17617.084867,True,14407,282028.300926,3.279272e+05
5,2036,10,15335,345.852003,1.427549e+06,28508147.65,6607790.37,21900357.28,1459,0,1459,1473,17624.072728,17621.907126,True,12934,264404.228197,2.671924e+05
6,2037,11,13855,262.092260,1.215189e+06,31900357.28,5734673.85,26165683.43,1452,0,1452,1475,17629.522174,17624.927317,True,11459,246774.706023,1.931097e+05
7,2038,12,12373,226.580264,1.123844e+06,36165683.43,5842705.02,30322978.41,1344,0,1344,1364,17622.539766,17622.498063,True,10095,229152.166257,1.738538e+05
8,2039,13,10999,207.339584,1.120177e+06,40322978.41,6952368.43,33370609.98,1168,0,1168,1191,17634.193028,17627.051217,True,8904,211517.973229,1.771719e+05
9,2040,14,9802,190.449863,9.448244e+05,43370609.98,6473898.92,36896711.06,1142,0,1142,1159,17620.203153,17619.951109,True,7745,193897.770077,1.267709e+05


In [259]:
print(
    "All annual targets reached:",
    simulation_results_s02["Target Reached"].all()
)

print(
    "Final iron pipes remaining:",
    simulation_results_s02["Iron Remaining"].iloc[-1]
)

print(
    "Final iron length remaining:",
    simulation_results_s02["Iron Length Remaining"].iloc[-1]
)

print(
    "Any budget violation:",
    (
        simulation_results_s02["CAPEX"]
        > simulation_results_s02["Available Budget"]
    ).any()
)

print(
    "Total CAPEX:",
    simulation_results_s02["CAPEX"].sum()
)

print(
    "Final unused budget:",
    simulation_results_s02["Unused Budget"].iloc[-1]
)

print(
    "Total work units, 2027–2051:",
    sum(len(p) for p in baseline_02_annual_plans.values())
)

All annual targets reached: True
Final iron pipes remaining: 0
Final iron length remaining: 0.0
Any budget violation: False
Total CAPEX: 165155838.84
Final unused budget: 59717926.26999998
Total work units, 2027–2051: 24415


In [260]:

import pandas as pd
import numpy as np
import csv
from pathlib import Path

# ==========================================
# SUBMISSION 02 — FINAL CSV GENERATION
# ==========================================

# 1. Confirm all 25 annual plans exist
expected_years = list(range(2027, 2052))

assert sorted(baseline_02_annual_plans.keys()) == expected_years

# 2. Combine all annual work-unit plans
all_plans_s02 = []

for year in expected_years:
    plan = baseline_02_annual_plans[year].copy()
    plan["Year"] = year
    all_plans_s02.append(plan)

submission_02_all = pd.concat(
    all_plans_s02,
    ignore_index=True
)

# 3. Attach original work-unit geometry
geometry_s02 = work_units[
    ["Center Pipe ID", "Center x", "Center y", "Radius",
     "Covered Pipe IDs"]
].copy()

submission_02_all = submission_02_all.merge(
    geometry_s02,
    on="Center Pipe ID",
    how="left",
    validate="many_to_one"
)

# 4. Validate geometry
assert submission_02_all[
    ["Center x", "Center y", "Radius"]
].notna().all().all()

assert (submission_02_all["Radius"] > 0).all()

# 5. Check duplicate work units
duplicate_count = submission_02_all[
    "Center Pipe ID"
].duplicated().sum()

print("Duplicate work units:", duplicate_count)

# 6. Verify iron-pipe coverage
selected_covered_ids = set()

for ids in submission_02_all["Covered Pipe IDs"]:
    selected_covered_ids.update(ids)

original_iron_ids = set(
    network_2027.loc[
        network_2027["Material_2027"].isin(iron_materials),
        "Pipe ID"
    ]
)

uncovered_iron_ids = (
    original_iron_ids - selected_covered_ids
)

print("Original iron pipes:", len(original_iron_ids))
print("Uncovered iron pipes:", len(uncovered_iron_ids))

# 7. Create exact submission format
submission_02_csv = pd.DataFrame({
    "coordinate": [
        f"({x:.9f}, {y:.9f})"
        for x, y in zip(
            submission_02_all["Center x"],
            submission_02_all["Center y"]
        )
    ],
    "radius": submission_02_all["Radius"]
})

# 8. Save in submissions folder
submissions_dir = Path.cwd() / "submissions"
submissions_dir.mkdir(parents=True, exist_ok=True)

submission_02_path = (
    submissions_dir / "submission_02.csv"
)

submission_02_csv.to_csv(
    submission_02_path,
    index=False,
    header=False,
    quoting=csv.QUOTE_MINIMAL,
    float_format="%.9f"
)

# 9. Final checks
print("\n========== SUBMISSION 02 ==========")
print("Total work units:", len(submission_02_csv))
print("Years:", submission_02_all["Year"].nunique())
print("Year range:",
      submission_02_all["Year"].min(),
      "-",
      submission_02_all["Year"].max())
print("Duplicate units:", duplicate_count)
print("Uncovered iron pipes:", len(uncovered_iron_ids))
print("File exists:", submission_02_path.exists())
print("File size (MB):",
      round(submission_02_path.stat().st_size / 1e6, 2))
print("Saved to:", submission_02_path)

print("\nFirst 10 lines:")
with open(submission_02_path, "r") as f:
    for _ in range(10):
        print(f.readline().rstrip())


Duplicate work units: 0
Original iron pipes: 26237
Uncovered iron pipes: 0

========== SUBMISSION 02 ==========
Total work units: 24415
Years: 25
Year range: 2027 - 2051
Duplicate units: 0
Uncovered iron pipes: 0
File exists: True
File size (MB): 1.18
Saved to: e:\User\nsim\Documents\User\nsim\Documents\stat440\STAT440-star-kingdom-project\notebooks\Amy_Sim\submissions\submission_02.csv

First 10 lines:
"(1784.040000000, 2800.980000000)",1.186971935
"(1359.095000000, 5417.305000000)",1.051595028
"(1486.920000000, 3253.255000000)",1.838947709
"(1628.065000000, 2652.035000000)",2.114250276
"(1360.355000000, 5416.460000000)",1.138914956
"(745.855000000, 3168.290000000)",2.072831191
"(1626.335000000, 2655.535000000)",2.704377083
"(1533.140000000, 2906.350000000)",2.728095573
"(1626.465000000, 2649.895000000)",2.966320268
"(1542.170000000, 3424.830000000)",2.632452329
